# SETTING ENVIRONMENT

In [ ]:
# =============================================================================
# CELL 1 — BASE ENV (RUN ONCE) then HARD RESTART
# - Fix NumPy/OpenCV ABI in Colab
# - Fix Pillow to avoid: ImportError: cannot import name '_Ink' from PIL._typing
# =============================================================================
!pip -q uninstall -y opencv-python opencv-python-headless numpy pillow
!pip -q install "numpy==1.26.4"
!pip -q install "opencv-python-headless==4.9.0.80"
!pip -q install -U transformers huggingface_hub safetensors accelerate "pillow<12" matplotlib
!pip -q install -U pyarrow
!pip -q install scikit-learn seaborn tqdm

import os
os.kill(os.getpid(), 9)

# 2) IMPORTS AND DATA INGESTION

In [ ]:
# =============================================================================
# CELL 2 — IMPORTS + WORKSPACE
# =============================================================================
# EXPLANATION:
# This cell centralizes imports, defines the project workspace, creates the
# folder structure for the pipeline, and initializes global constants used
# across the notebook.
#
# PIPELINE ALIGNMENT (CURRENT REVISION):
#   1) Upload and extract ZIP file to Bronze/source_images
#   2) Detect/select the target pole from each raw image
#   3) Save selected pole ROI crops to Silver/pole_rois
#   4) Run branch 1 asset prompt detection on pole ROIs
#   5) Run branch 2 crossarm detection + evolving post-processing on pole ROIs
#   6) Save final cleaned outputs into Gold
#
# IMPORTANT:
# - Bronze contains only untouched source inputs
# - Silver contains shared pole ROIs plus raw/intermediate branch outputs
# - Gold contains only final cleaned outputs
# =============================================================================

# -----------------------------------------------------------------------------
# Standard library
# -----------------------------------------------------------------------------
import os
import gc
import re
import io
import json
import time
import glob
import math
import shutil
import random
import zipfile
import warnings
from getpass import getpass
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

# -----------------------------------------------------------------------------
# Colab / notebook utilities
# -----------------------------------------------------------------------------
import pandas as pd
from google.colab import files

# -----------------------------------------------------------------------------
# Data science and visualization
# -----------------------------------------------------------------------------
import cv2
import numpy as np
import matplotlib.pyplot as plt
import PIL
from PIL import Image, ImageOps

# -----------------------------------------------------------------------------
# Hugging Face / model loading
# -----------------------------------------------------------------------------
from huggingface_hub import login, HfApi, hf_hub_download
from transformers import Sam3Model, Sam3Processor

# -----------------------------------------------------------------------------
# Machine learning / utilities
# -----------------------------------------------------------------------------
import torch

# -----------------------------------------------------------------------------
# Global warning behavior
# -----------------------------------------------------------------------------
warnings.filterwarnings("ignore")

# =============================================================================
# WORKSPACE ROOT
# =============================================================================
WORK_DIR = "/content/sam3_crossarm_project"

# -----------------------------------------------------------------------------
# Core state / artifact folders
# -----------------------------------------------------------------------------
STATE_DIR = os.path.join(WORK_DIR, "state")
DF_DIR    = os.path.join(STATE_DIR, "dataframes")
ART_DIR   = os.path.join(WORK_DIR, "artifacts")

# =============================================================================
# PROJECT FOLDER STRUCTURE
# =============================================================================

# -----------------------------------------------------------------------------
# Bronze: raw source data
# -----------------------------------------------------------------------------
# EXPLANATION:
# Bronze contains only untouched source inputs.
#
# WHAT LIVES HERE:
# - uploaded ZIP file(s)
# - extracted raw full-resolution source images
# -----------------------------------------------------------------------------
BRONZE_ROOT          = os.path.join(WORK_DIR, "bronze")
BRONZE_INPUT_ZIPS    = os.path.join(BRONZE_ROOT, "input_zips")
BRONZE_SOURCE_IMAGES = os.path.join(BRONZE_ROOT, "source_images")

# -----------------------------------------------------------------------------
# Silver: shared pole ROI layer
# -----------------------------------------------------------------------------
# EXPLANATION:
# These are the selected pole crops that become the shared input to all
# downstream SAM3 runs.
# -----------------------------------------------------------------------------
SILVER_ROOT      = os.path.join(WORK_DIR, "silver")
SILVER_POLE_ROIS = os.path.join(SILVER_ROOT, "pole_rois")

# -----------------------------------------------------------------------------
# Silver branch 1: asset detection candidates
# -----------------------------------------------------------------------------
# EXPLANATION:
# This is the simple exploratory branch where different SAM3 prompts such as
# transformer / insulator / shroud are run on the pole ROI.
#
# WHAT LIVES HERE:
# - raw prompt-run outputs
# - overlay images
# - optional masks / crops / review exports
#
# IMPORTANT:
# - these are candidate detections only
# - no final cleanup or downstream logic is assumed here yet
# -----------------------------------------------------------------------------
SILVER_ASSET_DETECTION_CANDIDATES = os.path.join(SILVER_ROOT, "asset_detection_candidates")
SILVER_ASSET_PROMPT_RUNS          = os.path.join(SILVER_ASSET_DETECTION_CANDIDATES, "prompt_runs")
SILVER_ASSET_OVERLAYS             = os.path.join(SILVER_ASSET_DETECTION_CANDIDATES, "overlays")
SILVER_ASSET_MASKS                = os.path.join(SILVER_ASSET_DETECTION_CANDIDATES, "masks")
SILVER_ASSET_CROPS                = os.path.join(SILVER_ASSET_DETECTION_CANDIDATES, "crops")
SILVER_ASSET_REVIEW_EXPORTS       = os.path.join(SILVER_ASSET_DETECTION_CANDIDATES, "review_exports")

# -----------------------------------------------------------------------------
# Silver branch 2: crossarm detection
# -----------------------------------------------------------------------------
# EXPLANATION:
# This branch is expected to evolve beyond a simple prompt run.
#
# WHAT LIVES HERE:
# - candidates   : raw SAM3 outputs and early candidate artifacts
# - processing   : all intermediate post-processing stages
# - review       : debug / QA / before-after review outputs
#
# IMPORTANT:
# - do not over-specify post-processing subfolders yet
# - use the broad "processing" bucket until the workflow stabilizes
# -----------------------------------------------------------------------------
SILVER_CROSSARM_DETECTION = os.path.join(SILVER_ROOT, "crossarm_detection")
SILVER_CROSSARM_CANDIDATES = os.path.join(SILVER_CROSSARM_DETECTION, "candidates")
SILVER_CROSSARM_PROCESSING = os.path.join(SILVER_CROSSARM_DETECTION, "processing")
SILVER_CROSSARM_REVIEW     = os.path.join(SILVER_CROSSARM_DETECTION, "review")

# -----------------------------------------------------------------------------
# Gold: final curated outputs
# -----------------------------------------------------------------------------
# EXPLANATION:
# Gold contains only cleaned, decision-ready outputs.
#
# WHAT LIVES HERE:
# - final asset detections
# - final crossarm detections
# - one-row-per-image joined pole summary
# -----------------------------------------------------------------------------
GOLD_ROOT                = os.path.join(WORK_DIR, "gold")
GOLD_ASSET_DETECTIONS    = os.path.join(GOLD_ROOT, "asset_detections")
GOLD_CROSSARM_DETECTIONS = os.path.join(GOLD_ROOT, "crossarm_detections")
GOLD_POLE_SUMMARY        = os.path.join(GOLD_ROOT, "pole_summary")

# -----------------------------------------------------------------------------
# Config: reproducible pipeline settings
# -----------------------------------------------------------------------------
CONFIG_ROOT    = os.path.join(WORK_DIR, "config")
CONFIG_PROMPTS = os.path.join(CONFIG_ROOT, "prompts")
CONFIG_THRESH  = os.path.join(CONFIG_ROOT, "thresholds")

# -----------------------------------------------------------------------------
# Experiments / cache
# -----------------------------------------------------------------------------
EXPERIMENTS     = os.path.join(WORK_DIR, "experiments")
MODEL_CACHE_DIR = os.path.join(WORK_DIR, "model_cache")

# -----------------------------------------------------------------------------
# Create directory tree
# -----------------------------------------------------------------------------
for d in [
    WORK_DIR,
    STATE_DIR,
    DF_DIR,
    ART_DIR,

    BRONZE_ROOT,
    BRONZE_INPUT_ZIPS,
    BRONZE_SOURCE_IMAGES,

    SILVER_ROOT,
    SILVER_POLE_ROIS,

    SILVER_ASSET_DETECTION_CANDIDATES,
    SILVER_ASSET_PROMPT_RUNS,
    SILVER_ASSET_OVERLAYS,
    SILVER_ASSET_MASKS,
    SILVER_ASSET_CROPS,
    SILVER_ASSET_REVIEW_EXPORTS,

    SILVER_CROSSARM_DETECTION,
    SILVER_CROSSARM_CANDIDATES,
    SILVER_CROSSARM_PROCESSING,
    SILVER_CROSSARM_REVIEW,

    GOLD_ROOT,
    GOLD_ASSET_DETECTIONS,
    GOLD_CROSSARM_DETECTIONS,
    GOLD_POLE_SUMMARY,

    CONFIG_ROOT,
    CONFIG_PROMPTS,
    CONFIG_THRESH,

    EXPERIMENTS,
    MODEL_CACHE_DIR,
]:
    os.makedirs(d, exist_ok=True)

# =============================================================================
# CONSTANTS
# =============================================================================

# -----------------------------------------------------------------------------
# Reproducibility constants
# -----------------------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -----------------------------------------------------------------------------
# Project identity
# -----------------------------------------------------------------------------
PROJECT_NAME = "sam3_crossarm_detection"
RUN_STAMP = pd.Timestamp.utcnow().strftime("%Y%m%d_%H%M%S")

# -----------------------------------------------------------------------------
# Dataset / input hints
# -----------------------------------------------------------------------------
LOCAL_ZIP_HINT = r"C:\Users\shary\Documents\Defect_Project\Xarm.zip"

PROJECT_ZIP_NAME = "uploaded_dataset.zip"
PROJECT_ZIP_PATH = os.path.join(BRONZE_INPUT_ZIPS, PROJECT_ZIP_NAME)

IMAGE_EXTS = (
    ".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff",
    ".JPG", ".JPEG", ".PNG", ".BMP", ".WEBP", ".TIF", ".TIFF"
)

# -----------------------------------------------------------------------------
# SAM3 model and inference constants
# -----------------------------------------------------------------------------
SAM3_MODEL_ID = "facebook/sam3"

# -----------------------------------------------------------------------------
# Global fallbacks
# -----------------------------------------------------------------------------
# Used only if a task/prompt does not provide its own value.
GLOBAL_TEXT_SCORE_THRESHOLD = 0.45
GLOBAL_MASK_THRESHOLD = 0.50
MAX_IMAGES_FOR_DEBUG_GALLERY = 40

# -----------------------------------------------------------------------------
# Branch-specific prompt sets and thresholds
# -----------------------------------------------------------------------------
SAM3_TASK_CONFIG = {
    "pole_detection": {
        "prompts": [
            "timber utility power pole",
        ],
        "text_score_threshold": 0.45,
        "mask_threshold": 0.50,
    },

    "crossarm_detection": {
        "prompts": [
            "utility pole crossarm",
            "electricity pole crossarm",
            "distribution pole crossarm",
        ],
        "text_score_threshold": 0.45,
        "mask_threshold": 0.50,
    },

    "asset_detection": {
        "prompt_config": {
            "transformer": {
                "text_score_threshold": 0.40,
                "mask_threshold": 0.50,
            },
            "insulator": {
                "text_score_threshold": 0.30,
                "mask_threshold": 0.50,
            },
            "shroud": {
                "text_score_threshold": 0.25,
                "mask_threshold": 0.50,
            },
        }
    },
}

# -----------------------------------------------------------------------------
# Persist config
# -----------------------------------------------------------------------------
SAM3_TASK_CONFIG_JSON = os.path.join(CONFIG_PROMPTS, "sam3_task_config.json")
with open(SAM3_TASK_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(SAM3_TASK_CONFIG, f, indent=2)

# Optional convenience aliases if later cells want a direct variable
POLE_PROMPTS = SAM3_TASK_CONFIG["pole_detection"]["prompts"]
CROSSARM_PROMPTS = SAM3_TASK_CONFIG["crossarm_detection"]["prompts"]

# -----------------------------------------------------------------------------
# Helpful runtime info
# -----------------------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("Project workspace ready.")
print(f"PROJECT_NAME                    : {PROJECT_NAME}")
print(f"WORK_DIR                        : {WORK_DIR}")
print(f"DEVICE                          : {DEVICE}")
print(f"MODEL_DTYPE                     : {MODEL_DTYPE}")
print(f"SAM3_MODEL_ID                   : {SAM3_MODEL_ID}")
print(f"LOCAL_ZIP_HINT                  : {LOCAL_ZIP_HINT}")
print(f"PROJECT_ZIP_PATH                : {PROJECT_ZIP_PATH}")
print(f"BRONZE_SOURCE_IMAGES            : {BRONZE_SOURCE_IMAGES}")
print(f"SILVER_POLE_ROIS                : {SILVER_POLE_ROIS}")
print(f"SILVER_ASSET_PROMPT_RUNS        : {SILVER_ASSET_PROMPT_RUNS}")
print(f"SILVER_ASSET_OVERLAYS           : {SILVER_ASSET_OVERLAYS}")
print(f"SILVER_CROSSARM_CANDIDATES      : {SILVER_CROSSARM_CANDIDATES}")
print(f"SILVER_CROSSARM_PROCESSING      : {SILVER_CROSSARM_PROCESSING}")
print(f"SILVER_CROSSARM_REVIEW          : {SILVER_CROSSARM_REVIEW}")
print(f"GOLD_ASSET_DETECTIONS           : {GOLD_ASSET_DETECTIONS}")
print(f"GOLD_CROSSARM_DETECTIONS        : {GOLD_CROSSARM_DETECTIONS}")
print(f"GOLD_POLE_SUMMARY               : {GOLD_POLE_SUMMARY}")
print(f"SAM3_TASK_CONFIG_JSON           : {SAM3_TASK_CONFIG_JSON}")
print("CROSSARM_PROMPTS:")
for p in CROSSARM_PROMPTS:
    print(f"  - {p}")

In [ ]:
# =============================================================================
# CELL 2b — DATA INGESTION: UPLOAD AND EXTRACT ZIP INTO BRONZE
# =============================================================================
# EXPLANATION:
# This cell continues from Cell 2 and handles raw data ingestion only.
#
# WHAT THIS CELL DOES:
#   1) uploads exactly one ZIP file
#   2) saves it to the stable Bronze ZIP path
#   3) clears the Bronze/source_images folder
#   4) extracts the ZIP contents into Bronze/source_images
#   5) discovers image files recursively
#   6) builds a simple images_df manifest for later pole detection
#
# IMPORTANT:
# - this cell now uses BRONZE_SOURCE_IMAGES, not the old XARM_EXTRACT_DIR
# - Bronze should contain only raw extracted source images
# - no pole ROI logic or downstream SAM3 logic belongs here
# =============================================================================
# 0. Validate that Cell 2 has already been run
# -----------------------------------------------------------------------------
# EXPLANATION:
# This cell depends on the workspace variables created in Cell 2.
# If Cell 2 did not run, fail early with a clear message.
# -----------------------------------------------------------------------------
if "BRONZE_INPUT_ZIPS" not in globals():
    raise NameError("BRONZE_INPUT_ZIPS is not defined. Please run Cell 2 first.")

if "BRONZE_SOURCE_IMAGES" not in globals():
    raise NameError("BRONZE_SOURCE_IMAGES is not defined. Please run Cell 2 first.")

if "IMAGE_EXTS" not in globals():
    raise NameError("IMAGE_EXTS is not defined. Please run Cell 2 first.")

if "LOCAL_ZIP_HINT" not in globals():
    LOCAL_ZIP_HINT = r"C:\Users\shary\Documents\Defect_Project\Xarm.zip"

if "PROJECT_ZIP_NAME" not in globals():
    PROJECT_ZIP_NAME = "uploaded_dataset.zip"

if "PROJECT_ZIP_PATH" not in globals():
    PROJECT_ZIP_PATH = os.path.join(BRONZE_INPUT_ZIPS, PROJECT_ZIP_NAME)

# -----------------------------------------------------------------------------
# 0b. Normalize image extensions once
# -----------------------------------------------------------------------------
# EXPLANATION:
# Cell 2 may define IMAGE_EXTS with mixed case variants such as .jpg and .JPG.
# We normalize once here so the discovery loop stays clean and avoids rebuilding
# the same lowercase tuple repeatedly.
# -----------------------------------------------------------------------------
IMAGE_EXTS_LOWER = tuple(sorted({ext.lower() for ext in IMAGE_EXTS}))

# -----------------------------------------------------------------------------
# 1. Trigger Colab upload widget
# -----------------------------------------------------------------------------
# EXPLANATION:
# The user should upload exactly one ZIP file containing the raw images.
# -----------------------------------------------------------------------------
print("Please choose exactly ONE ZIP file from your computer.")
print(f"Local path hint: {LOCAL_ZIP_HINT}")

uploaded = files.upload()

# -----------------------------------------------------------------------------
# 2. Validate uploaded file count and type
# -----------------------------------------------------------------------------
# EXPLANATION:
# We enforce a strict ingestion rule:
# - exactly one uploaded file
# - that file must be a ZIP
# This avoids ambiguous or messy project state.
# -----------------------------------------------------------------------------
uploaded_names = list(uploaded.keys())

if len(uploaded_names) == 0:
    raise FileNotFoundError("No file was uploaded.")

if len(uploaded_names) > 1:
    raise ValueError(
        f"Please upload only ONE ZIP file.\n"
        f"Uploaded files were: {uploaded_names}"
    )

uploaded_name = uploaded_names[0]

if not uploaded_name.lower().endswith(".zip"):
    raise ValueError(f"Uploaded file is not a ZIP file: {uploaded_name}")

print(f"Uploaded ZIP detected: {uploaded_name}")

# -----------------------------------------------------------------------------
# 3. Save uploaded ZIP to one stable Bronze path
# -----------------------------------------------------------------------------
# EXPLANATION:
# Even if the uploaded filename changes each run, we save it to one fixed
# internal path. This keeps later cells simple and reproducible.
# -----------------------------------------------------------------------------
with open(PROJECT_ZIP_PATH, "wb") as f:
    f.write(uploaded[uploaded_name])

UPLOADED_ZIP_DISPLAY_NAME = uploaded_name

print(f"Saved project ZIP to: {PROJECT_ZIP_PATH}")

# -----------------------------------------------------------------------------
# 4. Validate ZIP integrity before extraction
# -----------------------------------------------------------------------------
# EXPLANATION:
# This catches broken uploads before we try to extract.
# -----------------------------------------------------------------------------
if not zipfile.is_zipfile(PROJECT_ZIP_PATH):
    raise ValueError(f"Not a valid ZIP archive: {PROJECT_ZIP_PATH}")

# -----------------------------------------------------------------------------
# 5. Clear previous Bronze/source_images folder and extract fresh
# -----------------------------------------------------------------------------
# EXPLANATION:
# We wipe the old extracted image folder first so there are no ghost files from
# earlier runs. Bronze/source_images should always reflect the current upload.
# -----------------------------------------------------------------------------
if os.path.isdir(BRONZE_SOURCE_IMAGES):
    shutil.rmtree(BRONZE_SOURCE_IMAGES)

os.makedirs(BRONZE_SOURCE_IMAGES, exist_ok=True)

with zipfile.ZipFile(PROJECT_ZIP_PATH, "r") as zf:
    zf.extractall(BRONZE_SOURCE_IMAGES)

print(f"Extracted ZIP into: {BRONZE_SOURCE_IMAGES}")

# -----------------------------------------------------------------------------
# 6. Recursively discover extracted image files
# -----------------------------------------------------------------------------
# EXPLANATION:
# The ZIP may contain nested folders, so we scan recursively for valid image
# files using the normalized supported extensions prepared above.
# -----------------------------------------------------------------------------
image_files = []

for root, _, dir_files in os.walk(BRONZE_SOURCE_IMAGES):
    for fn in dir_files:
        if fn.lower().endswith(IMAGE_EXTS_LOWER):
            image_files.append(os.path.join(root, fn))

image_files = sorted(image_files)

print(f"Extracted image count: {len(image_files)}")

if len(image_files) == 0:
    raise ValueError(
        "No image files were found after extraction.\n"
        "Check whether the ZIP contains images in nested folders or unsupported formats."
    )

print("\nFirst few extracted image paths:")
for fp in image_files[:10]:
    print(f"  - {fp}")

# -----------------------------------------------------------------------------
# 7. Build the raw image manifest
# -----------------------------------------------------------------------------
# EXPLANATION:
# images_df becomes the basic tracking table for the raw source images.
# At this stage it is intentionally simple because no pole detection or ROI
# selection has happened yet.
# -----------------------------------------------------------------------------
images_df = pd.DataFrame({
    "image_path": image_files,
})

images_df["file_name"] = images_df["image_path"].map(os.path.basename)
images_df["stem"] = images_df["file_name"].map(lambda x: os.path.splitext(x)[0])
images_df["ext"] = images_df["file_name"].map(lambda x: os.path.splitext(x)[1])

# Optional but useful bookkeeping columns
images_df["uploaded_zip_name"] = UPLOADED_ZIP_DISPLAY_NAME
images_df["project_zip_path"] = PROJECT_ZIP_PATH
images_df["source_layer"] = "bronze"
images_df["source_folder"] = BRONZE_SOURCE_IMAGES

# -----------------------------------------------------------------------------
# 8. Print summary and preview manifest
# -----------------------------------------------------------------------------
# EXPLANATION:
# This is a quick sanity check before later cells begin pole detection.
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("BRONZE INGESTION SUMMARY")
print("=" * 80)
print(f"PROJECT_ZIP_PATH      : {PROJECT_ZIP_PATH}")
print(f"Uploaded ZIP name     : {UPLOADED_ZIP_DISPLAY_NAME}")
print(f"BRONZE_SOURCE_IMAGES  : {BRONZE_SOURCE_IMAGES}")
print(f"images_df shape       : {images_df.shape}")

try:
    display(images_df.head())
except NameError:
    print(images_df.head())

# 3) SAVE / RESTORE POINT

In [ ]:
# =============================================================================
# CELL 3 — CONTINUITY: SAVE / RESTORE
# =============================================================================
# EXPLANATION:
# Colab sessions can disconnect, restart, or clear in-memory variables.
# This cell provides a lightweight persistence mechanism so you do not lose
# important tables and configuration every time the runtime resets.
#
# WHAT THIS CELL DOES:
#   1) Defines which DataFrames are worth saving by default
#   2) Provides a save_state() function
#   3) Provides a restore_state() function
#   4) Saves both:
#        - configuration / paths / constants
#        - selected pandas DataFrames
#
# PIPELINE ALIGNMENT:
# - Cell 2 defines workspace paths, folder structure, and SAM3 task config
# - Cell 2b builds images_df and stores uploaded ZIP metadata
# - This cell allows those outputs to be saved and restored later
# =============================================================================

# =============================================================================
# DEFAULT DATAFRAME REGISTRY
# =============================================================================
# EXPLANATION:
# This is the default list of DataFrame variable names the notebook will try
# to save if you call save_state() without explicitly passing df_names.
#
# IMPORTANT:
# - These are only default names.
# - A DataFrame is saved only if the global exists AND is a pandas DataFrame.
# - Missing names do not crash the save process.
# =============================================================================
PIPELINE_DF_NAMES = [
    # -------------------------------------------------------------------------
    # Bronze / ingestion
    # -------------------------------------------------------------------------
    "images_df",                      # one row per discovered raw image
    "run_images_df",                  # cleaned working image table for pole detection
    "debug_images_df",                # smaller stable debug subset

    # -------------------------------------------------------------------------
    # Pole selection / ROI stage
    # -------------------------------------------------------------------------
    "pole_candidates_df",
    "pole_selection_df",
    "pole_rois_df",
    "pole_raw_candidates_df",
    "pole_feature_debug_df",

    # -------------------------------------------------------------------------
    # Asset detection branch
    # -------------------------------------------------------------------------
    "asset_prompt_manifest_df",
    "asset_prompt_results_df",
    "asset_candidates_df",

    # -------------------------------------------------------------------------
    # Crossarm detection branch
    # -------------------------------------------------------------------------
    "crossarm_prompt_manifest_df",
    "crossarm_candidates_df",
    "crossarm_processing_df",
    "crossarm_masks_df",
    "crossarm_crops_df",

    # -------------------------------------------------------------------------
    # Final outputs / QA
    # -------------------------------------------------------------------------
    "pole_summary_df",
    "run_summary_df",
    "eval_df",
]

# =============================================================================
# SAVE STATE FUNCTION
# =============================================================================
def save_state(df_names=None, config_extra=None, nb_globals=None):
    """
    Save selected DataFrames and the current project configuration to disk.

    Args:
        df_names:
            Optional list of global DataFrame variable names to save.
            If None, the default PIPELINE_DF_NAMES list is used.

        config_extra:
            Optional dictionary of extra config values to merge into the
            saved JSON snapshot.

        nb_globals:
            Optional namespace dictionary.
            Defaults to the current notebook globals.

    Returns:
        str:
            Path to the saved config.json file.
    """

    # -------------------------------------------------------------------------
    # 1. Resolve the notebook namespace
    # -------------------------------------------------------------------------
    # Using an explicit namespace keeps the function flexible and makes the
    # save / restore logic easier to reason about.
    if nb_globals is None:
        nb_globals = globals()

    # -------------------------------------------------------------------------
    # 2. Resolve which DataFrame names should be considered for saving
    # -------------------------------------------------------------------------
    if df_names is None:
        df_names = PIPELINE_DF_NAMES

    # -------------------------------------------------------------------------
    # 3. Resolve required output directories safely
    # -------------------------------------------------------------------------
    # These must exist to save the config JSON and parquet DataFrames.
    state_dir = nb_globals.get("STATE_DIR")
    df_dir = nb_globals.get("DF_DIR")

    if not state_dir:
        raise NameError("STATE_DIR is not defined. Please run Cell 2 first.")

    if not df_dir:
        raise NameError("DF_DIR is not defined. Please run Cell 2 first.")

    os.makedirs(state_dir, exist_ok=True)
    os.makedirs(df_dir, exist_ok=True)

    # -------------------------------------------------------------------------
    # 4. Build the configuration snapshot dictionary
    # -------------------------------------------------------------------------
    # This captures the important notebook constants, paths, runtime details,
    # and pipeline settings so the environment can be reconstructed later.
    cfg = {
        # ---------------------------------------------------------------------
        # Timestamps
        # ---------------------------------------------------------------------
        "timestamp_utc": time.time(),
        "timestamp_utc_iso": pd.Timestamp.utcnow().isoformat(),

        # ---------------------------------------------------------------------
        # Project identity
        # ---------------------------------------------------------------------
        "PROJECT_NAME": nb_globals.get("PROJECT_NAME"),
        "RUN_STAMP": nb_globals.get("RUN_STAMP"),

        # ---------------------------------------------------------------------
        # Core workspace paths
        # ---------------------------------------------------------------------
        "WORK_DIR": nb_globals.get("WORK_DIR"),
        "STATE_DIR": nb_globals.get("STATE_DIR"),
        "DF_DIR": nb_globals.get("DF_DIR"),
        "ART_DIR": nb_globals.get("ART_DIR"),

        # ---------------------------------------------------------------------
        # Bronze layer paths
        # ---------------------------------------------------------------------
        "BRONZE_ROOT": nb_globals.get("BRONZE_ROOT"),
        "BRONZE_INPUT_ZIPS": nb_globals.get("BRONZE_INPUT_ZIPS"),
        "BRONZE_SOURCE_IMAGES": nb_globals.get("BRONZE_SOURCE_IMAGES"),

        # ---------------------------------------------------------------------
        # ZIP handling
        # ---------------------------------------------------------------------
        # PROJECT_ZIP_PATH is the stable internal path.
        # UPLOADED_ZIP_DISPLAY_NAME is the original user-facing file name.
        "LOCAL_ZIP_HINT": nb_globals.get("LOCAL_ZIP_HINT"),
        "PROJECT_ZIP_NAME": nb_globals.get("PROJECT_ZIP_NAME"),
        "PROJECT_ZIP_PATH": nb_globals.get("PROJECT_ZIP_PATH"),
        "UPLOADED_ZIP_DISPLAY_NAME": nb_globals.get("UPLOADED_ZIP_DISPLAY_NAME"),

        # ---------------------------------------------------------------------
        # Silver layer paths
        # ---------------------------------------------------------------------
        "SILVER_ROOT": nb_globals.get("SILVER_ROOT"),
        "SILVER_POLE_ROIS": nb_globals.get("SILVER_POLE_ROIS"),

        # Asset detection branch
        "SILVER_ASSET_DETECTION_CANDIDATES": nb_globals.get("SILVER_ASSET_DETECTION_CANDIDATES"),
        "SILVER_ASSET_PROMPT_RUNS": nb_globals.get("SILVER_ASSET_PROMPT_RUNS"),
        "SILVER_ASSET_OVERLAYS": nb_globals.get("SILVER_ASSET_OVERLAYS"),
        "SILVER_ASSET_MASKS": nb_globals.get("SILVER_ASSET_MASKS"),
        "SILVER_ASSET_CROPS": nb_globals.get("SILVER_ASSET_CROPS"),
        "SILVER_ASSET_REVIEW_EXPORTS": nb_globals.get("SILVER_ASSET_REVIEW_EXPORTS"),

        # Crossarm detection branch
        "SILVER_CROSSARM_DETECTION": nb_globals.get("SILVER_CROSSARM_DETECTION"),
        "SILVER_CROSSARM_CANDIDATES": nb_globals.get("SILVER_CROSSARM_CANDIDATES"),
        "SILVER_CROSSARM_PROCESSING": nb_globals.get("SILVER_CROSSARM_PROCESSING"),
        "SILVER_CROSSARM_REVIEW": nb_globals.get("SILVER_CROSSARM_REVIEW"),

        # ---------------------------------------------------------------------
        # Gold layer paths
        # ---------------------------------------------------------------------
        "GOLD_ROOT": nb_globals.get("GOLD_ROOT"),
        "GOLD_ASSET_DETECTIONS": nb_globals.get("GOLD_ASSET_DETECTIONS"),
        "GOLD_CROSSARM_DETECTIONS": nb_globals.get("GOLD_CROSSARM_DETECTIONS"),
        "GOLD_POLE_SUMMARY": nb_globals.get("GOLD_POLE_SUMMARY"),

        # ---------------------------------------------------------------------
        # Config / cache / experiments
        # ---------------------------------------------------------------------
        "CONFIG_ROOT": nb_globals.get("CONFIG_ROOT"),
        "CONFIG_PROMPTS": nb_globals.get("CONFIG_PROMPTS"),
        "CONFIG_THRESH": nb_globals.get("CONFIG_THRESH"),
        "EXPERIMENTS": nb_globals.get("EXPERIMENTS"),
        "MODEL_CACHE_DIR": nb_globals.get("MODEL_CACHE_DIR"),
        "SAM3_TASK_CONFIG_JSON": nb_globals.get("SAM3_TASK_CONFIG_JSON"),

        # ---------------------------------------------------------------------
        # Core constants
        # ---------------------------------------------------------------------
        # IMAGE_EXTS and IMAGE_EXTS_LOWER are cast to list for JSON compatibility.
        "SEED": nb_globals.get("SEED"),
        "IMAGE_EXTS": list(nb_globals.get("IMAGE_EXTS", [])),
        "IMAGE_EXTS_LOWER": list(nb_globals.get("IMAGE_EXTS_LOWER", [])),

        # ---------------------------------------------------------------------
        # SAM3 config
        # ---------------------------------------------------------------------
        "SAM3_MODEL_ID": nb_globals.get("SAM3_MODEL_ID"),
        "GLOBAL_TEXT_SCORE_THRESHOLD": nb_globals.get("GLOBAL_TEXT_SCORE_THRESHOLD"),
        "GLOBAL_MASK_THRESHOLD": nb_globals.get("GLOBAL_MASK_THRESHOLD"),
        "MAX_IMAGES_FOR_DEBUG_GALLERY": nb_globals.get("MAX_IMAGES_FOR_DEBUG_GALLERY"),
        "SAM3_TASK_CONFIG": nb_globals.get("SAM3_TASK_CONFIG"),
        "POLE_PROMPTS": nb_globals.get("POLE_PROMPTS"),
        "CROSSARM_PROMPTS": nb_globals.get("CROSSARM_PROMPTS"),

        # ---------------------------------------------------------------------
        # Runtime info
        # ---------------------------------------------------------------------
        "DEVICE": nb_globals.get("DEVICE"),
        "MODEL_DTYPE_NAME": str(nb_globals.get("MODEL_DTYPE")),
    }

    # -------------------------------------------------------------------------
    # 5. Merge in any additional caller-supplied config
    # -------------------------------------------------------------------------
    if isinstance(config_extra, dict):
        cfg.update(config_extra)

    # -------------------------------------------------------------------------
    # 6. Save config snapshot to JSON
    # -------------------------------------------------------------------------
    cfg_path = os.path.join(state_dir, "config.json")
    with open(cfg_path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2, default=str)

    # -------------------------------------------------------------------------
    # 7. Save selected DataFrames to parquet
    # -------------------------------------------------------------------------
    saved = []

    for name in df_names:
        obj = nb_globals.get(name)

        if isinstance(obj, pd.DataFrame):
            out_path = os.path.join(df_dir, f"{name}.parquet")
            obj.to_parquet(out_path, index=False)
            saved.append((name, len(obj)))

    # -------------------------------------------------------------------------
    # 8. Print save summary
    # -------------------------------------------------------------------------
    print(f"Saved config -> {cfg_path}")

    if saved:
        print("Saved DataFrames:")
        for name, n_rows in saved:
            print(f"  - {name}: {n_rows} rows")
    else:
        print("No DataFrames were found to save.")

    return cfg_path


# =============================================================================
# RESTORE STATE FUNCTION
# =============================================================================
def restore_state(nb_globals=None):
    """
    Restore saved config values and DataFrames back into notebook globals.

    Args:
        nb_globals:
            Optional namespace dictionary.
            Defaults to the current notebook globals.

    Returns:
        None
    """

    # -------------------------------------------------------------------------
    # 1. Resolve the notebook namespace
    # -------------------------------------------------------------------------
    if nb_globals is None:
        nb_globals = globals()

    # -------------------------------------------------------------------------
    # 2. Resolve required directories safely
    # -------------------------------------------------------------------------
    state_dir = nb_globals.get("STATE_DIR")
    df_dir = nb_globals.get("DF_DIR")

    if not state_dir:
        raise NameError(
            "STATE_DIR is not defined. Please run Cell 2 first before restore_state()."
        )

    if not df_dir:
        raise NameError(
            "DF_DIR is not defined. Please run Cell 2 first before restore_state()."
        )

    # -------------------------------------------------------------------------
    # 3. Locate the saved config snapshot
    # -------------------------------------------------------------------------
    cfg_path = os.path.join(state_dir, "config.json")

    # -------------------------------------------------------------------------
    # 4. Restore config values into notebook globals
    # -------------------------------------------------------------------------
    # IMPORTANT:
    # - this restores Python variables only
    # - it does NOT reload the SAM3 model into memory
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)

        for k, v in cfg.items():
            # -------------------------------------------------------------
            # JSON-compatible lists -> tuples for extension handling
            # -------------------------------------------------------------
            if k in {"IMAGE_EXTS", "IMAGE_EXTS_LOWER"} and isinstance(v, list):
                nb_globals[k] = tuple(v)

            # -------------------------------------------------------------
            # Reconstruct MODEL_DTYPE from the saved string when possible
            # -------------------------------------------------------------
            elif k == "MODEL_DTYPE_NAME":
                nb_globals[k] = v

                if "torch" in nb_globals:
                    if v == "torch.float16":
                        nb_globals["MODEL_DTYPE"] = torch.float16
                    elif v == "torch.float32":
                        nb_globals["MODEL_DTYPE"] = torch.float32
                    else:
                        # Fallback: preserve string if exact dtype mapping is unknown
                        nb_globals["MODEL_DTYPE"] = v
                else:
                    nb_globals["MODEL_DTYPE"] = v

            else:
                nb_globals[k] = v

        print(f"Restored config keys: {sorted(cfg.keys())}")
    else:
        print("No config.json found.")

    # -------------------------------------------------------------------------
    # 5. Discover saved parquet DataFrames
    # -------------------------------------------------------------------------
    df_files = sorted(glob.glob(os.path.join(df_dir, "*.parquet")))

    if not df_files:
        print("No saved DataFrames found.")
        return

    # -------------------------------------------------------------------------
    # 6. Reload each parquet file into the notebook namespace
    # -------------------------------------------------------------------------
    for fp in df_files:
        name = os.path.splitext(os.path.basename(fp))[0]
        nb_globals[name] = pd.read_parquet(fp)
        print(f"  Restored {name}: {len(nb_globals[name])} rows")


# =============================================================================
# READY MESSAGE
# =============================================================================
print(f"State functions ready. Workspace: {globals().get('WORK_DIR')}")
print(f"Standard project ZIP path : {globals().get('PROJECT_ZIP_PATH')}")
print(f"Uploaded ZIP display name : {globals().get('UPLOADED_ZIP_DISPLAY_NAME')}")

# 4) DIRECTORY CHECK

In [ ]:
# =============================================================================
# CELL 4 — GLOBAL DIRECTORY CHECK
# =============================================================================
# EXPLANATION:
# This cell verifies that the directory structure defined in Cell 2 still exists
# and that the notebook is using one consistent workspace layout.
#
# WHY THIS MATTERS:
# Later cells will try to save:
# - raw image manifests
# - selected pole ROI crops
# - asset prompt-run outputs
# - crossarm candidate / processing / review outputs
# - final Gold detections and summaries
#
# If any required folder is missing, later steps may fail in confusing ways.
# This cell gives an early filesystem sanity check before model loading,
# inference, or downstream processing.
#
# IMPORTANT:
# - This cell assumes Cell 2 has already been run
# =============================================================================

# -----------------------------------------------------------------------------
# 0. Safety check: confirm Cell 2 has already run
# -----------------------------------------------------------------------------
# EXPLANATION:
# We check the key variables needed to perform directory validation.
# If any are missing, the notebook state is incomplete and this cell should stop
# early with a clear error.
# -----------------------------------------------------------------------------
required_globals = [
    # -------------------------------------------------------------------------
    # Core workspace / state folders
    # -------------------------------------------------------------------------
    "WORK_DIR",
    "STATE_DIR",
    "DF_DIR",
    "ART_DIR",

    # -------------------------------------------------------------------------
    # Bronze folders
    # -------------------------------------------------------------------------
    "BRONZE_ROOT",
    "BRONZE_INPUT_ZIPS",
    "BRONZE_SOURCE_IMAGES",

    # -------------------------------------------------------------------------
    # Silver folders
    # -------------------------------------------------------------------------
    "SILVER_ROOT",
    "SILVER_POLE_ROIS",

    "SILVER_ASSET_DETECTION_CANDIDATES",
    "SILVER_ASSET_PROMPT_RUNS",
    "SILVER_ASSET_OVERLAYS",
    "SILVER_ASSET_MASKS",
    "SILVER_ASSET_CROPS",
    "SILVER_ASSET_REVIEW_EXPORTS",

    "SILVER_CROSSARM_DETECTION",
    "SILVER_CROSSARM_CANDIDATES",
    "SILVER_CROSSARM_PROCESSING",
    "SILVER_CROSSARM_REVIEW",

    # -------------------------------------------------------------------------
    # Gold folders
    # -------------------------------------------------------------------------
    "GOLD_ROOT",
    "GOLD_ASSET_DETECTIONS",
    "GOLD_CROSSARM_DETECTIONS",
    "GOLD_POLE_SUMMARY",

    # -------------------------------------------------------------------------
    # Config / experiment / cache folders
    # -------------------------------------------------------------------------
    "CONFIG_ROOT",
    "CONFIG_PROMPTS",
    "CONFIG_THRESH",
    "EXPERIMENTS",
    "MODEL_CACHE_DIR",

    # -------------------------------------------------------------------------
    # ZIP / config tracking
    # -------------------------------------------------------------------------
    "PROJECT_ZIP_PATH",
    "SAM3_TASK_CONFIG_JSON",
]

missing_globals = [name for name in required_globals if name not in globals()]

if missing_globals:
    raise NameError(
        "Cell 4 cannot run because some required variables are missing.\n"
        "Please run Cell 2 first.\n"
        f"Missing globals: {missing_globals}"
    )

# -----------------------------------------------------------------------------
# 1. Build the list of directories that should already exist
# -----------------------------------------------------------------------------
# EXPLANATION:
# These are the directories Cell 2 is responsible for creating. Cell 4 only
# checks whether they still exist on disk.
# -----------------------------------------------------------------------------
required_dirs = [
    # -------------------------------------------------------------------------
    # Core workspace / state folders
    # -------------------------------------------------------------------------
    WORK_DIR,
    STATE_DIR,
    DF_DIR,
    ART_DIR,

    # -------------------------------------------------------------------------
    # Bronze folders
    # -------------------------------------------------------------------------
    BRONZE_ROOT,
    BRONZE_INPUT_ZIPS,
    BRONZE_SOURCE_IMAGES,

    # -------------------------------------------------------------------------
    # Silver shared ROI layer
    # -------------------------------------------------------------------------
    SILVER_ROOT,
    SILVER_POLE_ROIS,

    # -------------------------------------------------------------------------
    # Silver branch 1: asset detection candidates
    # -------------------------------------------------------------------------
    SILVER_ASSET_DETECTION_CANDIDATES,
    SILVER_ASSET_PROMPT_RUNS,
    SILVER_ASSET_OVERLAYS,
    SILVER_ASSET_MASKS,
    SILVER_ASSET_CROPS,
    SILVER_ASSET_REVIEW_EXPORTS,

    # -------------------------------------------------------------------------
    # Silver branch 2: crossarm detection
    # -------------------------------------------------------------------------
    SILVER_CROSSARM_DETECTION,
    SILVER_CROSSARM_CANDIDATES,
    SILVER_CROSSARM_PROCESSING,
    SILVER_CROSSARM_REVIEW,

    # -------------------------------------------------------------------------
    # Gold folders
    # -------------------------------------------------------------------------
    GOLD_ROOT,
    GOLD_ASSET_DETECTIONS,
    GOLD_CROSSARM_DETECTIONS,
    GOLD_POLE_SUMMARY,

    # -------------------------------------------------------------------------
    # Config / experiment / cache folders
    # -------------------------------------------------------------------------
    CONFIG_ROOT,
    CONFIG_PROMPTS,
    CONFIG_THRESH,
    EXPERIMENTS,
    MODEL_CACHE_DIR,
]

# -----------------------------------------------------------------------------
# 2. Detect missing directories
# -----------------------------------------------------------------------------
# EXPLANATION:
# If any required path is missing on disk, collect it here so the report is
# easy to read.
# -----------------------------------------------------------------------------
missing_dirs = [d for d in required_dirs if not os.path.isdir(d)]

# -----------------------------------------------------------------------------
# 3. Print directory validation result
# -----------------------------------------------------------------------------
# EXPLANATION:
# This gives a quick pass / fail style summary before showing more detail.
# -----------------------------------------------------------------------------
print("=" * 80)
print("DIRECTORY VALIDATION")
print("=" * 80)

if missing_dirs:
    print("WARNING: The following required directories are missing:")
    for d in missing_dirs:
        print(f"  - {d}")
else:
    print("All key directories exist and are consistent.")

# -----------------------------------------------------------------------------
# 4. ZIP file status check
# -----------------------------------------------------------------------------
# EXPLANATION:
# PROJECT_ZIP_PATH is the one stable internal ZIP location used by the current
# pipeline.
#
# UPLOADED_ZIP_DISPLAY_NAME is just the original filename selected during the
# Colab upload step. It may be missing if Cell 2b has not been run yet.
# -----------------------------------------------------------------------------
print("\nZIP file summary:")

if os.path.isfile(PROJECT_ZIP_PATH):
    print(f"  PROJECT_ZIP_PATH         : {PROJECT_ZIP_PATH}")
else:
    print(f"  PROJECT_ZIP_PATH         : MISSING -> {PROJECT_ZIP_PATH}")

print(f"  UPLOADED_ZIP_DISPLAY_NAME: {globals().get('UPLOADED_ZIP_DISPLAY_NAME')}")

# -----------------------------------------------------------------------------
# 5. Bronze extracted image folder status
# -----------------------------------------------------------------------------
# EXPLANATION:
# This helps confirm whether Cell 2b has likely been run and whether extracted
# raw source images are present in Bronze/source_images.
# -----------------------------------------------------------------------------
print("\nBronze source image folder summary:")

if os.path.isdir(BRONZE_SOURCE_IMAGES):
    bronze_items = sorted(os.listdir(BRONZE_SOURCE_IMAGES))
    print("  BRONZE_SOURCE_IMAGES exists : Yes")
    print(f"  Top-level item count        : {len(bronze_items)}")
else:
    print("  BRONZE_SOURCE_IMAGES exists : No")

# -----------------------------------------------------------------------------
# 6. images_df status check
# -----------------------------------------------------------------------------
# EXPLANATION:
# This helps confirm whether the raw image manifest table already exists in
# memory from Cell 2b.
# -----------------------------------------------------------------------------
print("\nDataFrame summary:")

images_df_obj = globals().get("images_df")

if isinstance(images_df_obj, pd.DataFrame):
    print("  images_df exists           : Yes")
    print(f"  images_df rows             : {len(images_df_obj)}")
    print(f"  images_df columns          : {list(images_df_obj.columns)}")
else:
    print("  images_df exists           : No")

# -----------------------------------------------------------------------------
# 7. Optional state/config file status check
# -----------------------------------------------------------------------------
# EXPLANATION:
# These checks help confirm whether Cell 2 and Cell 3 have already written
# their expected config files.
# -----------------------------------------------------------------------------
print("\nConfig / state file summary:")

if os.path.isfile(SAM3_TASK_CONFIG_JSON):
    print(f"  SAM3_TASK_CONFIG_JSON      : {SAM3_TASK_CONFIG_JSON}")
else:
    print(f"  SAM3_TASK_CONFIG_JSON      : MISSING -> {SAM3_TASK_CONFIG_JSON}")

state_config_path = os.path.join(STATE_DIR, "config.json")
if os.path.isfile(state_config_path):
    print(f"  saved state config         : {state_config_path}")
else:
    print(f"  saved state config         : not found yet")

# -----------------------------------------------------------------------------
# 8. Print readable summary of the active project structure
# -----------------------------------------------------------------------------
# EXPLANATION:
# This is a compact, human-readable snapshot of the current folder layout being
# used by the notebook.
# -----------------------------------------------------------------------------
print("\nDirectory summary:")
print(f"  WORK_DIR                         : {WORK_DIR}")
print(f"  STATE_DIR                        : {STATE_DIR}")
print(f"  DF_DIR                           : {DF_DIR}")
print(f"  ART_DIR                          : {ART_DIR}")

print(f"  BRONZE_ROOT                      : {BRONZE_ROOT}")
print(f"  BRONZE_INPUT_ZIPS                : {BRONZE_INPUT_ZIPS}")
print(f"  BRONZE_SOURCE_IMAGES             : {BRONZE_SOURCE_IMAGES}")

print(f"  SILVER_ROOT                      : {SILVER_ROOT}")
print(f"  SILVER_POLE_ROIS                 : {SILVER_POLE_ROIS}")

print(f"  SILVER_ASSET_DETECTION_CANDIDATES: {SILVER_ASSET_DETECTION_CANDIDATES}")
print(f"  SILVER_ASSET_PROMPT_RUNS         : {SILVER_ASSET_PROMPT_RUNS}")
print(f"  SILVER_ASSET_OVERLAYS            : {SILVER_ASSET_OVERLAYS}")
print(f"  SILVER_ASSET_MASKS               : {SILVER_ASSET_MASKS}")
print(f"  SILVER_ASSET_CROPS               : {SILVER_ASSET_CROPS}")
print(f"  SILVER_ASSET_REVIEW_EXPORTS      : {SILVER_ASSET_REVIEW_EXPORTS}")

print(f"  SILVER_CROSSARM_DETECTION        : {SILVER_CROSSARM_DETECTION}")
print(f"  SILVER_CROSSARM_CANDIDATES       : {SILVER_CROSSARM_CANDIDATES}")
print(f"  SILVER_CROSSARM_PROCESSING       : {SILVER_CROSSARM_PROCESSING}")
print(f"  SILVER_CROSSARM_REVIEW           : {SILVER_CROSSARM_REVIEW}")

print(f"  GOLD_ROOT                        : {GOLD_ROOT}")
print(f"  GOLD_ASSET_DETECTIONS            : {GOLD_ASSET_DETECTIONS}")
print(f"  GOLD_CROSSARM_DETECTIONS         : {GOLD_CROSSARM_DETECTIONS}")
print(f"  GOLD_POLE_SUMMARY                : {GOLD_POLE_SUMMARY}")

print(f"  CONFIG_ROOT                      : {CONFIG_ROOT}")
print(f"  CONFIG_PROMPTS                   : {CONFIG_PROMPTS}")
print(f"  CONFIG_THRESH                    : {CONFIG_THRESH}")

print(f"  EXPERIMENTS                      : {EXPERIMENTS}")
print(f"  MODEL_CACHE_DIR                  : {MODEL_CACHE_DIR}")

# 5) HUGGING FACE LOGIN AND SAM3 MODEL DOWNLOAD

In [ ]:
# =============================================================================
# CELL 5a — HF LOGIN & GATED ACCESS TEST
# =============================================================================
# EXPLANATION:
# This cell authenticates to Hugging Face and confirms that the current account
# can access the gated "facebook/sam3" repository.
#
# WHY THIS CELL EXISTS:
# - SAM3 access is gated, so package installation alone is not enough
# - we want to fail early if token access is missing
# - we do NOT load model weights here; this is auth/access validation only
#
# PIPELINE ALIGNMENT:
# - Cell 2 defines SAM3_MODEL_ID and MODEL_CACHE_DIR
# - Cell 3 defines save_state()
# - This cell verifies auth and access before Cell 5b loads the model
# =============================================================================
# 0. Safety checks
# -----------------------------------------------------------------------------
# These variables should already exist from Cell 2.
required_globals = [
    "SAM3_MODEL_ID",
    "MODEL_CACHE_DIR",
]

missing_globals = [name for name in required_globals if name not in globals()]

if missing_globals:
    raise NameError(
        "Cell 5a cannot run because some required variables are missing.\n"
        "Please run Cell 2 first.\n"
        f"Missing globals: {missing_globals}"
    )

# -----------------------------------------------------------------------------
# 1. Try environment variables first
# -----------------------------------------------------------------------------
token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
token_source = "environment"

# -----------------------------------------------------------------------------
# 2. Try Colab secrets if not found
# -----------------------------------------------------------------------------
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        token_source = "colab_userdata"
    except Exception:
        token = None

# -----------------------------------------------------------------------------
# 3. Hidden manual prompt if still missing
# -----------------------------------------------------------------------------
if not token:
    token = getpass("Paste your HF token (hf_...): ").strip()
    token_source = "manual_getpass"

# -----------------------------------------------------------------------------
# 4. Basic token validation
# -----------------------------------------------------------------------------
if not token:
    raise ValueError("No Hugging Face token was provided.")

if not isinstance(token, str) or not token.startswith("hf_"):
    raise ValueError("Token does not look like a valid Hugging Face token.")

# -----------------------------------------------------------------------------
# 5. Store token for downstream cells
# -----------------------------------------------------------------------------
os.environ["HF_TOKEN"] = token
os.environ["HUGGINGFACE_HUB_TOKEN"] = token

# -----------------------------------------------------------------------------
# 6. Login
# -----------------------------------------------------------------------------
login(token=token, add_to_git_credential=False)

# -----------------------------------------------------------------------------
# 7. Verify identity
# -----------------------------------------------------------------------------
api = HfApi(token=token)
me = api.whoami()

HF_AUTHENTICATED_USER = me.get("name") or me.get("email") or str(me)
HF_TOKEN_SOURCE = token_source

print(f"Logged in as: {HF_AUTHENTICATED_USER}")
print(f"Token source: {HF_TOKEN_SOURCE}")

# -----------------------------------------------------------------------------
# 8. Gated repo access smoke test
# -----------------------------------------------------------------------------
# We download a small known file from the gated repo to confirm that:
# - the token is valid
# - the account has been approved for facebook/sam3
# - the local model cache path is working
try:
    SAM3_CONFIG_PATH = hf_hub_download(
        repo_id=SAM3_MODEL_ID,
        filename="config.json",
        token=token,
        cache_dir=MODEL_CACHE_DIR,
    )

    print("Gated file download OK.")
    print(f"SAM3 config accessible: {SAM3_CONFIG_PATH}")

except Exception as e:
    raise RuntimeError(
        f"Hugging Face login succeeded, but gated access to '{SAM3_MODEL_ID}' failed: {e}"
    )

# -----------------------------------------------------------------------------
# 9. Persist useful auth/access metadata
# -----------------------------------------------------------------------------
if "save_state" in globals():
    save_state(
        df_names=[],
        config_extra={
            "HF_AUTHENTICATED_USER": HF_AUTHENTICATED_USER,
            "HF_TOKEN_SOURCE": HF_TOKEN_SOURCE,
            "SAM3_CONFIG_PATH": SAM3_CONFIG_PATH,
        },
        nb_globals=globals(),
    )

In [ ]:
# =============================================================================
# CELL 5b — LOAD HF SAM3 MODEL + PROCESSOR
# =============================================================================
# EXPLANATION:
# Cell 5a already handled:
# - Hugging Face login
# - gated access validation
# - confirmation that the SAM3 repo can be reached
#
# This cell now loads the actual SAM3 model + processor into memory so later
# cells can run text-prompt-based crossarm masking on the uploaded images.
#
# OUTPUTS:
#   - DEVICE
#   - MODEL_DTYPE
#   - model
#   - processor
#   - SAM3_READY
#
# WHY THIS MATTERS:
# - Cell 5a proves access is valid
# - this cell does the heavier step of loading the model weights
# - later inference cells will use:
#     - model
#     - processor
#     - DEVICE
#
# DESIGN NOTES:
# - uses MODEL_CACHE_DIR from Cell 2
# - uses the HF token already stored by Cell 5a
# - safe to rerun because it clears old model objects first
# - saves a config snapshot using the updated Cell 3 save_state() signature
# =============================================================================
# 0. Safety checks
# -----------------------------------------------------------------------------
# These variables should already exist if the earlier cells ran correctly.
required_globals = [
    "SAM3_MODEL_ID",
    "MODEL_CACHE_DIR",
    "save_state",
]

missing_globals = [name for name in required_globals if name not in globals()]

if missing_globals:
    raise NameError(
        "Cell 5b cannot run because some required variables are missing.\n"
        "Please make sure Cells 2, 3, and 5a have already run.\n"
        f"Missing globals: {missing_globals}"
    )

# -----------------------------------------------------------------------------
# 1. Resolve HF token from environment
# -----------------------------------------------------------------------------
# Cell 5a stored the token in the process environment so downstream cells can
# reuse it without asking again.
token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")

if not token:
    raise ValueError(
        "HF token not found in environment.\n"
        "Please run Cell 5a first."
    )

# -----------------------------------------------------------------------------
# 2. Clean up any previously loaded SAM3 objects
# -----------------------------------------------------------------------------
# This makes the cell safe to rerun.
#
# WHY THIS MATTERS:
# - if you rerun the cell without cleanup, old objects may still occupy memory
# - clearing them first reduces the chance of GPU OOM issues
if "model" in globals():
    try:
        del model
    except Exception:
        pass

if "processor" in globals():
    try:
        del processor
    except Exception:
        pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# -----------------------------------------------------------------------------
# 3. Resolve runtime device and dtype
# -----------------------------------------------------------------------------
# DEVICE:
# - cuda if GPU is available
# - cpu otherwise
#
# MODEL_DTYPE:
# - float16 on GPU for lower memory usage / faster inference
# - float32 on CPU for safer compatibility
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("Preparing SAM3 load...")
print(f"  SAM3_MODEL_ID   : {SAM3_MODEL_ID}")
print(f"  MODEL_CACHE_DIR : {MODEL_CACHE_DIR}")
print(f"  DEVICE          : {DEVICE}")
print(f"  MODEL_DTYPE     : {MODEL_DTYPE}")

# -----------------------------------------------------------------------------
# 4. Load processor
# -----------------------------------------------------------------------------
# The processor handles text/image preparation and later post-processing.
print("\nLoading SAM3 processor...")

processor = Sam3Processor.from_pretrained(
    SAM3_MODEL_ID,
    token=token,
    cache_dir=MODEL_CACHE_DIR,
)

print("Processor loaded.")

# -----------------------------------------------------------------------------
# 5. Load model weights
# -----------------------------------------------------------------------------
# low_cpu_mem_usage=True helps reduce RAM pressure during loading.
#
# After loading:
# - move model to DEVICE
# - switch to eval() because this is inference, not training
print("Loading SAM3 model weights...")

model = Sam3Model.from_pretrained(
    SAM3_MODEL_ID,
    token=token,
    cache_dir=MODEL_CACHE_DIR,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)

model = model.to(DEVICE)
model.eval()

# -----------------------------------------------------------------------------
# 6. Runtime / hardware summary
# -----------------------------------------------------------------------------
SAM3_READY = True

print("\nSAM3 load complete.")

if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    alloc_gb = torch.cuda.memory_allocated() / 1e9
    reserved_gb = torch.cuda.memory_reserved() / 1e9

    print("\nGPU summary:")
    print(f"  GPU name        : {gpu_name}")
    print(f"  Total VRAM      : {vram_gb:.2f} GB")
    print(f"  Allocated VRAM  : {alloc_gb:.2f} GB")
    print(f"  Reserved VRAM   : {reserved_gb:.2f} GB")
else:
    print("\nRunning on CPU.")

print("\nSAM3 objects ready:")
print(f"  DEVICE          : {DEVICE}")
print(f"  MODEL_DTYPE     : {MODEL_DTYPE}")
print(f"  SAM3_READY      : {SAM3_READY}")

# -----------------------------------------------------------------------------
# 7. Persist runtime snapshot
# -----------------------------------------------------------------------------
# Save the key runtime values so a later restore has the updated state.
#
# NOTE:
# - this does NOT save model weights to your notebook state
# - it only saves metadata / config
save_state(
    df_names=["images_df"] if isinstance(globals().get("images_df"), pd.DataFrame) else [],
    config_extra={
        "DEVICE": DEVICE,
        "SAM3_DEVICE": DEVICE,
        "SAM3_DTYPE": str(MODEL_DTYPE),
        "SAM3_READY": SAM3_READY,
        "SAM3_MODEL_ID": SAM3_MODEL_ID,
        "MODEL_CACHE_DIR": MODEL_CACHE_DIR,
    },
    nb_globals=globals(),
)

print("\nCell 5b finished successfully.")

# 6a and b) IMAGE PREP FOR POLE INSPECTION

In [ ]:
# =============================================================================
# CELL 6a — PREPARE IMAGE DATAFRAME FOR POLE DETECTION / DEBUG
# =============================================================================
# EXPLANATION:
# This cell does NOT run SAM3 yet.
#
# It prepares the uploaded Bronze image manifest for the next stage of the
# pipeline, which is pole detection and later ROI generation.
#
# PURPOSE:
# - start from images_df created in Cell 2b
# - validate that the required columns exist
# - create a clean working copy
# - ensure useful helper columns are present
# - sort images into a stable processing order
# - optionally select a smaller subset for debug work
#
# WHY THIS MATTERS:
# Later cells should not work directly on the raw images_df if we want:
# - safer experimentation
# - easier debugging
# - reproducible image order
# - smaller debug subsets when testing pole detection
#
# OUTPUTS:
# - run_images_df   : full cleaned working image table
# - debug_images_df : optional small subset for debug testing
# =============================================================================

# -----------------------------------------------------------------------------
# 0. Safety checks
# -----------------------------------------------------------------------------
# images_df should already exist from Cell 2b.
if "images_df" not in globals():
    raise NameError(
        "images_df not found.\n"
        "Please run Cell 2b first."
    )

if not isinstance(images_df, pd.DataFrame):
    raise TypeError(
        "images_df exists but is not a pandas DataFrame."
    )

# -----------------------------------------------------------------------------
# 1. Check that the minimum required column exists
# -----------------------------------------------------------------------------
# For this pipeline, image_path is the essential column because later cells
# use it to load the raw source image from disk.
required_cols = ["image_path"]
missing_cols = [c for c in required_cols if c not in images_df.columns]

if missing_cols:
    raise ValueError(
        f"images_df is missing required columns: {missing_cols}"
    )

# -----------------------------------------------------------------------------
# 2. Make a working copy
# -----------------------------------------------------------------------------
# We do NOT want to mutate the raw Bronze manifest from Cell 2b directly.
run_images_df = images_df.copy()

# -----------------------------------------------------------------------------
# 3. Ensure helper columns exist
# -----------------------------------------------------------------------------
if "file_name" not in run_images_df.columns:
    run_images_df["file_name"] = run_images_df["image_path"].map(os.path.basename)

if "stem" not in run_images_df.columns:
    run_images_df["stem"] = run_images_df["file_name"].map(
        lambda x: os.path.splitext(x)[0] if isinstance(x, str) else None
    )

if "ext" not in run_images_df.columns:
    run_images_df["ext"] = run_images_df["file_name"].map(
        lambda x: os.path.splitext(x)[1] if isinstance(x, str) else None
    )

# -----------------------------------------------------------------------------
# 4. Sort into a stable processing order
# -----------------------------------------------------------------------------
sort_cols = [c for c in ["file_name", "image_path"] if c in run_images_df.columns]
run_images_df = run_images_df.sort_values(sort_cols).reset_index(drop=True)

# -----------------------------------------------------------------------------
# 5. Create a stable unique image_id if one does not already exist
# -----------------------------------------------------------------------------
if "image_id" not in run_images_df.columns:
    run_images_df["image_id"] = (
        run_images_df["stem"].fillna("image").astype(str)
        + "_"
        + run_images_df.index.astype(str)
    )

# -----------------------------------------------------------------------------
# 6. Optional: build a smaller debug subset
# -----------------------------------------------------------------------------
# Uses the global debug-gallery limit if available; otherwise falls back to 6.
NUM_DEBUG_IMAGES = int(globals().get("MAX_IMAGES_FOR_DEBUG_GALLERY", 6))

debug_images_df = run_images_df.head(NUM_DEBUG_IMAGES).copy()

# -----------------------------------------------------------------------------
# 7. Print summary
# -----------------------------------------------------------------------------
print("Image manifest preparation complete.\n")

print("Full working DataFrame:")
print(f"  run_images_df rows   : {len(run_images_df)}")
print(f"  run_images_df cols   : {list(run_images_df.columns)}")

print("\nDebug subset:")
print(f"  debug_images_df rows : {len(debug_images_df)}")

# -----------------------------------------------------------------------------
# 8. Preview tables
# -----------------------------------------------------------------------------
print("\nrun_images_df preview:")
try:
    display(run_images_df.head(10))
except NameError:
    print(run_images_df.head(10))

print("\ndebug_images_df preview:")
try:
    display(debug_images_df)
except NameError:
    print(debug_images_df)

In [ ]:
# =============================================================================
# CELL 6b — LOAD AND INSPECT ONE DEBUG IMAGE
# =============================================================================
# EXPLANATION:
# This cell performs a single-image sanity check using the debug image subset
# created in Cell 6a.
#
# PURPOSE:
# - confirm that debug_images_df exists and is non-empty
# - confirm that the selected image file actually exists on disk
# - load exactly one image using PIL
# - print key metadata for inspection
# - display the image inline
# - confirm that the image is in RGB mode (or convert it if not)
#
# WHY THIS MATTERS:
# Before running pole detection, we want to verify that:
# - the image paths from Cell 2b / 6a are valid
# - the files were extracted correctly
# - the image can be opened without error
# - the image is ready for downstream model use
#
# IMPORTANT:
# - this is observation only
# - no resizing
# - no model inference
# - no loop over all images
# =============================================================================

# -----------------------------------------------------------------------------
# 0. Safety checks
# -----------------------------------------------------------------------------
# EXPLANATION:
# debug_images_df should already exist from Cell 6a and should contain a small,
# stable subset of images for early debugging.
# -----------------------------------------------------------------------------
if "debug_images_df" not in globals():
    raise NameError(
        "debug_images_df not found.\n"
        "Please run Cell 6a first."
    )

if not isinstance(debug_images_df, pd.DataFrame):
    raise TypeError(
        "debug_images_df exists but is not a pandas DataFrame."
    )

if debug_images_df.empty:
    raise ValueError(
        "debug_images_df exists but is empty.\n"
        "Please check Cell 6a."
    )

# -----------------------------------------------------------------------------
# 1. Choose exactly one debug row
# -----------------------------------------------------------------------------
# EXPLANATION:
# Keep this as a single-image check only so the output stays easy to inspect.
# -----------------------------------------------------------------------------
DEBUG_ROW_INDEX = 0

if DEBUG_ROW_INDEX >= len(debug_images_df):
    raise IndexError(
        f"DEBUG_ROW_INDEX={DEBUG_ROW_INDEX} is out of range for "
        f"debug_images_df with {len(debug_images_df)} rows."
    )

row = debug_images_df.iloc[DEBUG_ROW_INDEX]

# -----------------------------------------------------------------------------
# 2. Extract metadata from the selected row
# -----------------------------------------------------------------------------
# EXPLANATION:
# Use safe fallbacks in case helper fields are missing or null.
# -----------------------------------------------------------------------------
image_path = row["image_path"]

image_id = row.get("image_id", None)
if pd.isna(image_id):
    image_id = None

file_name = row.get("file_name", None)
if pd.isna(file_name) or not isinstance(file_name, str) or len(file_name.strip()) == 0:
    file_name = os.path.basename(image_path)

# -----------------------------------------------------------------------------
# 3. Confirm the file exists before opening it
# -----------------------------------------------------------------------------
if not isinstance(image_path, str) or len(image_path.strip()) == 0:
    raise ValueError(
        f"Selected image_path is invalid: {image_path}"
    )

if not os.path.exists(image_path):
    raise FileNotFoundError(
        "The selected image file does not exist on disk.\n"
        f"image_path: {image_path}\n\n"
        "This likely means Cell 2b extraction did not complete correctly "
        "or the extracted files were later removed."
    )

# -----------------------------------------------------------------------------
# 4. Load the image using PIL
# -----------------------------------------------------------------------------
# EXPLANATION:
# We inspect the original mode first, then convert to RGB if needed.
#
# IMPORTANT:
# We force the copied image to fully load inside the `with` block so it remains
# safe to use after the file handle is closed.
# -----------------------------------------------------------------------------
with Image.open(image_path) as img:
    original_mode = img.mode
    width, height = img.size

    # -------------------------------------------------------------------------
    # 5. Convert to RGB if needed
    # -------------------------------------------------------------------------
    if original_mode != "RGB":
        print(
            f"WARNING: Image mode is '{original_mode}', not 'RGB'. "
            "Converting to RGB for downstream compatibility."
        )
        img_rgb = img.convert("RGB")
    else:
        img_rgb = img.copy()

    # Force full decode into memory before leaving the with-block
    img_rgb.load()

# -----------------------------------------------------------------------------
# 6. Print metadata summary
# -----------------------------------------------------------------------------
print("Single-image sanity check:\n")
print(f"  DEBUG_ROW_INDEX : {DEBUG_ROW_INDEX}")
print(f"  image_id        : {image_id}")
print(f"  file_name       : {file_name}")
print(f"  image_path      : {image_path}")
print(f"  width           : {width}")
print(f"  height          : {height}")
print(f"  original_mode   : {original_mode}")
print(f"  final_mode      : {img_rgb.mode}")

# -----------------------------------------------------------------------------
# 7. Display the image inline
# -----------------------------------------------------------------------------
plt.figure(figsize=(10, 8))
plt.imshow(img_rgb)
plt.title(
    f"{file_name}\n"
    f"image_id={image_id} | size={width}x{height} | mode={img_rgb.mode}",
    fontsize=11
)
plt.axis("off")
plt.show()
plt.close()

# -----------------------------------------------------------------------------
# 8. Final confirmation
# -----------------------------------------------------------------------------
print("Cell 6b completed successfully.")
print("The selected image loaded correctly and is ready for pole detection.")

# -----------------------------------------------------------------------------
# 9. Cleanup
# -----------------------------------------------------------------------------
# EXPLANATION:
# Keep this cell purely diagnostic. If a later cell needs the image again, it
# should reload it from image_path rather than relying on notebook memory.
# -----------------------------------------------------------------------------
del img_rgb

# 7a) DEBUG - TIMBER POWER POLE DETECTION

In [ ]:
# =============================================================================
# CELL 7a — STEP BY STEP DEBUGGING PROCESS - TIMBER POWER POLE POST-PROCESSING
# =============================================================================
# EXPLANATION:
# This cell runs ONE pole prompt only: "timber power pole".
#
# It performs the following flow for exactly two debug images:
#
#   A) run raw SAM3 pole detection
#   B) collect all returned pole boxes
#   C) compute ranking features
#   D) drop weak / tiny / short / non-vertical candidates
#   E) score the remaining candidates
#   F) choose the best remaining candidate as the selected center pole
#
# IMPORTANT:
# - this cell is self-contained
# - it shows example outputs for 2 images at each stage
# - it uses your existing loaded `model` and `processor`
# - it does NOT require your old helper functions
# =============================================================================

# -----------------------------------------------------------------------------
# 0. SAFETY CHECKS
# -----------------------------------------------------------------------------
required_globals = [
    "debug_images_df",
    "model",
    "processor",
    "DEVICE",
]

# Check that all required globals exist before continuing.
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise NameError(
        "Cell 7a cannot run because some required variables are missing.\n"
        "Please make sure the earlier cells have already run.\n"
        f"Missing globals: {missing_globals}"
    )

# Confirm the debug image table is a pandas DataFrame.
if not isinstance(debug_images_df, pd.DataFrame):
    raise TypeError("debug_images_df exists but is not a pandas DataFrame.")

# Stop early if there are no debug rows to process.
if debug_images_df.empty:
    raise ValueError("debug_images_df is empty. Please check Cell 6a.")

# -----------------------------------------------------------------------------
# 1. POLE CONFIG
# -----------------------------------------------------------------------------
# EXPLANATION:
# We read the task config if it exists, but we force the active prompt
# to "timber power pole" for clarity and consistency with production.
# -----------------------------------------------------------------------------
POLE_PROMPT = "timber power pole"

# Pull pole-specific config if it exists.
POLE_TASK_CFG = {}
if "SAM3_TASK_CONFIG" in globals() and isinstance(SAM3_TASK_CONFIG, dict):
    POLE_TASK_CFG = SAM3_TASK_CONFIG.get("pole_detection", {}) or {}

# Use task-config thresholds if present, otherwise fall back to global defaults.
POLE_TEXT_THRESHOLD = float(
    POLE_TASK_CFG.get("text_score_threshold", globals().get("GLOBAL_TEXT_SCORE_THRESHOLD", 0.45))
)
POLE_MASK_THRESHOLD = float(
    POLE_TASK_CFG.get("mask_threshold", globals().get("GLOBAL_MASK_THRESHOLD", 0.50))
)

print("Pole detection config:")
print(f"  POLE_PROMPT         : {POLE_PROMPT}")
print(f"  POLE_TEXT_THRESHOLD : {POLE_TEXT_THRESHOLD}")
print(f"  POLE_MASK_THRESHOLD : {POLE_MASK_THRESHOLD}")

# -----------------------------------------------------------------------------
# 2. POST-PROCESSING CONSTANTS
# -----------------------------------------------------------------------------
# EXPLANATION:
# These are the starting rules for selecting the center pole.
#
# PREFILTERS:
# - drop weak detections
# - drop very short detections
# - drop very small detections
# - drop non-vertical detections
#
# FINAL SCORE:
# - emphasize horizontal closeness to image center
# - then emphasize pole height
# - then smaller contributions from area / confidence / edge safety
# -----------------------------------------------------------------------------
POLE_DEBUG_IMAGE_COUNT = min(2, len(debug_images_df))

POLE_MIN_SCORE = 0.25
POLE_MIN_AREA_FRAC = 0.005      # 0.5% of image area
POLE_MIN_HEIGHT_FRAC = 0.15     # 15% of image height
POLE_MIN_ASPECT = 1.80          # box_h / box_w must be at least this vertical
POLE_MAX_WIDTH_FRAC = 0.08      # max 8% of image width
# Optional absolute guard if image scale is very stable:
POLE_MAX_BOX_W_PX = 400

W_X_CENTER = 0.45
W_HEIGHT   = 0.30
W_AREA     = 0.10
W_CONF     = 0.10
W_EDGE     = 0.05

# -----------------------------------------------------------------------------
# 3. HELPER: SAFE DISPLAY
# -----------------------------------------------------------------------------
def _safe_display(obj):
    """
    Display a pandas object in a notebook if possible; otherwise print it.

    Args:
        obj:
            Object to display.

    Returns:
        None
    """
    try:
        display(obj)
    except Exception:
        print(obj)

# -----------------------------------------------------------------------------
# 4. HELPER: SAFE COLUMN SUBSET
# -----------------------------------------------------------------------------
def _existing_cols(df, cols):
    """
    Return only the requested columns that actually exist in the DataFrame.

    Args:
        df:
            Input pandas DataFrame.

        cols:
            Requested column names.

    Returns:
        List[str]:
            Existing column names only.
    """
    if df is None or not isinstance(df, pd.DataFrame):
        return []

    return [c for c in cols if c in df.columns]

# -----------------------------------------------------------------------------
# 5. HELPER: CLIP BOX TO IMAGE
# -----------------------------------------------------------------------------
def clip_box_to_image(x1, y1, x2, y2, image_w, image_h):
    """
    Clip box coordinates safely to image bounds.

    Args:
        x1, y1, x2, y2:
            Raw box coordinates.

        image_w, image_h:
            Image width and height.

    Returns:
        Tuple[float, float, float, float]:
            Clipped and ordered box coordinates.
    """
    # Clamp all coordinates so they stay within image bounds.
    x1 = float(np.clip(x1, 0, max(image_w - 1, 0)))
    y1 = float(np.clip(y1, 0, max(image_h - 1, 0)))
    x2 = float(np.clip(x2, 0, max(image_w - 1, 0)))
    y2 = float(np.clip(y2, 0, max(image_h - 1, 0)))

    # Reorder endpoints just in case the model returned inverted corners.
    x1, x2 = sorted([x1, x2])
    y1, y2 = sorted([y1, y2])

    return x1, y1, x2, y2

# -----------------------------------------------------------------------------
# 6. HELPER: NORMALIZE RAW BOXES
# -----------------------------------------------------------------------------
def normalize_raw_boxes(raw_boxes):
    """
    Normalize raw box output into a list of [x1, y1, x2, y2].

    Args:
        raw_boxes:
            Boxes from SAM3 post-processing. Can be tensor, ndarray, list, etc.

    Returns:
        List[List[float]]:
            Standardized list of boxes.
    """
    # Return an empty list if no boxes were produced.
    if raw_boxes is None:
        return []

    # Convert tensors / ndarrays to standard Python lists first.
    if isinstance(raw_boxes, torch.Tensor):
        raw_boxes = raw_boxes.detach().cpu().tolist()
    elif isinstance(raw_boxes, np.ndarray):
        raw_boxes = raw_boxes.tolist()

    # If the result is not list-like, treat it as invalid.
    if not isinstance(raw_boxes, (list, tuple)):
        return []

    # Handle the no-box case.
    if len(raw_boxes) == 0:
        return []

    # Handle the special case where a single box comes back as a flat list.
    if isinstance(raw_boxes[0], (int, float, np.number)):
        if len(raw_boxes) >= 4:
            return [[
                float(raw_boxes[0]),
                float(raw_boxes[1]),
                float(raw_boxes[2]),
                float(raw_boxes[3]),
            ]]
        return []

    boxes = []

    # Standardize each candidate box into [x1, y1, x2, y2].
    for box in raw_boxes:
        if isinstance(box, torch.Tensor):
            box = box.detach().cpu().tolist()
        elif isinstance(box, np.ndarray):
            box = box.tolist()

        if isinstance(box, (list, tuple)) and len(box) >= 4:
            boxes.append([
                float(box[0]),
                float(box[1]),
                float(box[2]),
                float(box[3]),
            ])

    return boxes

# -----------------------------------------------------------------------------
# 7. HELPER: NORMALIZE RAW SCORES
# -----------------------------------------------------------------------------
def normalize_raw_scores(raw_scores):
    """
    Normalize raw score output into a flat float list.

    Args:
        raw_scores:
            Scores from SAM3 post-processing. Can be tensor, ndarray, list, etc.

    Returns:
        List[float]:
            Standardized list of scores.
    """
    # Return an empty list if no scores were produced.
    if raw_scores is None:
        return []

    # Convert tensors / ndarrays to flat Python lists.
    if isinstance(raw_scores, torch.Tensor):
        raw_scores = raw_scores.detach().cpu().flatten().tolist()
    elif isinstance(raw_scores, np.ndarray):
        raw_scores = raw_scores.flatten().tolist()

    # If the result is not list-like, treat it as invalid.
    if not isinstance(raw_scores, (list, tuple)):
        return []

    scores = []

    # Convert every score to a Python float.
    for s in raw_scores:
        if isinstance(s, torch.Tensor):
            s = s.detach().cpu().item()
        elif isinstance(s, np.ndarray):
            s = float(np.asarray(s).reshape(-1)[0])

        scores.append(float(s))

    return scores

# -----------------------------------------------------------------------------
# 8. HELPER: BUILD STANDARDIZED RAW DETECTION DATAFRAME
# -----------------------------------------------------------------------------
def build_raw_pole_df(
    raw_boxes,
    raw_scores,
    image_id,
    file_name,
    image_path,
    image_w,
    image_h,
    prompt,
):
    """
    Convert raw SAM3 outputs into one standard DataFrame.

    Args:
        raw_boxes:
            Iterable of raw boxes.

        raw_scores:
            Iterable of raw confidence scores.

        image_id, file_name, image_path:
            Image-level metadata.

        image_w, image_h:
            Image dimensions.

        prompt:
            Prompt used for detection.

    Returns:
        pd.DataFrame:
            Standardized raw candidate table.
    """
    # Standardize the box and score outputs first.
    boxes = normalize_raw_boxes(raw_boxes)
    scores = normalize_raw_scores(raw_scores)

    # If there are no boxes at all, return an empty DataFrame.
    if len(boxes) == 0:
        return pd.DataFrame()

    # If scores are missing, fill with zeros so the rest of the pipeline still runs.
    if len(scores) == 0:
        scores = [0.0] * len(boxes)

    # Only keep the number of rows supported by both boxes and scores.
    n = min(len(boxes), len(scores))
    rows = []

    for det_idx in range(n):
        # Read and clip the raw box to the image boundary.
        x1, y1, x2, y2 = boxes[det_idx]
        x1, y1, x2, y2 = clip_box_to_image(x1, y1, x2, y2, image_w, image_h)

        # Build one standardized candidate row.
        rows.append({
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "image_w": int(image_w),
            "image_h": int(image_h),
            "prompt": prompt,
            "det_idx": int(det_idx),
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),
            "score": float(scores[det_idx]),
        })

    return pd.DataFrame(rows)

# -----------------------------------------------------------------------------
# 9. HELPER: RUN RAW "TIMBER POWER POLE" SAM3 DETECTION
# -----------------------------------------------------------------------------
def run_power_pole_raw_detections(
    image_pil,
    image_id,
    file_name,
    image_path,
    prompt,
    text_threshold,
    mask_threshold,
):
    """
    Run raw SAM3 timber power pole detection and return ALL detections.

    Args:
        image_pil:
            PIL RGB image.

        image_id, file_name, image_path:
            Image-level metadata.

        prompt:
            The text prompt to run.

        text_threshold:
            Threshold for instance post-processing.

        mask_threshold:
            Mask threshold for instance post-processing.

    Returns:
        pd.DataFrame:
            Raw candidate detections with columns:
            x1, y1, x2, y2, score, plus image metadata.
    """
    # Read the original image size for downstream box clipping and scaling.
    image_w, image_h = image_pil.size

    # Tokenize the image + text prompt for SAM3.
    inputs = processor(
        images=image_pil,
        text=prompt,
        return_tensors="pt",
    )

    # Move the full batch to the active device.
    inputs = inputs.to(DEVICE)

    # Prefer processor-provided original sizes if available.
    if "original_sizes" in inputs:
        target_sizes = inputs["original_sizes"].detach().cpu().tolist()
    else:
        target_sizes = [[image_h, image_w]]

    # Run inference without gradients.
    with torch.no_grad():
        outputs = model(**inputs)

    # Convert model outputs into instance segmentation predictions.
    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=text_threshold,
        mask_threshold=mask_threshold,
        target_sizes=target_sizes,
    )

    # Pull out the first image result safely.
    result = results[0] if isinstance(results, list) and len(results) > 0 else {}

    # Get boxes and scores from the post-processed result.
    raw_boxes = result.get("boxes", None)
    raw_scores = result.get("scores", None)

    # Convert everything into the standard raw candidate DataFrame.
    return build_raw_pole_df(
        raw_boxes=raw_boxes,
        raw_scores=raw_scores,
        image_id=image_id,
        file_name=file_name,
        image_path=image_path,
        image_w=image_w,
        image_h=image_h,
        prompt=prompt,
    )

# -----------------------------------------------------------------------------
# 10. HELPER: ADD POLE FEATURES
# -----------------------------------------------------------------------------
def add_pole_features(raw_df):
    """
    Compute ranking features for pole candidates.

    Args:
        raw_df:
            Raw pole candidate DataFrame with at least:
            x1, y1, x2, y2, score, image_w, image_h

    Returns:
        pd.DataFrame:
            Candidate table with computed features added.
    """
    df = raw_df.copy()

    # Return immediately if there is nothing to score.
    if df.empty:
        return df

    # -------------------------------------------------------------------------
    # Basic box geometry
    # -------------------------------------------------------------------------
    # Compute width, height, and area for each candidate box.
    df["box_w"] = (df["x2"] - df["x1"]).clip(lower=1.0)
    df["box_h"] = (df["y2"] - df["y1"]).clip(lower=1.0)
    df["box_area"] = df["box_w"] * df["box_h"]

    # Compute the pole box center coordinates.
    df["pole_cx"] = (df["x1"] + df["x2"]) / 2.0
    df["pole_cy"] = (df["y1"] + df["y2"]) / 2.0

    # -------------------------------------------------------------------------
    # Normalized size / shape features
    # -------------------------------------------------------------------------
    # Normalize size-based measurements against the image dimensions.
    df["image_area"] = df["image_w"] * df["image_h"]
    df["area_frac"] = df["box_area"] / df["image_area"].clip(lower=1.0)
    df["height_frac"] = df["box_h"] / df["image_h"].clip(lower=1.0)
    df["aspect_ratio"] = df["box_h"] / df["box_w"].clip(lower=1.0)
    # -------------------------------------------------------------------------
    # Normalized size / shape features
    # -------------------------------------------------------------------------
    df["image_area"] = df["image_w"] * df["image_h"]
    df["area_frac"] = df["box_area"] / df["image_area"].clip(lower=1.0)
    df["height_frac"] = df["box_h"] / df["image_h"].clip(lower=1.0)
    df["width_frac"] = df["box_w"] / df["image_w"].clip(lower=1.0)
    df["aspect_ratio"] = df["box_h"] / df["box_w"].clip(lower=1.0)

    # -------------------------------------------------------------------------
    # Horizontal center score
    # -------------------------------------------------------------------------
    # Measure how close each candidate is to the horizontal center of the image.
    image_cx = df["image_w"] / 2.0
    x_center_dist_norm = (4
    df["x_center_dist_norm"] = x_center_dist_norm
    df["x_center_score"] = 1.0 - np.clip(x_center_dist_norm, 0.0, 1.0)

    # -------------------------------------------------------------------------
    # Height / area / confidence scores
    # -------------------------------------------------------------------------
    # Scale height and area relative to the largest candidate in this image.
    max_h = max(float(df["box_h"].max()), 1.0)
    max_a = max(float(df["box_area"].max()), 1.0)

    df["height_score"] = df["box_h"] / max_h
    df["area_score"] = df["box_area"] / max_a
    df["conf_score"] = df["score"]

    # -------------------------------------------------------------------------
    # Edge safety score
    # -------------------------------------------------------------------------
    # Compute how safely away from the image border the box is.
    edge_margin = np.minimum.reduce([
        df["x1"].values,
        df["y1"].values,
        (df["image_w"] - df["x2"]).values,
        (df["image_h"] - df["y2"]).values,
    ])

    # Normalize the edge margin by 5% of the smaller image dimension.
    edge_norm_denom = 0.05 * np.minimum(df["image_w"], df["image_h"])
    edge_norm_denom = edge_norm_denom.clip(lower=1.0)

    df["edge_margin"] = edge_margin
    df["edge_score"] = np.clip(df["edge_margin"] / edge_norm_denom, 0.0, 1.0)

    return df

# -----------------------------------------------------------------------------
# 11. HELPER: APPLY PREFILTER
# -----------------------------------------------------------------------------
def apply_pole_prefilter(feature_df):
    """
    Drop obviously bad pole candidates before final scoring.

    Args:
        feature_df:
            Feature-enriched candidate table.

    Returns:
        pd.DataFrame:
            Same table with boolean keep flags added.
    """
    df = feature_df.copy()

    # Return immediately if there is nothing to filter.
    if df.empty:
        return df

    # Evaluate each prefilter rule independently for easier debugging.
    df["keep_score"] = df["score"] >= POLE_MIN_SCORE
    df["keep_area"] = df["area_frac"] >= POLE_MIN_AREA_FRAC
    df["keep_height"] = df["height_frac"] >= POLE_MIN_HEIGHT_FRAC
    df["keep_aspect"] = df["aspect_ratio"] >= POLE_MIN_ASPECT

    # NEW: reject boxes that are too wide to be a single pole.
    df["keep_width_frac"] = df["width_frac"] <= POLE_MAX_WIDTH_FRAC
    df["keep_width_px"] = df["box_w"] <= POLE_MAX_BOX_W_PX

    df["is_kept_after_prefilter"] = (
        df["keep_score"] &
        df["keep_area"] &
        df["keep_height"] &
        df["keep_aspect"] &
        df["keep_width_frac"] &
        df["keep_width_px"]
    )

    return df

# -----------------------------------------------------------------------------
# 12. HELPER: SCORE AND SELECT CENTER POLE
# -----------------------------------------------------------------------------
def score_pole_candidates(prefilter_df):
    """
    Score the remaining pole candidates and select the best center pole.

    Args:
        prefilter_df:
            Candidate table with keep flags.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]:
            - scored_df   : all candidates with final scores
            - selected_df : selected single winning row (or empty)
    """
    df = prefilter_df.copy()

    # Return empty outputs immediately if there are no rows.
    if df.empty:
        return df, df.iloc[0:0].copy()

    # Initialize default output columns for the full candidate table.
    df["selection_mode"] = "not_kept"
    df["final_score"] = np.nan
    df["is_selected_pole"] = False

    # Keep only rows that survived prefiltering.
    kept_df = df[df["is_kept_after_prefilter"]].copy()

    # Fallback: if everything was filtered out, still rank all candidates.
    if kept_df.empty:
        kept_df = df.copy()
        kept_df["selection_mode"] = "fallback_all_candidates"
    else:
        kept_df["selection_mode"] = "prefilter_kept"

    # Compute the final weighted score used for ranking.
    kept_df["final_score"] = (
        W_X_CENTER * kept_df["x_center_score"] +
        W_HEIGHT   * kept_df["height_score"] +
        W_AREA     * kept_df["area_score"] +
        W_CONF     * kept_df["conf_score"] +
        W_EDGE     * kept_df["edge_score"]
    )

    # Sort best-to-worst using final score first, then tie-breakers.
    kept_df = kept_df.sort_values(
        by=["final_score", "score", "box_h", "x_center_score"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    # Mark the top-ranked candidate as the selected pole.
    if len(kept_df) > 0:
        kept_df.loc[0, "is_selected_pole"] = True

    # Merge the ranking outputs back into the full candidate table.
    score_cols = ["image_id", "det_idx", "selection_mode", "final_score", "is_selected_pole"]
    df = df.drop(columns=["selection_mode", "final_score", "is_selected_pole"], errors="ignore")
    df = df.merge(
        kept_df[score_cols],
        on=["image_id", "det_idx"],
        how="left",
    )

    # Fill rows that were never ranked because they were not kept.
    df["selection_mode"] = df["selection_mode"].fillna("not_kept")
    df["is_selected_pole"] = df["is_selected_pole"].fillna(False)

    # Build a one-row selected table for convenience.
    selected_df = kept_df[kept_df["is_selected_pole"] == True].copy()

    return df, selected_df

# -----------------------------------------------------------------------------
# 13. HELPER: DRAW BOXES
# -----------------------------------------------------------------------------
def draw_boxes_on_axis(
    ax,
    image_rgb,
    boxes_df,
    title,
    stage="raw",          # "raw" | "kept" | "selected"
    selected_only=False,
):
    """
    Draw pole boxes with fixed stage colors and standard labeling.

    Args:
        ax:
            Matplotlib axis.

        image_rgb:
            RGB numpy image.

        boxes_df:
            DataFrame containing x1, y1, x2, y2 and optional score fields.

        title:
            Plot title.

        stage:
            One of:
                - "raw"      -> yellow
                - "kept"     -> cyan
                - "selected" -> red

        selected_only:
            If True, only selected rows are drawn.

    Returns:
        None
    """
    import matplotlib.patches as patches

    STAGE_COLORS = {
        "raw": "yellow",
        "kept": "cyan",
        "selected": "red",
    }

    STAGE_LINEWIDTHS = {
        "raw": 2.0,
        "kept": 2.5,
        "selected": 3.5,
    }

    # Resolve the display style for this stage.
    color = STAGE_COLORS.get(stage, "yellow")
    lw = STAGE_LINEWIDTHS.get(stage, 2.0)

    # Show the underlying image.
    ax.imshow(image_rgb)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

    # Nothing to draw if the candidate table is missing or empty.
    if boxes_df is None or len(boxes_df) == 0:
        return

    plot_df = boxes_df.copy()

    # Keep only the selected row if requested.
    if selected_only:
        if "is_selected_pole" in plot_df.columns:
            plot_df = plot_df[plot_df["is_selected_pole"] == True].copy()
        else:
            plot_df = plot_df.iloc[0:0].copy()

    # Exit if no rows remain after filtering.
    if plot_df.empty:
        return

    for _, r in plot_df.iterrows():
        # Read the current box coordinates.
        x1, y1, x2, y2 = r["x1"], r["y1"], r["x2"], r["y2"]
        w = x2 - x1
        h = y2 - y1

        # Draw the rectangle overlay.
        rect = patches.Rectangle(
            (x1, y1),
            w,
            h,
            linewidth=lw,
            edgecolor=color,
            facecolor="none",
        )
        ax.add_patch(rect)

        label_bits = []

        # Start the label with a fixed object type prefix.
        label_bits.append("POLE")

        # Prefer the row-level prompt if available.
        prompt_text = None
        if "prompt" in r and pd.notna(r["prompt"]) and str(r["prompt"]).strip():
            prompt_text = str(r["prompt"]).strip()
        elif "POLE_PROMPT" in globals():
            prompt_text = str(POLE_PROMPT).strip()

        if prompt_text:
            label_bits.append(prompt_text)

        # Add the raw confidence score when present.
        if "score" in r and pd.notna(r["score"]):
            label_bits.append(f"score={float(r['score']):.3f}")

        # Add the final score for selected/final overlays when present.
        if stage == "selected" and "final_score" in r and pd.notna(r["final_score"]):
            label_bits.append(f"final={float(r['final_score']):.3f}")

        label = " | ".join(label_bits)

        # Draw the text label above the box.
        if label:
            ax.text(
                x1,
                max(y1 - 6, 8),
                label,
                fontsize=8.5,
                color="white",
                bbox=dict(
                    facecolor=color,
                    alpha=0.85,
                    edgecolor="none",
                    pad=2.0,
                ),
            )

# -----------------------------------------------------------------------------
# 14. HELPER: TWO-IMAGE STEP GALLERY
# -----------------------------------------------------------------------------
def show_two_image_step_gallery(
    step_title,
    step_results,
    df_key,
    stage="raw",          # "raw" | "kept" | "selected"
    selected_only=False,
):
    """
    Show a side-by-side gallery for two images for one processing step.

    Args:
        step_title:
            Title for this processing step.

        step_results:
            List of per-image result dictionaries.

        df_key:
            Key inside each step_results item to use as the box DataFrame.

        stage:
            Controls fixed overlay color.

        selected_only:
            If True, only the selected pole is drawn.

    Returns:
        None
    """
    n = len(step_results)
    fig, axes = plt.subplots(1, n, figsize=(11 * n, 8))

    # Normalize the axis container for the single-image case.
    if n == 1:
        axes = [axes]

    for ax, item in zip(axes, step_results):
        draw_boxes_on_axis(
            ax=ax,
            image_rgb=item["image_rgb"],
            boxes_df=item[df_key],
            title=f"{item['file_name']}\n{step_title}",
            stage=stage,
            selected_only=selected_only,
        )

    plt.tight_layout()
    plt.show()
    plt.close()

# -----------------------------------------------------------------------------
# 15. CHOOSE EXACTLY TWO DEBUG IMAGES
# -----------------------------------------------------------------------------
# Take the first two debug rows for the step-by-step gallery.
pole_debug_input_df = debug_images_df.head(POLE_DEBUG_IMAGE_COUNT).copy().reset_index(drop=True)

if pole_debug_input_df.empty:
    raise ValueError("No debug images available for pole step debugging.")

print("\nUsing these debug images for pole post-processing:")

# Only display columns that actually exist.
debug_cols = _existing_cols(
    pole_debug_input_df,
    ["image_id", "file_name", "image_path"]
)
_safe_display(pole_debug_input_df[debug_cols])

# -----------------------------------------------------------------------------
# 16. RUN STEP-BY-STEP POLE PIPELINE ON TWO DEBUG IMAGES
# -----------------------------------------------------------------------------
step_results = []

all_raw_rows = []
all_feature_rows = []
all_scored_rows = []
all_selected_rows = []

for _, row in pole_debug_input_df.iterrows():
    # Use explicit pandas index checks instead of row.get(...).
    image_id = row["image_id"] if "image_id" in row.index and pd.notna(row["image_id"]) else None
    file_name = row["file_name"] if "file_name" in row.index and pd.notna(row["file_name"]) else None
    image_path = row["image_path"] if "image_path" in row.index and pd.notna(row["image_path"]) else None

    # Stop early if the image path is missing or invalid.
    if pd.isna(image_path) or not isinstance(image_path, str) or len(image_path.strip()) == 0:
        raise ValueError("A debug row is missing a valid image_path.")

    # If file_name is missing, recover it from the file path.
    if (
        pd.isna(file_name)
        or not isinstance(file_name, str)
        or len(file_name.strip()) == 0
    ):
        file_name = os.path.basename(image_path) if isinstance(image_path, str) else "unknown_file"

    print("\n" + "=" * 100)
    print(f"PROCESSING IMAGE: {file_name}")
    print("=" * 100)

    # -------------------------------------------------------------------------
    # Load RGB image
    # -------------------------------------------------------------------------
    # Open the image safely and convert it to RGB.
    with Image.open(image_path) as img:
        image_pil = img.convert("RGB")
        image_pil.load()

    # Convert the PIL image to a NumPy array for visualization.
    image_rgb = np.array(image_pil)

    # -------------------------------------------------------------------------
    # STEP A/B — RAW DETECTIONS
    # -------------------------------------------------------------------------
    # Run the raw SAM3 timber power pole prompt and collect every candidate box.
    raw_df = run_power_pole_raw_detections(
        image_pil=image_pil,
        image_id=image_id,
        file_name=file_name,
        image_path=image_path,
        prompt=POLE_PROMPT,
        text_threshold=POLE_TEXT_THRESHOLD,
        mask_threshold=POLE_MASK_THRESHOLD,
    )

    # Defensively normalize a None return to an empty DataFrame.
    if raw_df is None:
        raw_df = pd.DataFrame()

    # Enforce the expected return type.
    if not isinstance(raw_df, pd.DataFrame):
        raise TypeError(
            f"run_power_pole_raw_detections() must return a DataFrame, got: {type(raw_df)}"
        )

    # If there are no detections, keep all downstream outputs as empty DataFrames.
    if len(raw_df) == 0:
        print("No raw pole detections returned for this image.")

        # Keep empty placeholders so later steps remain structurally consistent.
        feature_df = raw_df.copy()
        prefilter_df = raw_df.copy()
        scored_df = raw_df.copy()
        selected_df = raw_df.copy()
    else:
        # ---------------------------------------------------------------------
        # STEP C — FEATURE COMPUTATION
        # ---------------------------------------------------------------------
        # Compute box geometry, center distance, area fraction, aspect ratio, etc.
        feature_df = add_pole_features(raw_df)

        # ---------------------------------------------------------------------
        # STEP D — PREFILTER
        # ---------------------------------------------------------------------
        # Mark each candidate as kept / filtered using the rule-based gates.
        prefilter_df = apply_pole_prefilter(feature_df)

        # ---------------------------------------------------------------------
        # STEP E/F — SCORE + SELECT
        # ---------------------------------------------------------------------
        # Rank the kept candidates and select the best center pole.
        scored_df, selected_df = score_pole_candidates(prefilter_df)

    # -------------------------------------------------------------------------
    # Save step-level per-image outputs
    # -------------------------------------------------------------------------
    # Store every stage so we can inspect both tables and overlays later.
    step_results.append({
        "image_id": image_id,
        "file_name": file_name,
        "image_path": image_path,
        "image_rgb": image_rgb,
        "raw_df": raw_df,
        "feature_df": feature_df,
        "prefilter_df": prefilter_df,
        "scored_df": scored_df,
        "selected_df": selected_df,
    })

    # Append non-empty outputs into global combined tables.
    if len(raw_df) > 0:
        all_raw_rows.append(raw_df)

    if len(feature_df) > 0:
        all_feature_rows.append(feature_df)

    if len(scored_df) > 0:
        all_scored_rows.append(scored_df)

    if len(selected_df) > 0:
        all_selected_rows.append(selected_df)

# -----------------------------------------------------------------------------
# 17. COMBINE OUTPUT TABLES
# -----------------------------------------------------------------------------
# Combine all raw candidate rows across both images.
pole_raw_candidates_df = (
    pd.concat(all_raw_rows, ignore_index=True)
    if len(all_raw_rows) > 0 else
    pd.DataFrame()
)

# Combine all feature rows across both images.
pole_feature_debug_df = (
    pd.concat(all_feature_rows, ignore_index=True)
    if len(all_feature_rows) > 0 else
    pd.DataFrame()
)

# Combine all scored candidate rows across both images.
pole_candidates_df = (
    pd.concat(all_scored_rows, ignore_index=True)
    if len(all_scored_rows) > 0 else
    pd.DataFrame()
)

# Combine all selected rows across both images.
pole_selection_df = (
    pd.concat(all_selected_rows, ignore_index=True)
    if len(all_selected_rows) > 0 else
    pd.DataFrame()
)

# -----------------------------------------------------------------------------
# 18. STEP OUTPUTS — TABLES
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("STEP A/B — RAW DETECTIONS TABLE")
print("=" * 100)
if pole_raw_candidates_df.empty:
    print("No raw detections found across the two debug images.")
else:
    raw_cols = _existing_cols(
        pole_raw_candidates_df,
        ["image_id", "file_name", "prompt", "det_idx", "score", "x1", "y1", "x2", "y2"]
    )
    _safe_display(
        pole_raw_candidates_df[raw_cols]
        .sort_values(["file_name", "score"], ascending=[True, False])
        .reset_index(drop=True)
    )

print("\n" + "=" * 100)
print("STEP C — FEATURE TABLE")
print("=" * 100)
if pole_feature_debug_df.empty:
    print("No feature rows available.")
else:
    feature_cols = _existing_cols(
        pole_feature_debug_df,
        [
            "image_id", "file_name", "det_idx", "score",
            "box_w", "box_h", "box_area",
            "pole_cx", "pole_cy",
            "x_center_score", "height_score", "area_score",
            "conf_score", "edge_score", "aspect_ratio"
        ]
    )
    _safe_display(
        pole_feature_debug_df[feature_cols]
        .sort_values(["file_name", "score"], ascending=[True, False])
        .reset_index(drop=True)
    )

print("\n" + "=" * 100)
print("STEP D — PREFILTER TABLE")
print("=" * 100)
if pole_candidates_df.empty:
    print("No prefilter rows available.")
else:
    prefilter_cols = _existing_cols(
        pole_candidates_df,
        [
            "image_id", "file_name", "det_idx", "score",
            "area_frac", "height_frac", "aspect_ratio",
            "keep_score", "keep_area", "keep_height", "keep_aspect",
            "is_kept_after_prefilter"
        ]
    )
    _safe_display(
        pole_candidates_df[prefilter_cols]
        .sort_values(["file_name", "score"], ascending=[True, False])
        .reset_index(drop=True)
    )

print("\n" + "=" * 100)
print("STEP E/F — FINAL RANKING + SELECTED POLE TABLE")
print("=" * 100)
if pole_candidates_df.empty:
    print("No scored rows available.")
else:
    final_cols = _existing_cols(
        pole_candidates_df,
        [
            "image_id", "file_name", "det_idx", "score",
            "x_center_score", "height_score", "area_score",
            "conf_score", "edge_score",
            "final_score", "selection_mode", "is_selected_pole"
        ]
    )
    _safe_display(
        pole_candidates_df[final_cols]
        .sort_values(
            ["file_name", "is_selected_pole", "final_score", "score"],
            ascending=[True, False, False, False]
        )
        .reset_index(drop=True)
    )

# -----------------------------------------------------------------------------
# 19. STEP OUTPUTS — 2-IMAGE GALLERIES
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("STEP A/B — RAW DETECTIONS OVERLAY (2 IMAGES)")
print("=" * 100)
show_two_image_step_gallery(
    step_title="STEP A/B — RAW TIMBER POWER POLE DETECTIONS",
    step_results=step_results,
    df_key="raw_df",
    stage="raw",
    selected_only=False,
)

print("\n" + "=" * 100)
print("STEP D — KEPT CANDIDATES AFTER PREFILTER (2 IMAGES)")
print("=" * 100)

for item in step_results:
    # Build the kept-only view for the cyan overlay stage.
    if len(item["prefilter_df"]) > 0 and "is_kept_after_prefilter" in item["prefilter_df"].columns:
        item["kept_df"] = item["prefilter_df"][
            item["prefilter_df"]["is_kept_after_prefilter"] == True
        ].copy()
    else:
        item["kept_df"] = item["prefilter_df"].copy()

show_two_image_step_gallery(
    step_title="STEP D — PREFILTER-KEPT CANDIDATES",
    step_results=step_results,
    df_key="kept_df",
    stage="kept",
    selected_only=False,
)

print("\n" + "=" * 100)
print("STEP F — FINAL SELECTED CENTER POLE (2 IMAGES)")
print("=" * 100)
show_two_image_step_gallery(
    step_title="STEP F — SELECTED CENTER POLE",
    step_results=step_results,
    df_key="scored_df",
    stage="selected",
    selected_only=True,
)

# -----------------------------------------------------------------------------
# 20. SELECTED POLE TABLE
# -----------------------------------------------------------------------------
if not pole_selection_df.empty:
    print("\nSelected poles:")

    selected_cols = _existing_cols(
        pole_selection_df,
        [
            "image_id", "file_name", "det_idx", "score",
            "x1", "y1", "x2", "y2",
            "pole_cx", "pole_cy",
            "final_score", "selection_mode"
        ]
    )
    _safe_display(
        pole_selection_df[selected_cols].reset_index(drop=True)
    )

# -----------------------------------------------------------------------------
# 21. SAVE OUTPUTS INTO NOTEBOOK STATE
# -----------------------------------------------------------------------------
if "save_state" in globals():
    save_state(
        df_names=[
            name for name in [
                "images_df",
                "run_images_df",
                "debug_images_df",
                "pole_raw_candidates_df",
                "pole_feature_debug_df",
                "pole_candidates_df",
                "pole_selection_df",
            ]
            if isinstance(globals().get(name), pd.DataFrame)
        ],
        config_extra={
            "POLE_PROMPT": POLE_PROMPT,
            "POLE_TEXT_THRESHOLD": POLE_TEXT_THRESHOLD,
            "POLE_MASK_THRESHOLD": POLE_MASK_THRESHOLD,
            "POLE_MIN_SCORE": POLE_MIN_SCORE,
            "POLE_MIN_AREA_FRAC": POLE_MIN_AREA_FRAC,
            "POLE_MIN_HEIGHT_FRAC": POLE_MIN_HEIGHT_FRAC,
            "POLE_MIN_ASPECT": POLE_MIN_ASPECT,
            "W_X_CENTER": W_X_CENTER,
            "W_HEIGHT": W_HEIGHT,
            "W_AREA": W_AREA,
            "W_CONF": W_CONF,
            "W_EDGE": W_EDGE,
        },
        nb_globals=globals(),
    )

print("\nCell 7a completed.")
print("You now have:")
print("  - pole_raw_candidates_df")
print("  - pole_feature_debug_df")
print("  - pole_candidates_df")
print("  - pole_selection_df")

In [ ]:
# =============================================================================
# CELL 7a — STEP BY STEP DEBUGGING PROCESS - TIMBER POWER POLE POST-PROCESSING
# =============================================================================
# EXPLANATION:
# This cell runs ONE pole prompt only: "timber power pole".
#
# It performs the following flow for exactly two debug images:
#
#   A) run raw SAM3 pole detection
#   B) collect all returned pole boxes
#   C) compute ranking features
#   D) drop weak / tiny / short / non-vertical / too-wide candidates
#   E) score the remaining candidates
#   F) choose the best remaining candidate as the selected center pole
#
# IMPORTANT:
# - this cell is self-contained
# - it shows example outputs for 2 images at each stage
# - it uses your existing loaded `model` and `processor`
# - it does NOT require your old helper functions
# =============================================================================

# -----------------------------------------------------------------------------
# 0. SAFETY CHECKS
# -----------------------------------------------------------------------------
required_globals = [
    "debug_images_df",
    "model",
    "processor",
    "DEVICE",
]

missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise NameError(
        "Cell 7a cannot run because some required variables are missing.\n"
        "Please make sure the earlier cells have already run.\n"
        f"Missing globals: {missing_globals}"
    )

if not isinstance(debug_images_df, pd.DataFrame):
    raise TypeError("debug_images_df exists but is not a pandas DataFrame.")

if debug_images_df.empty:
    raise ValueError("debug_images_df is empty. Please check Cell 6a.")

# -----------------------------------------------------------------------------
# 1. POLE CONFIG
# -----------------------------------------------------------------------------
POLE_PROMPT = "timber power pole"

POLE_TASK_CFG = {}
if "SAM3_TASK_CONFIG" in globals() and isinstance(SAM3_TASK_CONFIG, dict):
    POLE_TASK_CFG = SAM3_TASK_CONFIG.get("pole_detection", {}) or {}

POLE_TEXT_THRESHOLD = float(
    POLE_TASK_CFG.get("text_score_threshold", globals().get("GLOBAL_TEXT_SCORE_THRESHOLD", 0.45))
)
POLE_MASK_THRESHOLD = float(
    POLE_TASK_CFG.get("mask_threshold", globals().get("GLOBAL_MASK_THRESHOLD", 0.50))
)

print("Pole detection config:")
print(f"  POLE_PROMPT         : {POLE_PROMPT}")
print(f"  POLE_TEXT_THRESHOLD : {POLE_TEXT_THRESHOLD}")
print(f"  POLE_MASK_THRESHOLD : {POLE_MASK_THRESHOLD}")

# -----------------------------------------------------------------------------
# 2. POST-PROCESSING CONSTANTS
# -----------------------------------------------------------------------------
POLE_DEBUG_IMAGE_COUNT = min(2, len(debug_images_df))

POLE_MIN_SCORE = 0.25
POLE_MIN_AREA_FRAC = 0.005      # 0.5% of image area
POLE_MIN_HEIGHT_FRAC = 0.15     # 15% of image height
POLE_MIN_ASPECT = 1.80          # box_h / box_w must be at least this vertical

# NEW: width guards
POLE_MAX_WIDTH_FRAC = 0.08      # max 8% of image width
POLE_MAX_BOX_W_PX = 400         # optional absolute guard

W_X_CENTER = 0.45
W_HEIGHT   = 0.30
W_AREA     = 0.10
W_CONF     = 0.10
W_EDGE     = 0.05

# -----------------------------------------------------------------------------
# 3. HELPER: SAFE DISPLAY
# -----------------------------------------------------------------------------
def _safe_display(obj):
    """
    Display a pandas object in a notebook if possible; otherwise print it.

    Args:
        obj:
            Object to display.

    Returns:
        None
    """
    try:
        display(obj)
    except Exception:
        print(obj)

# -----------------------------------------------------------------------------
# 4. HELPER: SAFE COLUMN SUBSET
# -----------------------------------------------------------------------------
def _existing_cols(df, cols):
    """
    Return only the requested columns that actually exist in the DataFrame.

    Args:
        df:
            Input pandas DataFrame.

        cols:
            Requested column names.

    Returns:
        List[str]:
            Existing column names only.
    """
    if df is None or not isinstance(df, pd.DataFrame):
        return []

    return [c for c in cols if c in df.columns]

# -----------------------------------------------------------------------------
# 5. HELPER: CLIP BOX TO IMAGE
# -----------------------------------------------------------------------------
def clip_box_to_image(x1, y1, x2, y2, image_w, image_h):
    """
    Clip box coordinates safely to image bounds.

    Args:
        x1, y1, x2, y2:
            Raw box coordinates.

        image_w, image_h:
            Image width and height.

    Returns:
        Tuple[float, float, float, float]:
            Clipped and ordered box coordinates.
    """
    x1 = float(np.clip(x1, 0, max(image_w - 1, 0)))
    y1 = float(np.clip(y1, 0, max(image_h - 1, 0)))
    x2 = float(np.clip(x2, 0, max(image_w - 1, 0)))
    y2 = float(np.clip(y2, 0, max(image_h - 1, 0)))

    x1, x2 = sorted([x1, x2])
    y1, y2 = sorted([y1, y2])

    return x1, y1, x2, y2

# -----------------------------------------------------------------------------
# 6. HELPER: NORMALIZE RAW BOXES
# -----------------------------------------------------------------------------
def normalize_raw_boxes(raw_boxes):
    """
    Normalize raw box output into a list of [x1, y1, x2, y2].

    Args:
        raw_boxes:
            Boxes from SAM3 post-processing. Can be tensor, ndarray, list, etc.

    Returns:
        List[List[float]]:
            Standardized list of boxes.
    """
    if raw_boxes is None:
        return []

    if isinstance(raw_boxes, torch.Tensor):
        raw_boxes = raw_boxes.detach().cpu().tolist()
    elif isinstance(raw_boxes, np.ndarray):
        raw_boxes = raw_boxes.tolist()

    if not isinstance(raw_boxes, (list, tuple)):
        return []

    if len(raw_boxes) == 0:
        return []

    if isinstance(raw_boxes[0], (int, float, np.number)):
        if len(raw_boxes) >= 4:
            return [[
                float(raw_boxes[0]),
                float(raw_boxes[1]),
                float(raw_boxes[2]),
                float(raw_boxes[3]),
            ]]
        return []

    boxes = []
    for box in raw_boxes:
        if isinstance(box, torch.Tensor):
            box = box.detach().cpu().tolist()
        elif isinstance(box, np.ndarray):
            box = box.tolist()

        if isinstance(box, (list, tuple)) and len(box) >= 4:
            boxes.append([
                float(box[0]),
                float(box[1]),
                float(box[2]),
                float(box[3]),
            ])

    return boxes

# -----------------------------------------------------------------------------
# 7. HELPER: NORMALIZE RAW SCORES
# -----------------------------------------------------------------------------
def normalize_raw_scores(raw_scores):
    """
    Normalize raw score output into a flat float list.

    Args:
        raw_scores:
            Scores from SAM3 post-processing. Can be tensor, ndarray, list, etc.

    Returns:
        List[float]:
            Standardized list of scores.
    """
    if raw_scores is None:
        return []

    if isinstance(raw_scores, torch.Tensor):
        raw_scores = raw_scores.detach().cpu().flatten().tolist()
    elif isinstance(raw_scores, np.ndarray):
        raw_scores = raw_scores.flatten().tolist()

    if not isinstance(raw_scores, (list, tuple)):
        return []

    scores = []
    for s in raw_scores:
        if isinstance(s, torch.Tensor):
            s = s.detach().cpu().item()
        elif isinstance(s, np.ndarray):
            s = float(np.asarray(s).reshape(-1)[0])

        scores.append(float(s))

    return scores

# -----------------------------------------------------------------------------
# 8. HELPER: BUILD STANDARDIZED RAW DETECTION DATAFRAME
# -----------------------------------------------------------------------------
def build_raw_pole_df(
    raw_boxes,
    raw_scores,
    image_id,
    file_name,
    image_path,
    image_w,
    image_h,
    prompt,
):
    """
    Convert raw SAM3 outputs into one standard DataFrame.

    Args:
        raw_boxes:
            Iterable of raw boxes.

        raw_scores:
            Iterable of raw confidence scores.

        image_id, file_name, image_path:
            Image-level metadata.

        image_w, image_h:
            Image dimensions.

        prompt:
            Prompt used for detection.

    Returns:
        pd.DataFrame:
            Standardized raw candidate table.
    """
    boxes = normalize_raw_boxes(raw_boxes)
    scores = normalize_raw_scores(raw_scores)

    if len(boxes) == 0:
        return pd.DataFrame()

    if len(scores) == 0:
        scores = [0.0] * len(boxes)

    n = min(len(boxes), len(scores))
    rows = []

    for det_idx in range(n):
        x1, y1, x2, y2 = boxes[det_idx]
        x1, y1, x2, y2 = clip_box_to_image(x1, y1, x2, y2, image_w, image_h)

        rows.append({
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "image_w": int(image_w),
            "image_h": int(image_h),
            "prompt": prompt,
            "det_idx": int(det_idx),
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),
            "score": float(scores[det_idx]),
        })

    return pd.DataFrame(rows)

# -----------------------------------------------------------------------------
# 9. HELPER: RUN RAW "TIMBER POWER POLE" SAM3 DETECTION
# -----------------------------------------------------------------------------
def run_power_pole_raw_detections(
    image_pil,
    image_id,
    file_name,
    image_path,
    prompt,
    text_threshold,
    mask_threshold,
):
    """
    Run raw SAM3 timber power pole detection and return ALL detections.

    Args:
        image_pil:
            PIL RGB image.

        image_id, file_name, image_path:
            Image-level metadata.

        prompt:
            The text prompt to run.

        text_threshold:
            Threshold for instance post-processing.

        mask_threshold:
            Mask threshold for instance post-processing.

    Returns:
        pd.DataFrame:
            Raw candidate detections with columns:
            x1, y1, x2, y2, score, plus image metadata.
    """
    image_w, image_h = image_pil.size

    inputs = processor(
        images=image_pil,
        text=prompt,
        return_tensors="pt",
    )
    inputs = inputs.to(DEVICE)

    if "original_sizes" in inputs:
        target_sizes = inputs["original_sizes"].detach().cpu().tolist()
    else:
        target_sizes = [[image_h, image_w]]

    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=text_threshold,
        mask_threshold=mask_threshold,
        target_sizes=target_sizes,
    )

    result = results[0] if isinstance(results, list) and len(results) > 0 else {}
    raw_boxes = result.get("boxes", None)
    raw_scores = result.get("scores", None)

    return build_raw_pole_df(
        raw_boxes=raw_boxes,
        raw_scores=raw_scores,
        image_id=image_id,
        file_name=file_name,
        image_path=image_path,
        image_w=image_w,
        image_h=image_h,
        prompt=prompt,
    )

# -----------------------------------------------------------------------------
# 10. HELPER: ADD POLE FEATURES
# -----------------------------------------------------------------------------
def add_pole_features(raw_df):
    """
    Compute ranking features for pole candidates.

    Args:
        raw_df:
            Raw pole candidate DataFrame with at least:
            x1, y1, x2, y2, score, image_w, image_h

    Returns:
        pd.DataFrame:
            Candidate table with computed features added.
    """
    df = raw_df.copy()

    if df.empty:
        return df

    # -------------------------------------------------------------------------
    # Basic box geometry
    # -------------------------------------------------------------------------
    df["box_w"] = (df["x2"] - df["x1"]).clip(lower=1.0)
    df["box_h"] = (df["y2"] - df["y1"]).clip(lower=1.0)
    df["box_area"] = df["box_w"] * df["box_h"]

    df["pole_cx"] = (df["x1"] + df["x2"]) / 2.0
    df["pole_cy"] = (df["y1"] + df["y2"]) / 2.0

    # -------------------------------------------------------------------------
    # Normalized size / shape features
    # -------------------------------------------------------------------------
    df["image_area"] = df["image_w"] * df["image_h"]
    df["area_frac"] = df["box_area"] / df["image_area"].clip(lower=1.0)
    df["height_frac"] = df["box_h"] / df["image_h"].clip(lower=1.0)
    df["width_frac"] = df["box_w"] / df["image_w"].clip(lower=1.0)
    df["aspect_ratio"] = df["box_h"] / df["box_w"].clip(lower=1.0)

    # -------------------------------------------------------------------------
    # Horizontal center score
    # -------------------------------------------------------------------------
    image_cx = df["image_w"] / 2.0
    center_norm_denom = (df["image_w"] / 2.0).clip(lower=1.0)
    x_center_dist_norm = (df["pole_cx"] - image_cx).abs() / center_norm_denom
    df["x_center_dist_norm"] = x_center_dist_norm
    df["x_center_score"] = 1.0 - np.clip(x_center_dist_norm, 0.0, 1.0)

    # -------------------------------------------------------------------------
    # Height / area / confidence scores
    # -------------------------------------------------------------------------
    max_h = max(float(df["box_h"].max()), 1.0)
    max_a = max(float(df["box_area"].max()), 1.0)

    df["height_score"] = df["box_h"] / max_h
    df["area_score"] = df["box_area"] / max_a
    df["conf_score"] = df["score"]

    # -------------------------------------------------------------------------
    # Edge safety score
    # -------------------------------------------------------------------------
    edge_margin = np.minimum.reduce([
        df["x1"].values,
        df["y1"].values,
        (df["image_w"] - df["x2"]).values,
        (df["image_h"] - df["y2"]).values,
    ])

    edge_norm_denom = 0.05 * np.minimum(df["image_w"], df["image_h"])
    edge_norm_denom = edge_norm_denom.clip(lower=1.0)

    df["edge_margin"] = edge_margin
    df["edge_score"] = np.clip(df["edge_margin"] / edge_norm_denom, 0.0, 1.0)

    return df

# -----------------------------------------------------------------------------
# 11. HELPER: APPLY PREFILTER
# -----------------------------------------------------------------------------
def apply_pole_prefilter(feature_df):
    """
    Drop obviously bad pole candidates before final scoring.

    Args:
        feature_df:
            Feature-enriched candidate table.

    Returns:
        pd.DataFrame:
            Same table with boolean keep flags added.
    """
    df = feature_df.copy()

    if df.empty:
        return df

    df["keep_score"] = df["score"] >= POLE_MIN_SCORE
    df["keep_area"] = df["area_frac"] >= POLE_MIN_AREA_FRAC
    df["keep_height"] = df["height_frac"] >= POLE_MIN_HEIGHT_FRAC
    df["keep_aspect"] = df["aspect_ratio"] >= POLE_MIN_ASPECT
    df["keep_width_frac"] = df["width_frac"] <= POLE_MAX_WIDTH_FRAC
    df["keep_width_px"] = df["box_w"] <= POLE_MAX_BOX_W_PX

    df["is_kept_after_prefilter"] = (
        df["keep_score"] &
        df["keep_area"] &
        df["keep_height"] &
        df["keep_aspect"] &
        df["keep_width_frac"] &
        df["keep_width_px"]
    )

    return df

# -----------------------------------------------------------------------------
# 12. HELPER: SCORE AND SELECT CENTER POLE
# -----------------------------------------------------------------------------
def score_pole_candidates(prefilter_df):
    """
    Score the remaining pole candidates and select the best center pole.

    Args:
        prefilter_df:
            Candidate table with keep flags.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]:
            - scored_df   : all candidates with final scores
            - selected_df : selected single winning row (or empty)
    """
    df = prefilter_df.copy()

    if df.empty:
        return df, df.iloc[0:0].copy()

    df["selection_mode"] = "not_kept"
    df["final_score"] = np.nan
    df["is_selected_pole"] = False

    kept_df = df[df["is_kept_after_prefilter"]].copy()

    # IMPORTANT:
    # Do NOT fall back to all candidates. If nothing survives the prefilter,
    # return "no selection" instead of reviving bad wide boxes.
    if kept_df.empty:
        return df, df.iloc[0:0].copy()

    kept_df["selection_mode"] = "prefilter_kept"

    kept_df["final_score"] = (
        W_X_CENTER * kept_df["x_center_score"] +
        W_HEIGHT   * kept_df["height_score"] +
        W_AREA     * kept_df["area_score"] +
        W_CONF     * kept_df["conf_score"] +
        W_EDGE     * kept_df["edge_score"]
    )

    kept_df = kept_df.sort_values(
        by=["final_score", "score", "box_h", "x_center_score"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    if len(kept_df) > 0:
        kept_df.loc[0, "is_selected_pole"] = True

    score_cols = ["image_id", "det_idx", "selection_mode", "final_score", "is_selected_pole"]
    df = df.drop(columns=["selection_mode", "final_score", "is_selected_pole"], errors="ignore")
    df = df.merge(
        kept_df[score_cols],
        on=["image_id", "det_idx"],
        how="left",
    )

    df["selection_mode"] = df["selection_mode"].fillna("not_kept")
    df["is_selected_pole"] = df["is_selected_pole"].fillna(False)

    selected_df = kept_df[kept_df["is_selected_pole"] == True].copy()

    return df, selected_df

# -----------------------------------------------------------------------------
# 13. HELPER: DRAW BOXES
# -----------------------------------------------------------------------------
def draw_boxes_on_axis(
    ax,
    image_rgb,
    boxes_df,
    title,
    stage="raw",          # "raw" | "kept" | "selected"
    selected_only=False,
):
    """
    Draw pole boxes with fixed stage colors and standard labeling.

    Args:
        ax:
            Matplotlib axis.

        image_rgb:
            RGB numpy image.

        boxes_df:
            DataFrame containing x1, y1, x2, y2 and optional score fields.

        title:
            Plot title.

        stage:
            One of:
                - "raw"      -> yellow
                - "kept"     -> cyan
                - "selected" -> red

        selected_only:
            If True, only selected rows are drawn.

    Returns:
        None
    """
    import matplotlib.patches as patches

    STAGE_COLORS = {
        "raw": "yellow",
        "kept": "cyan",
        "selected": "red",
    }

    STAGE_LINEWIDTHS = {
        "raw": 2.0,
        "kept": 2.5,
        "selected": 3.5,
    }

    color = STAGE_COLORS.get(stage, "yellow")
    lw = STAGE_LINEWIDTHS.get(stage, 2.0)

    ax.imshow(image_rgb)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

    if boxes_df is None or len(boxes_df) == 0:
        return

    plot_df = boxes_df.copy()

    if selected_only:
        if "is_selected_pole" in plot_df.columns:
            plot_df = plot_df[plot_df["is_selected_pole"] == True].copy()
        else:
            plot_df = plot_df.iloc[0:0].copy()

    if plot_df.empty:
        return

    for _, r in plot_df.iterrows():
        x1, y1, x2, y2 = r["x1"], r["y1"], r["x2"], r["y2"]
        w = x2 - x1
        h = y2 - y1

        rect = patches.Rectangle(
            (x1, y1),
            w,
            h,
            linewidth=lw,
            edgecolor=color,
            facecolor="none",
        )
        ax.add_patch(rect)

        label_bits = ["POLE"]

        prompt_text = None
        if "prompt" in r and pd.notna(r["prompt"]) and str(r["prompt"]).strip():
            prompt_text = str(r["prompt"]).strip()
        elif "POLE_PROMPT" in globals():
            prompt_text = str(POLE_PROMPT).strip()

        if prompt_text:
            label_bits.append(prompt_text)

        if "score" in r and pd.notna(r["score"]):
            label_bits.append(f"score={float(r['score']):.3f}")

        if stage == "selected" and "final_score" in r and pd.notna(r["final_score"]):
            label_bits.append(f"final={float(r['final_score']):.3f}")

        label = " | ".join(label_bits)

        if label:
            ax.text(
                x1,
                max(y1 - 6, 8),
                label,
                fontsize=8.5,
                color="white",
                bbox=dict(
                    facecolor=color,
                    alpha=0.85,
                    edgecolor="none",
                    pad=2.0,
                ),
            )

# -----------------------------------------------------------------------------
# 14. HELPER: TWO-IMAGE STEP GALLERY
# -----------------------------------------------------------------------------
def show_two_image_step_gallery(
    step_title,
    step_results,
    df_key,
    stage="raw",          # "raw" | "kept" | "selected"
    selected_only=False,
):
    """
    Show a side-by-side gallery for two images for one processing step.

    Args:
        step_title:
            Title for this processing step.

        step_results:
            List of per-image result dictionaries.

        df_key:
            Key inside each step_results item to use as the box DataFrame.

        stage:
            Controls fixed overlay color.

        selected_only:
            If True, only the selected pole is drawn.

    Returns:
        None
    """
    n = len(step_results)
    fig, axes = plt.subplots(1, n, figsize=(11 * n, 8))

    if n == 1:
        axes = [axes]

    for ax, item in zip(axes, step_results):
        draw_boxes_on_axis(
            ax=ax,
            image_rgb=item["image_rgb"],
            boxes_df=item[df_key],
            title=f"{item['file_name']}\n{step_title}",
            stage=stage,
            selected_only=selected_only,
        )

    plt.tight_layout()
    plt.show()
    plt.close()

# -----------------------------------------------------------------------------
# 15. CHOOSE EXACTLY TWO DEBUG IMAGES
# -----------------------------------------------------------------------------
pole_debug_input_df = debug_images_df.head(POLE_DEBUG_IMAGE_COUNT).copy().reset_index(drop=True)

if pole_debug_input_df.empty:
    raise ValueError("No debug images available for pole step debugging.")

print("\nUsing these debug images for pole post-processing:")

debug_cols = _existing_cols(
    pole_debug_input_df,
    ["image_id", "file_name", "image_path"]
)
_safe_display(pole_debug_input_df[debug_cols])

# -----------------------------------------------------------------------------
# 16. RUN STEP-BY-STEP POLE PIPELINE ON TWO DEBUG IMAGES
# -----------------------------------------------------------------------------
step_results = []

all_raw_rows = []
all_feature_rows = []
all_scored_rows = []
all_selected_rows = []

for _, row in pole_debug_input_df.iterrows():
    image_id = row["image_id"] if "image_id" in row.index and pd.notna(row["image_id"]) else None
    file_name = row["file_name"] if "file_name" in row.index and pd.notna(row["file_name"]) else None
    image_path = row["image_path"] if "image_path" in row.index and pd.notna(row["image_path"]) else None

    if pd.isna(image_path) or not isinstance(image_path, str) or len(image_path.strip()) == 0:
        raise ValueError("A debug row is missing a valid image_path.")

    if (
        pd.isna(file_name)
        or not isinstance(file_name, str)
        or len(file_name.strip()) == 0
    ):
        file_name = os.path.basename(image_path) if isinstance(image_path, str) else "unknown_file"

    print("\n" + "=" * 100)
    print(f"PROCESSING IMAGE: {file_name}")
    print("=" * 100)

    with Image.open(image_path) as img:
        image_pil = img.convert("RGB")
        image_pil.load()

    image_rgb = np.array(image_pil)

    # STEP A/B — RAW DETECTIONS
    raw_df = run_power_pole_raw_detections(
        image_pil=image_pil,
        image_id=image_id,
        file_name=file_name,
        image_path=image_path,
        prompt=POLE_PROMPT,
        text_threshold=POLE_TEXT_THRESHOLD,
        mask_threshold=POLE_MASK_THRESHOLD,
    )

    if raw_df is None:
        raw_df = pd.DataFrame()

    if not isinstance(raw_df, pd.DataFrame):
        raise TypeError(
            f"run_power_pole_raw_detections() must return a DataFrame, got: {type(raw_df)}"
        )

    if len(raw_df) == 0:
        print("No raw pole detections returned for this image.")
        feature_df = raw_df.copy()
        prefilter_df = raw_df.copy()
        scored_df = raw_df.copy()
        selected_df = raw_df.copy()
    else:
        # STEP C — FEATURE COMPUTATION
        feature_df = add_pole_features(raw_df)

        # STEP D — PREFILTER
        prefilter_df = apply_pole_prefilter(feature_df)

        # STEP E/F — SCORE + SELECT
        scored_df, selected_df = score_pole_candidates(prefilter_df)

    step_results.append({
        "image_id": image_id,
        "file_name": file_name,
        "image_path": image_path,
        "image_rgb": image_rgb,
        "raw_df": raw_df,
        "feature_df": feature_df,
        "prefilter_df": prefilter_df,
        "scored_df": scored_df,
        "selected_df": selected_df,
    })

    if len(raw_df) > 0:
        all_raw_rows.append(raw_df)

    if len(feature_df) > 0:
        all_feature_rows.append(feature_df)

    if len(scored_df) > 0:
        all_scored_rows.append(scored_df)

    if len(selected_df) > 0:
        all_selected_rows.append(selected_df)

# -----------------------------------------------------------------------------
# 17. COMBINE OUTPUT TABLES
# -----------------------------------------------------------------------------
pole_raw_candidates_df = (
    pd.concat(all_raw_rows, ignore_index=True)
    if len(all_raw_rows) > 0 else
    pd.DataFrame()
)

pole_feature_debug_df = (
    pd.concat(all_feature_rows, ignore_index=True)
    if len(all_feature_rows) > 0 else
    pd.DataFrame()
)

pole_candidates_df = (
    pd.concat(all_scored_rows, ignore_index=True)
    if len(all_scored_rows) > 0 else
    pd.DataFrame()
)

pole_selection_df = (
    pd.concat(all_selected_rows, ignore_index=True)
    if len(all_selected_rows) > 0 else
    pd.DataFrame()
)

# -----------------------------------------------------------------------------
# 18. STEP OUTPUTS — TABLES
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("STEP A/B — RAW DETECTIONS TABLE")
print("=" * 100)
if pole_raw_candidates_df.empty:
    print("No raw detections found across the two debug images.")
else:
    raw_cols = _existing_cols(
        pole_raw_candidates_df,
        ["image_id", "file_name", "prompt", "det_idx", "score", "x1", "y1", "x2", "y2"]
    )
    _safe_display(
        pole_raw_candidates_df[raw_cols]
        .sort_values(["file_name", "score"], ascending=[True, False])
        .reset_index(drop=True)
    )

print("\n" + "=" * 100)
print("STEP C — FEATURE TABLE")
print("=" * 100)
if pole_feature_debug_df.empty:
    print("No feature rows available.")
else:
    feature_cols = _existing_cols(
        pole_feature_debug_df,
        [
            "image_id", "file_name", "det_idx", "score",
            "box_w", "box_h", "box_area",
            "width_frac",
            "pole_cx", "pole_cy",
            "x_center_score", "height_score", "area_score",
            "conf_score", "edge_score", "aspect_ratio"
        ]
    )
    _safe_display(
        pole_feature_debug_df[feature_cols]
        .sort_values(["file_name", "score"], ascending=[True, False])
        .reset_index(drop=True)
    )

print("\n" + "=" * 100)
print("STEP D — PREFILTER TABLE")
print("=" * 100)
if pole_candidates_df.empty:
    print("No prefilter rows available.")
else:
    prefilter_cols = _existing_cols(
        pole_candidates_df,
        [
            "image_id", "file_name", "det_idx", "score",
            "area_frac", "height_frac", "width_frac", "aspect_ratio",
            "keep_score", "keep_area", "keep_height", "keep_aspect",
            "keep_width_frac", "keep_width_px",
            "is_kept_after_prefilter"
        ]
    )
    _safe_display(
        pole_candidates_df[prefilter_cols]
        .sort_values(["file_name", "score"], ascending=[True, False])
        .reset_index(drop=True)
    )

print("\n" + "=" * 100)
print("STEP E/F — FINAL RANKING + SELECTED POLE TABLE")
print("=" * 100)
if pole_candidates_df.empty:
    print("No scored rows available.")
else:
    final_cols = _existing_cols(
        pole_candidates_df,
        [
            "image_id", "file_name", "det_idx", "score",
            "x_center_score", "height_score", "area_score",
            "conf_score", "edge_score",
            "final_score", "selection_mode", "is_selected_pole"
        ]
    )
    _safe_display(
        pole_candidates_df[final_cols]
        .sort_values(
            ["file_name", "is_selected_pole", "final_score", "score"],
            ascending=[True, False, False, False]
        )
        .reset_index(drop=True)
    )

# -----------------------------------------------------------------------------
# 19. STEP OUTPUTS — 2-IMAGE GALLERIES
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("STEP A/B — RAW DETECTIONS OVERLAY (2 IMAGES)")
print("=" * 100)
show_two_image_step_gallery(
    step_title="STEP A/B — RAW TIMBER POWER POLE DETECTIONS",
    step_results=step_results,
    df_key="raw_df",
    stage="raw",
    selected_only=False,
)

print("\n" + "=" * 100)
print("STEP D — KEPT CANDIDATES AFTER PREFILTER (2 IMAGES)")
print("=" * 100)

for item in step_results:
    if len(item["prefilter_df"]) > 0 and "is_kept_after_prefilter" in item["prefilter_df"].columns:
        item["kept_df"] = item["prefilter_df"][
            item["prefilter_df"]["is_kept_after_prefilter"] == True
        ].copy()
    else:
        item["kept_df"] = item["prefilter_df"].copy()

show_two_image_step_gallery(
    step_title="STEP D — PREFILTER-KEPT CANDIDATES",
    step_results=step_results,
    df_key="kept_df",
    stage="kept",
    selected_only=False,
)

print("\n" + "=" * 100)
print("STEP F — FINAL SELECTED CENTER POLE (2 IMAGES)")
print("=" * 100)
show_two_image_step_gallery(
    step_title="STEP F — SELECTED CENTER POLE",
    step_results=step_results,
    df_key="scored_df",
    stage="selected",
    selected_only=True,
)

# -----------------------------------------------------------------------------
# 20. SELECTED POLE TABLE
# -----------------------------------------------------------------------------
if not pole_selection_df.empty:
    print("\nSelected poles:")

    selected_cols = _existing_cols(
        pole_selection_df,
        [
            "image_id", "file_name", "det_idx", "score",
            "x1", "y1", "x2", "y2",
            "pole_cx", "pole_cy",
            "box_w", "box_h", "width_frac",
            "final_score", "selection_mode"
        ]
    )
    _safe_display(
        pole_selection_df[selected_cols].reset_index(drop=True)
    )
else:
    print("\nNo pole survived the prefilter, so no selected poles were produced.")

# -----------------------------------------------------------------------------
# 21. SAVE OUTPUTS INTO NOTEBOOK STATE
# -----------------------------------------------------------------------------
if "save_state" in globals():
    save_state(
        df_names=[
            name for name in [
                "images_df",
                "run_images_df",
                "debug_images_df",
                "pole_raw_candidates_df",
                "pole_feature_debug_df",
                "pole_candidates_df",
                "pole_selection_df",
            ]
            if isinstance(globals().get(name), pd.DataFrame)
        ],
        config_extra={
            "POLE_PROMPT": POLE_PROMPT,
            "POLE_TEXT_THRESHOLD": POLE_TEXT_THRESHOLD,
            "POLE_MASK_THRESHOLD": POLE_MASK_THRESHOLD,
            "POLE_MIN_SCORE": POLE_MIN_SCORE,
            "POLE_MIN_AREA_FRAC": POLE_MIN_AREA_FRAC,
            "POLE_MIN_HEIGHT_FRAC": POLE_MIN_HEIGHT_FRAC,
            "POLE_MIN_ASPECT": POLE_MIN_ASPECT,
            "POLE_MAX_WIDTH_FRAC": POLE_MAX_WIDTH_FRAC,
            "POLE_MAX_BOX_W_PX": POLE_MAX_BOX_W_PX,
            "W_X_CENTER": W_X_CENTER,
            "W_HEIGHT": W_HEIGHT,
            "W_AREA": W_AREA,
            "W_CONF": W_CONF,
            "W_EDGE": W_EDGE,
        },
        nb_globals=globals(),
    )

print("\nCell 7a completed.")
print("You now have:")
print("  - pole_raw_candidates_df")
print("  - pole_feature_debug_df")
print("  - pole_candidates_df")
print("  - pole_selection_df")

# 7a) PRODUCTION - TIMBER POWER POLE DETECTION

In [ ]:
# =============================================================================
# CELL 7a — PRODUCTION POWER-POLE SELECTION
# =============================================================================
# EXPLANATION:
# Run production pole selection on all run images using one prompt only:
# "timber power pole".
#
# OUTPUTS:
# - pole_candidates_df : all scored candidates
# - pole_selection_df  : one row per image, with either a selected pole
#                        or "no_reliable_pole_found"
# =============================================================================

# -----------------------------------------------------------------------------
# 0. SAFETY CHECKS
# -----------------------------------------------------------------------------
required_globals = [
    "run_images_df",
    "model",
    "processor",
    "DEVICE",
]

# Check that all required globals exist before continuing.
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise NameError(
        "Cell 7a cannot run because some required variables are missing.\n"
        "Please make sure earlier cells have already run.\n"
        f"Missing globals: {missing_globals}"
    )

# Confirm the run image table is a pandas DataFrame.
if not isinstance(run_images_df, pd.DataFrame):
    raise TypeError("run_images_df exists but is not a pandas DataFrame.")

# Stop early if there are no rows to process.
if run_images_df.empty:
    raise ValueError("run_images_df is empty. Please check Cell 6a.")

# -----------------------------------------------------------------------------
# 1. POLE CONFIG
# -----------------------------------------------------------------------------
POLE_PROMPT = "timber power pole"

# Pull pole-specific config if it exists.
POLE_TASK_CFG = {}
if "SAM3_TASK_CONFIG" in globals() and isinstance(SAM3_TASK_CONFIG, dict):
    POLE_TASK_CFG = SAM3_TASK_CONFIG.get("pole_detection", {}) or {}

# Use task-config thresholds if present, otherwise fall back to global defaults.
POLE_TEXT_THRESHOLD = float(
    POLE_TASK_CFG.get("text_score_threshold", globals().get("GLOBAL_TEXT_SCORE_THRESHOLD", 0.45))
)
POLE_MASK_THRESHOLD = float(
    POLE_TASK_CFG.get("mask_threshold", globals().get("GLOBAL_MASK_THRESHOLD", 0.50))
)

print("Pole production config:")
print(f"  POLE_PROMPT         : {POLE_PROMPT}")
print(f"  POLE_TEXT_THRESHOLD : {POLE_TEXT_THRESHOLD}")
print(f"  POLE_MASK_THRESHOLD : {POLE_MASK_THRESHOLD}")

# -----------------------------------------------------------------------------
# 2. PREFILTER + SCORING CONSTANTS
# -----------------------------------------------------------------------------
POLE_MIN_SCORE = 0.25
POLE_MIN_AREA_FRAC = 0.005      # Minimum area fraction of full image.
POLE_MIN_HEIGHT_FRAC = 0.15     # Minimum height fraction of full image.
POLE_MIN_ASPECT = 1.80          # Minimum vertical aspect ratio.

W_X_CENTER = 0.45
W_HEIGHT   = 0.30
W_AREA     = 0.10
W_CONF     = 0.10
W_EDGE     = 0.05

# Limit the production gallery to a small number of images.
POLE_GALLERY_COUNT = min(2, len(run_images_df))

# -----------------------------------------------------------------------------
# 3. HELPER: SAFE DISPLAY
# -----------------------------------------------------------------------------
def _safe_display(obj):
    """
    Display a pandas object in a notebook if possible; otherwise print it.

    Args:
        obj:
            Object to display.

    Returns:
        None
    """
    try:
        display(obj)
    except Exception:
        print(obj)

# -----------------------------------------------------------------------------
# 4. HELPER: SAFE COLUMN SUBSET
# -----------------------------------------------------------------------------
def _existing_cols(df, cols):
    """
    Return only the requested columns that actually exist in the DataFrame.

    Args:
        df:
            Input pandas DataFrame.

        cols:
            Requested column names.

    Returns:
        List[str]:
            Existing column names only.
    """
    if df is None or not isinstance(df, pd.DataFrame):
        return []

    return [c for c in cols if c in df.columns]

# -----------------------------------------------------------------------------
# 5. HELPER: CLIP BOX TO IMAGE
# -----------------------------------------------------------------------------
def clip_box_to_image(x1, y1, x2, y2, image_w, image_h):
    """
    Clip box coordinates safely to image bounds.

    Args:
        x1, y1, x2, y2:
            Raw box coordinates.

        image_w, image_h:
            Image width and height.

    Returns:
        Tuple[float, float, float, float]:
            Clipped and ordered box coordinates.
    """
    # Clamp all coordinates to image bounds.
    x1 = float(np.clip(x1, 0, max(image_w - 1, 0)))
    y1 = float(np.clip(y1, 0, max(image_h - 1, 0)))
    x2 = float(np.clip(x2, 0, max(image_w - 1, 0)))
    y2 = float(np.clip(y2, 0, max(image_h - 1, 0)))

    # Reorder endpoints in case they came back inverted.
    x1, x2 = sorted([x1, x2])
    y1, y2 = sorted([y1, y2])

    return x1, y1, x2, y2

# -----------------------------------------------------------------------------
# 6. HELPER: NORMALIZE RAW BOXES
# -----------------------------------------------------------------------------
def normalize_raw_boxes(raw_boxes):
    """
    Normalize raw box output into a list of [x1, y1, x2, y2].

    Args:
        raw_boxes:
            Boxes from SAM3 post-processing. Can be tensor, ndarray, list, etc.

    Returns:
        List[List[float]]:
            Standardized list of boxes.
    """
    # Return an empty list if no boxes were produced.
    if raw_boxes is None:
        return []

    # Convert tensors / ndarrays to standard Python lists first.
    if isinstance(raw_boxes, torch.Tensor):
        raw_boxes = raw_boxes.detach().cpu().tolist()
    elif isinstance(raw_boxes, np.ndarray):
        raw_boxes = raw_boxes.tolist()

    # Return an empty list if the result is not list-like.
    if not isinstance(raw_boxes, (list, tuple)):
        return []

    # Handle the no-box case.
    if len(raw_boxes) == 0:
        return []

    # Handle the case where a single box comes back as a flat list.
    if isinstance(raw_boxes[0], (int, float, np.number)):
        if len(raw_boxes) >= 4:
            return [[
                float(raw_boxes[0]),
                float(raw_boxes[1]),
                float(raw_boxes[2]),
                float(raw_boxes[3]),
            ]]
        return []

    boxes = []

    # Standardize each candidate box into [x1, y1, x2, y2].
    for box in raw_boxes:
        if isinstance(box, torch.Tensor):
            box = box.detach().cpu().tolist()
        elif isinstance(box, np.ndarray):
            box = box.tolist()

        if isinstance(box, (list, tuple)) and len(box) >= 4:
            boxes.append([
                float(box[0]),
                float(box[1]),
                float(box[2]),
                float(box[3]),
            ])

    return boxes

# -----------------------------------------------------------------------------
# 7. HELPER: NORMALIZE RAW SCORES
# -----------------------------------------------------------------------------
def normalize_raw_scores(raw_scores):
    """
    Normalize raw score output into a flat float list.

    Args:
        raw_scores:
            Scores from SAM3 post-processing. Can be tensor, ndarray, list, etc.

    Returns:
        List[float]:
            Standardized list of scores.
    """
    # Return an empty list if no scores were produced.
    if raw_scores is None:
        return []

    # Convert tensors / ndarrays to flat Python lists.
    if isinstance(raw_scores, torch.Tensor):
        raw_scores = raw_scores.detach().cpu().flatten().tolist()
    elif isinstance(raw_scores, np.ndarray):
        raw_scores = raw_scores.flatten().tolist()

    # Return an empty list if the result is not list-like.
    if not isinstance(raw_scores, (list, tuple)):
        return []

    scores = []

    # Convert every score to a Python float.
    for s in raw_scores:
        if isinstance(s, torch.Tensor):
            s = s.detach().cpu().item()
        elif isinstance(s, np.ndarray):
            s = float(np.asarray(s).reshape(-1)[0])

        scores.append(float(s))

    return scores

# -----------------------------------------------------------------------------
# 8. HELPER: BUILD STANDARDIZED RAW DETECTION DATAFRAME
# -----------------------------------------------------------------------------
def build_raw_pole_df(
    raw_boxes,
    raw_scores,
    image_id,
    file_name,
    image_path,
    image_w,
    image_h,
    prompt,
):
    """
    Convert raw SAM3 outputs into one standard DataFrame.

    Args:
        raw_boxes:
            Iterable of raw boxes.

        raw_scores:
            Iterable of raw confidence scores.

        image_id, file_name, image_path:
            Image-level metadata.

        image_w, image_h:
            Image dimensions.

        prompt:
            Prompt used for detection.

    Returns:
        pd.DataFrame:
            Standardized raw candidate table.
    """
    # Standardize the box and score outputs first.
    boxes = normalize_raw_boxes(raw_boxes)
    scores = normalize_raw_scores(raw_scores)

    # Return an empty DataFrame if there are no boxes.
    if len(boxes) == 0:
        return pd.DataFrame()

    # Fill with zeros if scores are missing.
    if len(scores) == 0:
        scores = [0.0] * len(boxes)

    # Keep only rows supported by both boxes and scores.
    n = min(len(boxes), len(scores))
    rows = []

    for det_idx in range(n):
        # Read and clip the raw box to image bounds.
        x1, y1, x2, y2 = boxes[det_idx]
        x1, y1, x2, y2 = clip_box_to_image(x1, y1, x2, y2, image_w, image_h)

        # Build one standardized candidate row.
        rows.append({
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "image_w": int(image_w),
            "image_h": int(image_h),
            "prompt": prompt,
            "det_idx": int(det_idx),
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),
            "score": float(scores[det_idx]),
        })

    return pd.DataFrame(rows)

# -----------------------------------------------------------------------------
# 9. HELPER: RUN RAW "TIMBER POWER POLE" SAM3 DETECTION
# -----------------------------------------------------------------------------
def run_power_pole_raw_detections(
    image_pil,
    image_id,
    file_name,
    image_path,
    prompt,
    text_threshold,
    mask_threshold,
):
    """
    Run raw SAM3 timber power pole detection and return ALL detections.

    Args:
        image_pil:
            PIL RGB image.

        image_id, file_name, image_path:
            Image-level metadata.

        prompt:
            The text prompt to run.

        text_threshold:
            Threshold for instance post-processing.

        mask_threshold:
            Mask threshold for instance post-processing.

    Returns:
        pd.DataFrame:
            Raw candidate detections with columns:
            x1, y1, x2, y2, score, plus image metadata.
    """
    # Read the original image size for downstream box clipping and scaling.
    image_w, image_h = image_pil.size

    # Tokenize the image + text prompt for SAM3.
    inputs = processor(
        images=image_pil,
        text=prompt,
        return_tensors="pt",
    )

    # Move the full batch to the active device.
    inputs = inputs.to(DEVICE)

    # Prefer processor-provided original sizes if available.
    if "original_sizes" in inputs:
        target_sizes = inputs["original_sizes"].detach().cpu().tolist()
    else:
        target_sizes = [[image_h, image_w]]

    # Run inference without gradients.
    with torch.no_grad():
        outputs = model(**inputs)

    # Convert model outputs into instance segmentation predictions.
    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=text_threshold,
        mask_threshold=mask_threshold,
        target_sizes=target_sizes,
    )

    # Pull out the first image result safely.
    result = results[0] if isinstance(results, list) and len(results) > 0 else {}

    # Get boxes and scores from the post-processed result.
    raw_boxes = result.get("boxes", None)
    raw_scores = result.get("scores", None)

    # Convert everything into the standard raw candidate DataFrame.
    return build_raw_pole_df(
        raw_boxes=raw_boxes,
        raw_scores=raw_scores,
        image_id=image_id,
        file_name=file_name,
        image_path=image_path,
        image_w=image_w,
        image_h=image_h,
        prompt=prompt,
    )

# -----------------------------------------------------------------------------
# 10. HELPER: ADD POLE FEATURES
# -----------------------------------------------------------------------------
def add_pole_features(raw_df):
    """
    Compute ranking features for pole candidates.

    Args:
        raw_df:
            Raw pole candidate DataFrame with at least:
            x1, y1, x2, y2, score, image_w, image_h

    Returns:
        pd.DataFrame:
            Candidate table with computed features added.
    """
    df = raw_df.copy()

    # Return immediately if there is nothing to score.
    if df.empty:
        return df

    # Compute basic box geometry.
    df["box_w"] = (df["x2"] - df["x1"]).clip(lower=1.0)
    df["box_h"] = (df["y2"] - df["y1"]).clip(lower=1.0)
    df["box_area"] = df["box_w"] * df["box_h"]

    # Compute the pole box center coordinates.
    df["pole_cx"] = (df["x1"] + df["x2"]) / 2.0
    df["pole_cy"] = (df["y1"] + df["y2"]) / 2.0

    # Normalize size-based measurements against the image dimensions.
    df["image_area"] = df["image_w"] * df["image_h"]
    df["area_frac"] = df["box_area"] / df["image_area"].clip(lower=1.0)
    df["height_frac"] = df["box_h"] / df["image_h"].clip(lower=1.0)
    df["aspect_ratio"] = df["box_h"] / df["box_w"].clip(lower=1.0)

    # Measure horizontal closeness to the image center.
    image_cx = df["image_w"] / 2.0
    x_center_dist_norm = (
        (df["pole_cx"] - image_cx).abs() /
        (df["image_w"] / 2.0).clip(lower=1.0)
    )
    df["x_center_dist_norm"] = x_center_dist_norm
    df["x_center_score"] = 1.0 - np.clip(x_center_dist_norm, 0.0, 1.0)

    # Scale height and area relative to the largest candidate in this image.
    max_h = max(float(df["box_h"].max()), 1.0)
    max_a = max(float(df["box_area"].max()), 1.0)

    df["height_score"] = df["box_h"] / max_h
    df["area_score"] = df["box_area"] / max_a
    df["conf_score"] = df["score"]

    # Compute how safely away from the image border the box is.
    edge_margin = np.minimum.reduce([
        df["x1"].values,
        df["y1"].values,
        (df["image_w"] - df["x2"]).values,
        (df["image_h"] - df["y2"]).values,
    ])

    # Normalize the edge margin by 5% of the smaller image dimension.
    edge_norm_denom = 0.05 * np.minimum(df["image_w"], df["image_h"])
    edge_norm_denom = edge_norm_denom.clip(lower=1.0)

    df["edge_margin"] = edge_margin
    df["edge_score"] = np.clip(df["edge_margin"] / edge_norm_denom, 0.0, 1.0)

    return df

# -----------------------------------------------------------------------------
# 11. HELPER: APPLY PREFILTER
# -----------------------------------------------------------------------------
def apply_pole_prefilter(feature_df):
    """
    Drop obviously bad pole candidates before final scoring.

    Args:
        feature_df:
            Feature-enriched candidate table.

    Returns:
        pd.DataFrame:
            Same table with boolean keep flags added.
    """
    df = feature_df.copy()

    # Return immediately if there is nothing to filter.
    if df.empty:
        return df

    # Evaluate each prefilter rule independently.
    df["keep_score"] = df["score"] >= POLE_MIN_SCORE
    df["keep_area"] = df["area_frac"] >= POLE_MIN_AREA_FRAC
    df["keep_height"] = df["height_frac"] >= POLE_MIN_HEIGHT_FRAC
    df["keep_aspect"] = df["aspect_ratio"] >= POLE_MIN_ASPECT

    # Keep only candidates that satisfy all rules.
    df["is_kept_after_prefilter"] = (
        df["keep_score"] &
        df["keep_area"] &
        df["keep_height"] &
        df["keep_aspect"]
    )

    return df

# -----------------------------------------------------------------------------
# 12. HELPER: SCORE CANDIDATES WITHOUT FORCING A WINNER
# -----------------------------------------------------------------------------
def score_pole_candidates(prefilter_df):
    """
    Score only the kept candidates and select a winner only if a reliable
    candidate exists.

    IMPORTANT:
    - Unlike the debug cell, this version does NOT fall back to all candidates.
    - If no candidate survives prefilter, it returns an empty selected_df.

    Args:
        prefilter_df:
            Candidate table with keep flags.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]:
            - scored_df   : all candidates with final scores where applicable
            - selected_df : selected single winning row, or empty if no reliable
                            pole was found
    """
    df = prefilter_df.copy()

    # Return empty outputs immediately if there are no rows.
    if df.empty:
        return df, df.iloc[0:0].copy()

    # Initialize default output columns for the full candidate table.
    df["selection_mode"] = "filtered_out"
    df["selection_status"] = "not_selected"
    df["final_score"] = np.nan
    df["is_selected_pole"] = False

    # Keep only rows that survived prefiltering.
    kept_df = df[df["is_kept_after_prefilter"] == True].copy()

    # In production we do not force a winner if every candidate failed.
    if kept_df.empty:
        return df, df.iloc[0:0].copy()

    kept_df["selection_mode"] = "prefilter_kept"
    kept_df["selection_status"] = "candidate_not_selected"

    # Compute the final weighted score used for ranking.
    kept_df["final_score"] = (
        W_X_CENTER * kept_df["x_center_score"] +
        W_HEIGHT   * kept_df["height_score"] +
        W_AREA     * kept_df["area_score"] +
        W_CONF     * kept_df["conf_score"] +
        W_EDGE     * kept_df["edge_score"]
    )

    # Sort best-to-worst using final score first, then tie-breakers.
    kept_df = kept_df.sort_values(
        by=["final_score", "score", "box_h", "x_center_score"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    # Mark the top-ranked candidate as the selected pole.
    kept_df.loc[0, "is_selected_pole"] = True
    kept_df.loc[0, "selection_status"] = "selected"

    # Merge the ranking outputs back into the full candidate table.
    # Use image_path + det_idx so missing image_id values cannot collide.
    merge_keys = ["image_path", "det_idx"]
    score_cols = merge_keys + [
        "selection_mode",
        "selection_status",
        "final_score",
        "is_selected_pole",
    ]

    df = df.drop(
        columns=["selection_mode", "selection_status", "final_score", "is_selected_pole"],
        errors="ignore",
    )
    df = df.merge(
        kept_df[score_cols],
        on=merge_keys,
        how="left",
    )

    # Fill rows that were never ranked because they were not kept.
    df["selection_mode"] = df["selection_mode"].fillna("filtered_out")
    df["selection_status"] = df["selection_status"].fillna("not_selected")
    df["is_selected_pole"] = df["is_selected_pole"].fillna(False)

    # Build a one-row selected table for convenience.
    selected_df = kept_df[kept_df["is_selected_pole"] == True].copy()

    return df, selected_df

# -----------------------------------------------------------------------------
# 13. HELPER: DRAW BOXES
# -----------------------------------------------------------------------------
def draw_boxes_on_axis(
    ax,
    image_rgb,
    boxes_df,
    title,
    stage="raw",          # "raw" | "kept" | "selected"
    selected_only=False,
    scale_x=1.0,
    scale_y=1.0,
):
    """
    Draw pole boxes with fixed stage colors and standard labeling.

    Args:
        ax:
            Matplotlib axis.

        image_rgb:
            RGB numpy image.

        boxes_df:
            DataFrame containing x1, y1, x2, y2 and optional score fields.

        title:
            Plot title.

        stage:
            One of:
                - "raw"      -> yellow
                - "kept"     -> cyan
                - "selected" -> red

        selected_only:
            If True, only selected rows are drawn.

        scale_x, scale_y:
            Multipliers used to map original-image box coordinates onto the
            resized gallery image.

    Returns:
        None
    """
    import matplotlib.patches as patches

    STAGE_COLORS = {
        "raw": "yellow",
        "kept": "cyan",
        "selected": "red",
    }

    STAGE_LINEWIDTHS = {
        "raw": 2.0,
        "kept": 2.5,
        "selected": 3.5,
    }

    # Resolve the display style for this stage.
    color = STAGE_COLORS.get(stage, "yellow")
    lw = STAGE_LINEWIDTHS.get(stage, 2.0)

    # Show the underlying image.
    ax.imshow(image_rgb)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

    # Nothing to draw if the candidate table is missing or empty.
    if boxes_df is None or len(boxes_df) == 0:
        return

    plot_df = boxes_df.copy()

    # Keep only the selected row if requested.
    if selected_only:
        if "is_selected_pole" in plot_df.columns:
            plot_df = plot_df[plot_df["is_selected_pole"] == True].copy()
        else:
            plot_df = plot_df.iloc[0:0].copy()

    # Exit if no rows remain after filtering.
    if plot_df.empty:
        return

    for _, r in plot_df.iterrows():
        # Read the original box coordinates.
        x1, y1, x2, y2 = r["x1"], r["y1"], r["x2"], r["y2"]

        # Scale the box coordinates to match the resized gallery image.
        x1 = x1 * scale_x
        y1 = y1 * scale_y
        x2 = x2 * scale_x
        y2 = y2 * scale_y

        w = x2 - x1
        h = y2 - y1

        # Draw the rectangle overlay.
        rect = patches.Rectangle(
            (x1, y1),
            w,
            h,
            linewidth=lw,
            edgecolor=color,
            facecolor="none",
        )
        ax.add_patch(rect)

        label_bits = []

        # Start the label with a fixed object type prefix.
        label_bits.append("POLE")

        # Prefer the row-level prompt if available.
        prompt_text = None
        if "prompt" in r and pd.notna(r["prompt"]) and str(r["prompt"]).strip():
            prompt_text = str(r["prompt"]).strip()
        elif "POLE_PROMPT" in globals():
            prompt_text = str(POLE_PROMPT).strip()

        if prompt_text:
            label_bits.append(prompt_text)

        # Add the raw confidence score when present.
        if "score" in r and pd.notna(r["score"]):
            label_bits.append(f"score={float(r['score']):.3f}")

        # Add the final score for selected/final overlays when present.
        if stage == "selected" and "final_score" in r and pd.notna(r["final_score"]):
            label_bits.append(f"final={float(r['final_score']):.3f}")

        label = " | ".join(label_bits)

        # Draw the text label above the box.
        if label:
            ax.text(
                x1,
                max(y1 - 6, 8),
                label,
                fontsize=8.5,
                color="white",
                bbox=dict(
                    facecolor=color,
                    alpha=0.85,
                    edgecolor="none",
                    pad=2.0,
                ),
            )

# -----------------------------------------------------------------------------
# 14. HELPER: SMALL GALLERY
# -----------------------------------------------------------------------------
def show_image_step_gallery(
    step_title,
    step_results,
    df_key,
    stage="raw",          # "raw" | "kept" | "selected"
    selected_only=False,
):
    """
    Show a side-by-side gallery for a small set of images.

    Args:
        step_title:
            Title for this processing step.

        step_results:
            List of per-image result dictionaries.

        df_key:
            Key inside each step_results item to use as the box DataFrame.

        stage:
            Controls fixed overlay color.

        selected_only:
            If True, only selected poles are drawn.

    Returns:
        None
    """
    n = len(step_results)
    fig, axes = plt.subplots(1, n, figsize=(11 * n, 8))

    # Normalize the axis container for the single-image case.
    if n == 1:
        axes = [axes]

    for ax, item in zip(axes, step_results):
        title = f"{item['file_name']}\n{step_title}"

        # Add a note when production could not find a reliable pole.
        if item.get("image_selection_status") == "no_reliable_pole_found" and selected_only:
            title += "\n(no reliable pole found)"

        # Reload the image only when the gallery is drawn.
        with Image.open(item["image_path"]) as img:
            img = img.convert("RGB")

            # Record the original image size before resizing.
            orig_w, orig_h = img.size

            # Downsample for display only so gallery memory stays small.
            img.thumbnail((1600, 1600))

            # Read the resized display image size.
            disp_w, disp_h = img.size

            # Compute coordinate scaling from original image space to gallery image space.
            scale_x = disp_w / orig_w if orig_w > 0 else 1.0
            scale_y = disp_h / orig_h if orig_h > 0 else 1.0

            # Convert the resized display image to a NumPy array.
            image_rgb = np.array(img)

        draw_boxes_on_axis(
            ax=ax,
            image_rgb=image_rgb,
            boxes_df=item[df_key],
            title=title,
            stage=stage,
            selected_only=selected_only,
            scale_x=scale_x,
            scale_y=scale_y,
        )

    plt.tight_layout()
    plt.show()
    plt.close()

# -----------------------------------------------------------------------------
# 15. RUN POLE PRODUCTION PIPELINE ON ALL run_images_df ROWS
# -----------------------------------------------------------------------------
# Take all run rows for production processing.
pole_input_df = run_images_df.copy().reset_index(drop=True)

gallery_rows = []
all_candidate_rows = []
selection_rows = []

print(f"\nProcessing {len(pole_input_df)} image(s) for production pole selection...")

for idx, row in pole_input_df.iterrows():
    # Read key row values safely.
    image_id = row["image_id"] if "image_id" in row.index and pd.notna(row["image_id"]) else None
    file_name = row["file_name"] if "file_name" in row.index and pd.notna(row["file_name"]) else None
    image_path = row["image_path"] if "image_path" in row.index and pd.notna(row["image_path"]) else None

    # Stop early if the image path is missing or invalid.
    if pd.isna(image_path) or not isinstance(image_path, str) or len(image_path.strip()) == 0:
        raise ValueError("A production row is missing a valid image_path.")

    # Recover the filename from the path if needed.
    if (
        pd.isna(file_name)
        or not isinstance(file_name, str)
        or len(file_name.strip()) == 0
    ):
        file_name = os.path.basename(image_path) if isinstance(image_path, str) else "unknown_file"

    # Print occasional progress updates through the production loop.
    if (idx + 1) % 20 == 0 or idx == 0 or idx == len(pole_input_df) - 1:
        print(f"  [{idx + 1}/{len(pole_input_df)}] {file_name}")

    # Open the image safely and convert it to RGB.
    with Image.open(image_path) as img:
        image_pil = img.convert("RGB")
        image_pil.load()


    # Read image size for downstream use.
    image_w, image_h = image_pil.size

    # Run the raw SAM3 timber power pole prompt and collect every candidate box.
    raw_df = run_power_pole_raw_detections(
        image_pil=image_pil,
        image_id=image_id,
        file_name=file_name,
        image_path=image_path,
        prompt=POLE_PROMPT,
        text_threshold=POLE_TEXT_THRESHOLD,
        mask_threshold=POLE_MASK_THRESHOLD,
    )

    # Release the full-resolution PIL image once inference is done.
    del image_pil

    # Defensively normalize a None return to an empty DataFrame.
    if raw_df is None:
        raw_df = pd.DataFrame()

    # Enforce the expected return type.
    if not isinstance(raw_df, pd.DataFrame):
        raise TypeError(
            f"run_power_pole_raw_detections() must return a DataFrame, got: {type(raw_df)}"
        )

    # If there are no detections, record "no reliable pole found" for this image.
    if raw_df.empty:
        scored_df = raw_df.copy()
        selected_df = raw_df.copy()

        selection_rows.append({
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "image_w": image_w,
            "image_h": image_h,
            "prompt": POLE_PROMPT,
            "n_raw_candidates": 0,
            "n_kept_candidates": 0,
            "selection_status": "no_reliable_pole_found",
            "det_idx": np.nan,
            "score": np.nan,
            "x1": np.nan,
            "y1": np.nan,
            "x2": np.nan,
            "y2": np.nan,
            "pole_cx": np.nan,
            "pole_cy": np.nan,
            "final_score": np.nan,
        })
    else:
        # Compute ranking features for every raw candidate.
        feature_df = add_pole_features(raw_df)

        # Mark each candidate as kept or filtered out.
        prefilter_df = apply_pole_prefilter(feature_df)

        # Score the kept candidates and select the best one if it exists.
        scored_df, selected_df = score_pole_candidates(prefilter_df)

        # Save all scored candidates for the final candidate table.
        all_candidate_rows.append(scored_df)

        # Count how many raw and kept candidates this image produced.
        n_raw_candidates = len(raw_df)
        n_kept_candidates = (
            int(scored_df["is_kept_after_prefilter"].sum())
            if "is_kept_after_prefilter" in scored_df.columns
            else 0
        )

        # Record the no-selection case explicitly.
        if selected_df.empty:
            selection_rows.append({
                "image_id": image_id,
                "file_name": file_name,
                "image_path": image_path,
                "image_w": image_w,
                "image_h": image_h,
                "prompt": POLE_PROMPT,
                "n_raw_candidates": n_raw_candidates,
                "n_kept_candidates": n_kept_candidates,
                "selection_status": "no_reliable_pole_found",
                "det_idx": np.nan,
                "score": np.nan,
                "x1": np.nan,
                "y1": np.nan,
                "x2": np.nan,
                "y2": np.nan,
                "pole_cx": np.nan,
                "pole_cy": np.nan,
                "final_score": np.nan,
            })
        else:
            # Otherwise store the one selected production pole for this image.
            sel = selected_df.iloc[0]
            selection_rows.append({
                "image_id": sel["image_id"],
                "file_name": sel["file_name"],
                "image_path": sel["image_path"],
                "image_w": sel["image_w"],
                "image_h": sel["image_h"],
                "prompt": sel["prompt"],
                "n_raw_candidates": n_raw_candidates,
                "n_kept_candidates": n_kept_candidates,
                "selection_status": "selected",
                "det_idx": sel["det_idx"],
                "score": sel["score"],
                "x1": sel["x1"],
                "y1": sel["y1"],
                "x2": sel["x2"],
                "y2": sel["y2"],
                "pole_cx": sel["pole_cx"],
                "pole_cy": sel["pole_cy"],
                "final_score": sel["final_score"],
            })

    # Keep only the first few images for gallery display.
    if len(gallery_rows) < POLE_GALLERY_COUNT:
        gallery_rows.append({
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "raw_df": raw_df,
            "selected_df": selected_df,
            "image_selection_status": (
                "selected" if not selected_df.empty else "no_reliable_pole_found"
            ),
        })
# -----------------------------------------------------------------------------
# 16. BUILD FINAL OUTPUT TABLES
# -----------------------------------------------------------------------------
# Combine all scored candidate rows across all run images.
pole_candidates_df = (
    pd.concat(all_candidate_rows, ignore_index=True)
    if len(all_candidate_rows) > 0 else
    pd.DataFrame()
)

# Build the one-row-per-image production selection summary.
pole_selection_df = pd.DataFrame(selection_rows)

# -----------------------------------------------------------------------------
# 17. FINAL CANDIDATE TABLE
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("FINAL CANDIDATE TABLE")
print("=" * 100)

if pole_candidates_df.empty:
    print("No pole candidates were produced.")
else:
    candidate_cols = _existing_cols(
        pole_candidates_df,
        [
            "image_id",
            "file_name",
            "det_idx",
            "score",
            "box_w",
            "box_h",
            "box_area",
            "area_frac",
            "height_frac",
            "aspect_ratio",
            "x_center_score",
            "height_score",
            "area_score",
            "conf_score",
            "edge_score",
            "is_kept_after_prefilter",
            "final_score",
            "is_selected_pole",
            "selection_status",
        ]
    )

    # Display only columns that actually exist.
    _safe_display(
        pole_candidates_df[candidate_cols]
        .sort_values(
            ["file_name", "is_selected_pole", "final_score", "score"],
            ascending=[True, False, False, False]
        )
        .reset_index(drop=True)
    )

# -----------------------------------------------------------------------------
# 18. SELECTED POLE TABLE
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("SELECTED POLE TABLE")
print("=" * 100)

selection_cols = _existing_cols(
    pole_selection_df,
    [
        "image_id",
        "file_name",
        "selection_status",
        "n_raw_candidates",
        "n_kept_candidates",
        "det_idx",
        "score",
        "x1",
        "y1",
        "x2",
        "y2",
        "pole_cx",
        "pole_cy",
        "final_score",
    ]
)

# Display only columns that actually exist.
_safe_display(
    pole_selection_df[selection_cols].reset_index(drop=True)
)

# -----------------------------------------------------------------------------
# 19. RAW DETECTIONS GALLERY
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("RAW DETECTIONS GALLERY")
print("=" * 100)

show_image_step_gallery(
    step_title="RAW TIMBER POWER POLE DETECTIONS",
    step_results=gallery_rows,
    df_key="raw_df",
    stage="raw",
    selected_only=False,
)

# -----------------------------------------------------------------------------
# 20. FINAL SELECTED POLE GALLERY
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("FINAL SELECTED POLE GALLERY")
print("=" * 100)

show_image_step_gallery(
    step_title="FINAL SELECTED POLE",
    step_results=gallery_rows,
    df_key="selected_df",
    stage="selected",
    selected_only=True,
)

# -----------------------------------------------------------------------------
# 21. PERSIST ONLY PRODUCTION OUTPUTS
# -----------------------------------------------------------------------------
if "save_state" in globals():
    save_state(
        df_names=[
            name for name in [
                "pole_candidates_df",
                "pole_selection_df",
            ]
            if isinstance(globals().get(name), pd.DataFrame)
        ],
        config_extra={
            "POLE_PROMPT": POLE_PROMPT,
            "POLE_TEXT_THRESHOLD": POLE_TEXT_THRESHOLD,
            "POLE_MASK_THRESHOLD": POLE_MASK_THRESHOLD,
            "POLE_MIN_SCORE": POLE_MIN_SCORE,
            "POLE_MIN_AREA_FRAC": POLE_MIN_AREA_FRAC,
            "POLE_MIN_HEIGHT_FRAC": POLE_MIN_HEIGHT_FRAC,
            "POLE_MIN_ASPECT": POLE_MIN_ASPECT,
            "W_X_CENTER": W_X_CENTER,
            "W_HEIGHT": W_HEIGHT,
            "W_AREA": W_AREA,
            "W_CONF": W_CONF,
            "W_EDGE": W_EDGE,
            "POLE_GALLERY_COUNT": POLE_GALLERY_COUNT,
        },
        nb_globals=globals(),
    )

print("\nCell 7a production completed.")
print("Persisted outputs:")
print("  - pole_candidates_df")
print("  - pole_selection_df")

# 8a) DEBUG - ROI BUILD

In [ ]:
# =============================================================================
# CELL 8 — DRAWING CROPPING BOX
# =============================================================================
# EXPLANATION:
# This cell follows the production pole-selection step.
#
# WHAT THIS CELL DOES:
#   1) reads pole_selection_df from Cell 7a
#   2) keeps only rows where a reliable pole was selected
#   3) builds an expanded crop / ROI box around the selected pole
#   4) draws:
#        - selected pole box      = red solid
#        - expanded crop box      = cyan dashed
#   5) creates pole_rois_df for the next downstream crop/save step
#
# IMPORTANT:
# - this cell does NOT crop and save images yet
# - this cell only computes and visualizes the crop box geometry
# - this is the clean bridge between pole selection and actual ROI extraction
#
# INPUT:
# - pole_selection_df from Cell 7a
#
# OUTPUT:
# - pole_rois_df : one row per image, with selected pole + expanded ROI box
# =============================================================================

# -----------------------------------------------------------------------------
# 0. SAFETY CHECKS
# -----------------------------------------------------------------------------
required_globals = [
    "pole_selection_df",
]

missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise NameError(
        "Cell 8 cannot run because some required variables are missing.\n"
        "Please run Cell 7a first.\n"
        f"Missing globals: {missing_globals}"
    )

if not isinstance(pole_selection_df, pd.DataFrame):
    raise TypeError("pole_selection_df exists but is not a pandas DataFrame.")

if pole_selection_df.empty:
    raise ValueError("pole_selection_df is empty. Please check Cell 7a.")

# -----------------------------------------------------------------------------
# 1. CROP BOX CONFIG
# -----------------------------------------------------------------------------
# EXPLANATION:
# These settings are taken from your earlier cyan expanded-box logic.
#
# STRATEGY:
# - crop width comes mainly from pole height
# - top and bottom expansion are controlled separately
# - minimum width/top/bottom padding prevents boxes being too tight
# -----------------------------------------------------------------------------
EXPANDED_BOX_WIDTH_FACTOR_FROM_POLE_HEIGHT = 0.90
MIN_EXPANDED_BOX_WIDTH = 600

TOP_EXTRA_FACTOR_FROM_POLE_HEIGHT = 0.10
BOTTOM_EXTRA_FACTOR_FROM_POLE_HEIGHT = 0.20

MIN_TOP_EXTRA_PIXELS = 40
MIN_BOTTOM_EXTRA_PIXELS = 10

# Show only a small gallery
POLE_ROI_GALLERY_COUNT = min(4, len(pole_selection_df))

print("Crop-box config:")
print(f"  EXPANDED_BOX_WIDTH_FACTOR_FROM_POLE_HEIGHT : {EXPANDED_BOX_WIDTH_FACTOR_FROM_POLE_HEIGHT}")
print(f"  MIN_EXPANDED_BOX_WIDTH                     : {MIN_EXPANDED_BOX_WIDTH}")
print(f"  TOP_EXTRA_FACTOR_FROM_POLE_HEIGHT          : {TOP_EXTRA_FACTOR_FROM_POLE_HEIGHT}")
print(f"  BOTTOM_EXTRA_FACTOR_FROM_POLE_HEIGHT       : {BOTTOM_EXTRA_FACTOR_FROM_POLE_HEIGHT}")
print(f"  MIN_TOP_EXTRA_PIXELS                       : {MIN_TOP_EXTRA_PIXELS}")
print(f"  MIN_BOTTOM_EXTRA_PIXELS                    : {MIN_BOTTOM_EXTRA_PIXELS}")
print(f"  POLE_ROI_GALLERY_COUNT                     : {POLE_ROI_GALLERY_COUNT}")

# -----------------------------------------------------------------------------
# 2. HELPER: SAFE DISPLAY
# -----------------------------------------------------------------------------
def _safe_display(obj):
    """
    Display a pandas object in a notebook if possible; otherwise print it.

    Args:
        obj:
            Object to display.

    Returns:
        None
    """
    try:
        display(obj)
    except Exception:
        print(obj)

# -----------------------------------------------------------------------------
# 3. HELPER: BUILD EXPANDED BOX FROM SELECTED POLE
# -----------------------------------------------------------------------------
def _build_expanded_box_from_pole(
    pole_x1,
    pole_y1,
    pole_x2,
    pole_y2,
    image_w,
    image_h,
    width_factor_from_pole_height,
    min_box_width,
    top_extra_factor_from_pole_height,
    bottom_extra_factor_from_pole_height,
    min_top_extra_pixels,
    min_bottom_extra_pixels,
):
    """
    Build a larger ROI box around the selected pole.

    Args:
        pole_x1, pole_y1, pole_x2, pole_y2:
            Selected pole box coordinates.

        image_w, image_h:
            Full image dimensions.

        width_factor_from_pole_height:
            ROI width multiplier based on pole height.

        min_box_width:
            Minimum allowed ROI width in pixels.

        top_extra_factor_from_pole_height:
            Extra height above the pole top.

        bottom_extra_factor_from_pole_height:
            Extra height below the pole bottom.

        min_top_extra_pixels, min_bottom_extra_pixels:
            Minimum top / bottom expansion in pixels.

    Returns:
        dict:
            Expanded ROI geometry.
    """
    pole_w = max(0.0, pole_x2 - pole_x1)
    pole_h = max(0.0, pole_y2 - pole_y1)
    pole_cx = pole_x1 + pole_w / 2.0
    pole_cy = pole_y1 + pole_h / 2.0

    expanded_w = max(
        float(min_box_width),
        float(round(pole_h * width_factor_from_pole_height)),
    )

    top_extra = max(
        float(min_top_extra_pixels),
        float(round(pole_h * top_extra_factor_from_pole_height)),
    )

    bottom_extra = max(
        float(min_bottom_extra_pixels),
        float(round(pole_h * bottom_extra_factor_from_pole_height)),
    )

    half_w = expanded_w / 2.0

    ex1 = pole_cx - half_w
    ex2 = pole_cx + half_w

    ey1 = pole_y1 - top_extra
    ey2 = pole_y2 + bottom_extra

    # -------------------------------------------------------------------------
    # Clamp to image bounds
    # -------------------------------------------------------------------------
    if ex1 < 0:
        ex2 -= ex1
        ex1 = 0.0

    if ex2 > image_w:
        shift = ex2 - image_w
        ex1 -= shift
        ex2 = float(image_w)

    if ey1 < 0:
        ey1 = 0.0

    if ey2 > image_h:
        ey2 = float(image_h)

    ex1 = max(0.0, ex1)
    ex2 = min(float(image_w), ex2)
    ey1 = max(0.0, ey1)
    ey2 = min(float(image_h), ey2)

    final_w = max(0.0, ex2 - ex1)
    final_h = max(0.0, ey2 - ey1)

    return {
        "expanded_x1": float(ex1),
        "expanded_y1": float(ey1),
        "expanded_x2": float(ex2),
        "expanded_y2": float(ey2),
        "expanded_w": float(final_w),
        "expanded_h": float(final_h),
        "pole_w": float(pole_w),
        "pole_h": float(pole_h),
        "pole_cx": float(pole_cx),
        "pole_cy": float(pole_cy),
    }

# -----------------------------------------------------------------------------
# 4. HELPER: DRAW SELECTED POLE + EXPANDED CROP BOX
# -----------------------------------------------------------------------------
def draw_selected_pole_and_crop_box(ax, image_rgb, row, title):
    """
    Draw the selected pole box and the expanded crop box.

    VISUAL STYLE:
    - selected pole box = red solid
    - expanded crop box = cyan dashed

    Args:
        ax:
            Matplotlib axis.

        image_rgb:
            RGB numpy image.

        row:
            Row-like object containing selected pole + crop box coordinates.

        title:
            Plot title.

    Returns:
        None
    """
    import matplotlib.patches as patches

    ax.imshow(image_rgb)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

    # -------------------------------------------------------------------------
    # Red selected pole box
    # -------------------------------------------------------------------------
    pole_x1 = float(row["x1"])
    pole_y1 = float(row["y1"])
    pole_x2 = float(row["x2"])
    pole_y2 = float(row["y2"])

    pole_w = pole_x2 - pole_x1
    pole_h = pole_y2 - pole_y1

    raw_rect = patches.Rectangle(
        (pole_x1, pole_y1),
        pole_w,
        pole_h,
        linewidth=3.0,
        edgecolor="red",
        facecolor="none",
    )
    ax.add_patch(raw_rect)

    # -------------------------------------------------------------------------
    # Cyan dashed expanded crop box
    # -------------------------------------------------------------------------
    ex1 = float(row["expanded_x1"])
    ey1 = float(row["expanded_y1"])
    ew = float(row["expanded_w"])
    eh = float(row["expanded_h"])

    expanded_rect = patches.Rectangle(
        (ex1, ey1),
        ew,
        eh,
        linewidth=2.4,
        edgecolor="cyan",
        linestyle="--",
        facecolor="none",
        alpha=0.95,
    )
    ax.add_patch(expanded_rect)

    # -------------------------------------------------------------------------
    # Red label for selected pole
    # -------------------------------------------------------------------------
    prompt_text = None
    if "prompt" in row and pd.notna(row["prompt"]) and str(row["prompt"]).strip():
        prompt_text = str(row["prompt"]).strip()
    elif "POLE_PROMPT" in globals():
        prompt_text = str(POLE_PROMPT).strip()
    else:
        prompt_text = "timber power pole"

    pole_label = f"POLE | {prompt_text}"
    if "score" in row and pd.notna(row["score"]):
        pole_label += f" | score={float(row['score']):.3f}"

    ax.text(
        pole_x1,
        max(5, pole_y1 - 6),
        pole_label,
        color="white",
        fontsize=8.5,
        bbox=dict(facecolor="red", alpha=0.95, pad=1.5, edgecolor="none"),
    )

    # -------------------------------------------------------------------------
    # Cyan label for expanded crop box
    # -------------------------------------------------------------------------
    crop_label = f"CROP BOX | {int(round(ew))}x{int(round(eh))}"

    ax.text(
        ex1,
        min(ey1 + 18, row["image_h"] - 10),
        crop_label,
        color="black",
        fontsize=8.0,
        bbox=dict(facecolor="cyan", alpha=0.90, pad=1.5, edgecolor="none"),
    )

# -----------------------------------------------------------------------------
# 5. BUILD pole_rois_df FROM pole_selection_df
# -----------------------------------------------------------------------------
roi_rows = []
gallery_rows = []

for idx, row in pole_selection_df.reset_index(drop=True).iterrows():
    image_id = row.get("image_id", None)
    file_name = row.get("file_name", None)
    image_path = row.get("image_path", None)
    selection_status = row.get("selection_status", "unknown")

    if pd.isna(file_name) or not isinstance(file_name, str) or len(file_name.strip()) == 0:
        if isinstance(image_path, str):
            file_name = os.path.basename(image_path)
        else:
            file_name = None

    # -------------------------------------------------------------------------
    # If no reliable pole was found, keep a row but do not build a crop box
    # -------------------------------------------------------------------------
    if selection_status != "selected":
        roi_rows.append({
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "selection_status": selection_status,
            "roi_status": "no_reliable_pole_found",
            "prompt": row.get("prompt", globals().get("POLE_PROMPT", "timber power pole")),
            "score": np.nan,
            "x1": np.nan,
            "y1": np.nan,
            "x2": np.nan,
            "y2": np.nan,
            "pole_w": np.nan,
            "pole_h": np.nan,
            "pole_cx": np.nan,
            "pole_cy": np.nan,
            "image_w": row.get("image_w", np.nan),
            "image_h": row.get("image_h", np.nan),
            "expanded_x1": np.nan,
            "expanded_y1": np.nan,
            "expanded_x2": np.nan,
            "expanded_y2": np.nan,
            "expanded_w": np.nan,
            "expanded_h": np.nan,
        })
        continue

    # -------------------------------------------------------------------------
    # Load image so crop box is based on true image size
    # -------------------------------------------------------------------------
    if not isinstance(image_path, str) or len(image_path.strip()) == 0:
        raise ValueError(f"Invalid image_path for selected row: {image_path}")

    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image file not found: {image_path}")

    with Image.open(image_path) as img:
        image = img.convert("RGB")
        image.load()

    image_w, image_h = image.size
    image_rgb = np.array(image)

    pole_x1 = float(row["x1"])
    pole_y1 = float(row["y1"])
    pole_x2 = float(row["x2"])
    pole_y2 = float(row["y2"])

    expanded_box = _build_expanded_box_from_pole(
        pole_x1=pole_x1,
        pole_y1=pole_y1,
        pole_x2=pole_x2,
        pole_y2=pole_y2,
        image_w=image_w,
        image_h=image_h,
        width_factor_from_pole_height=EXPANDED_BOX_WIDTH_FACTOR_FROM_POLE_HEIGHT,
        min_box_width=MIN_EXPANDED_BOX_WIDTH,
        top_extra_factor_from_pole_height=TOP_EXTRA_FACTOR_FROM_POLE_HEIGHT,
        bottom_extra_factor_from_pole_height=BOTTOM_EXTRA_FACTOR_FROM_POLE_HEIGHT,
        min_top_extra_pixels=MIN_TOP_EXTRA_PIXELS,
        min_bottom_extra_pixels=MIN_BOTTOM_EXTRA_PIXELS,
    )

    roi_rows.append({
        "image_id": image_id,
        "file_name": file_name,
        "image_path": image_path,
        "selection_status": selection_status,
        "roi_status": "ok",
        "prompt": row.get("prompt", globals().get("POLE_PROMPT", "timber power pole")),
        "score": row.get("score", np.nan),
        "x1": pole_x1,
        "y1": pole_y1,
        "x2": pole_x2,
        "y2": pole_y2,
        "pole_w": expanded_box["pole_w"],
        "pole_h": expanded_box["pole_h"],
        "pole_cx": expanded_box["pole_cx"],
        "pole_cy": expanded_box["pole_cy"],
        "image_w": int(image_w),
        "image_h": int(image_h),
        "expanded_x1": expanded_box["expanded_x1"],
        "expanded_y1": expanded_box["expanded_y1"],
        "expanded_x2": expanded_box["expanded_x2"],
        "expanded_y2": expanded_box["expanded_y2"],
        "expanded_w": expanded_box["expanded_w"],
        "expanded_h": expanded_box["expanded_h"],
    })

    if len(gallery_rows) < POLE_ROI_GALLERY_COUNT:
        gallery_rows.append({
            "image_id": image_id,
            "file_name": file_name,
            "image_rgb": image_rgb,
            "row": {
                **roi_rows[-1],
                "image_h": image_h,
            },
        })

# -----------------------------------------------------------------------------
# 6. FINAL OUTPUT TABLE
# -----------------------------------------------------------------------------
pole_rois_df = pd.DataFrame(roi_rows)

print("\n" + "=" * 100)
print("POLE ROI TABLE")
print("=" * 100)

_safe_display(
    pole_rois_df[
        [
            "image_id",
            "file_name",
            "selection_status",
            "roi_status",
            "score",
            "x1",
            "y1",
            "x2",
            "y2",
            "pole_cx",
            "pole_cy",
            "expanded_x1",
            "expanded_y1",
            "expanded_x2",
            "expanded_y2",
            "expanded_w",
            "expanded_h",
        ]
    ].reset_index(drop=True)
)

# -----------------------------------------------------------------------------
# 7. GALLERY — SELECTED POLE + CROP BOX
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("DRAWING CROPPING BOX GALLERY")
print("=" * 100)

if len(gallery_rows) == 0:
    print("No selected poles were available for crop-box drawing.")
else:
    n = len(gallery_rows)
    fig, axes = plt.subplots(1, n, figsize=(11 * n, 8))

    if n == 1:
        axes = [axes]

    for ax, item in zip(axes, gallery_rows):
        draw_selected_pole_and_crop_box(
            ax=ax,
            image_rgb=item["image_rgb"],
            row=item["row"],
            title=f"{item['file_name']}\nSTEP 8 — DRAWING CROPPING BOX",
        )

    plt.tight_layout()
    plt.show()
    plt.close()

# -----------------------------------------------------------------------------
# 8. SAVE STATE
# -----------------------------------------------------------------------------
if "save_state" in globals():
    save_state(
        df_names=[
            name for name in [
                "pole_selection_df",
                "pole_rois_df",
            ]
            if isinstance(globals().get(name), pd.DataFrame)
        ],
        config_extra={
            "EXPANDED_BOX_WIDTH_FACTOR_FROM_POLE_HEIGHT": EXPANDED_BOX_WIDTH_FACTOR_FROM_POLE_HEIGHT,
            "MIN_EXPANDED_BOX_WIDTH": MIN_EXPANDED_BOX_WIDTH,
            "TOP_EXTRA_FACTOR_FROM_POLE_HEIGHT": TOP_EXTRA_FACTOR_FROM_POLE_HEIGHT,
            "BOTTOM_EXTRA_FACTOR_FROM_POLE_HEIGHT": BOTTOM_EXTRA_FACTOR_FROM_POLE_HEIGHT,
            "MIN_TOP_EXTRA_PIXELS": MIN_TOP_EXTRA_PIXELS,
            "MIN_BOTTOM_EXTRA_PIXELS": MIN_BOTTOM_EXTRA_PIXELS,
            "POLE_ROI_GALLERY_COUNT": POLE_ROI_GALLERY_COUNT,
        },
        nb_globals=globals(),
    )

print("\nCell 8 completed.")
print("Saved outputs:")
print("  - pole_rois_df")

# 8a) PROD ROI BUILD + CROP + SAVE TO SILVER

In [ ]:
# =============================================================================
# CELL 8 — PRODUCTION POLE-TOP FIXED CANVAS ROI + SHIFT + PAD + SAVE TO SILVER
# =============================================================================
# EXPLANATION:
# Build one fixed-size pole-top ROI for every selected pole.
#
# STRATEGY:
# - anchor from the selected pole top
# - keep the final ROI size fixed for every image
# - shift the requested box inside the image first when possible
# - pad only if needed
#
# RESULT:
# - every saved ROI image has exactly the same size
# - the pole-top hardware zone is framed more consistently
# - the gallery shows debug-style labels for both the pole box and crop box
# - draws:
#        - selected pole box      = red solid
#        - expanded crop box      = cyan dashed
# =============================================================================

# -----------------------------------------------------------------------------
# 0. SAFETY CHECKS
# -----------------------------------------------------------------------------
required_globals = [
    "pole_selection_df",
    "SILVER_POLE_ROIS",
    "save_state",
]

# Check that all required globals exist before continuing.
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise NameError(
        "Cell 8 cannot run because some required variables are missing.\n"
        "Please run Cell 7a first.\n"
        f"Missing globals: {missing_globals}"
    )

# Confirm the pole selection table is a pandas DataFrame.
if not isinstance(pole_selection_df, pd.DataFrame):
    raise TypeError("pole_selection_df exists but is not a pandas DataFrame.")

# Stop early if the selection table is empty.
if pole_selection_df.empty:
    raise ValueError("pole_selection_df is empty. Please check Cell 7a.")

# -----------------------------------------------------------------------------
# 1. KEEP ONLY SELECTED POLES
# -----------------------------------------------------------------------------
# Keep only rows where a reliable pole was selected.
if "selection_status" in pole_selection_df.columns:
    selected_poles_df = pole_selection_df[
        pole_selection_df["selection_status"] == "selected"
    ].copy()
else:
    selected_poles_df = pole_selection_df[
        pole_selection_df[["x1", "y1", "x2", "y2"]].notna().all(axis=1)
    ].copy()

# Stop early if there are no selected poles.
if selected_poles_df.empty:
    raise ValueError(
        "No selected poles were found in pole_selection_df.\n"
        "Please check Cell 7a production output."
    )

# -----------------------------------------------------------------------------
# 2. FIXED POLE-TOP ROI CONFIG
# -----------------------------------------------------------------------------
# Use one fixed final canvas size for every image.
FIXED_ROI_WIDTH = 2600
FIXED_ROI_HEIGHT = 3500

# Start the requested ROI this many pixels above the selected pole top.
POLE_TOP_BUFFER_ABOVE = 350

# Use a black background if padding is needed.
PAD_RGB = (0, 0, 0)

# Show only a small gallery of selected poles.
POLE_ROI_GALLERY_COUNT = min(6, len(selected_poles_df))

print("Fixed pole-top ROI config:")
print(f"  FIXED_ROI_WIDTH        : {FIXED_ROI_WIDTH}")
print(f"  FIXED_ROI_HEIGHT       : {FIXED_ROI_HEIGHT}")
print(f"  POLE_TOP_BUFFER_ABOVE  : {POLE_TOP_BUFFER_ABOVE}")
print(f"  PAD_RGB                : {PAD_RGB}")
print(f"  POLE_ROI_GALLERY_COUNT : {POLE_ROI_GALLERY_COUNT}")

# -----------------------------------------------------------------------------
# 3. HELPER: SAFE DISPLAY
# -----------------------------------------------------------------------------
def _safe_display(obj):
    """
    Display a pandas object in a notebook if possible; otherwise print it.
    """
    try:
        display(obj)
    except Exception:
        print(obj)

# -----------------------------------------------------------------------------
# 4. HELPER: SAFE COLUMN SUBSET
# -----------------------------------------------------------------------------
def _existing_cols(df, cols):
    """
    Return only the requested columns that actually exist in the DataFrame.
    """
    if df is None or not isinstance(df, pd.DataFrame):
        return []

    return [c for c in cols if c in df.columns]

# -----------------------------------------------------------------------------
# 5. HELPER: SAFE FILE-STEM
# -----------------------------------------------------------------------------
def make_safe_stem(text):
    """
    Convert a string into a filesystem-safe stem.
    """
    if text is None or (isinstance(text, float) and pd.isna(text)):
        text = "image"

    text = str(text).strip()
    if len(text) == 0:
        text = "image"

    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")

    return text if len(text) > 0 else "image"

# -----------------------------------------------------------------------------
# 6. HELPER: SHIFT A FIXED BOX INSIDE THE IMAGE WHEN POSSIBLE
# -----------------------------------------------------------------------------
def shift_box_inside_image(x1, y1, box_w, box_h, image_w, image_h):
    """
    Shift a fixed-size box inside the image when possible, while keeping the
    box size unchanged.

    Returns:
        Dict[str, int]:
            Shifted fixed-size box coordinates.
    """
    x1 = int(round(x1))
    y1 = int(round(y1))
    box_w = int(box_w)
    box_h = int(box_h)

    x2 = x1 + box_w
    y2 = y1 + box_h

    # Shift horizontally if the full box can fit inside the image.
    if image_w >= box_w:
        if x1 < 0:
            x2 += (-x1)
            x1 = 0
        if x2 > image_w:
            x1 -= (x2 - image_w)
            x2 = image_w

    # Shift vertically if the full box can fit inside the image.
    if image_h >= box_h:
        if y1 < 0:
            y2 += (-y1)
            y1 = 0
        if y2 > image_h:
            y1 -= (y2 - image_h)
            y2 = image_h

    # Recompute the far corner from the final fixed-size origin.
    x2 = x1 + box_w
    y2 = y1 + box_h

    return {
        "req_x1": int(x1),
        "req_y1": int(y1),
        "req_x2": int(x2),
        "req_y2": int(y2),
        "req_w": int(box_w),
        "req_h": int(box_h),
    }

# -----------------------------------------------------------------------------
# 7. HELPER: BUILD THE FIXED POLE-TOP ROI REQUEST
# -----------------------------------------------------------------------------
def build_pole_top_roi_request(row):
    """
    Build a fixed-size pole-top ROI request from the selected pole row.

    Returns:
        Dict[str, Any]:
            Requested fixed-size ROI geometry after shift-to-fit.
    """
    # Read the selected pole box.
    x1 = float(row["x1"])
    y1 = float(row["y1"])
    x2 = float(row["x2"])
    y2 = float(row["y2"])

    image_w = int(row["image_w"])
    image_h = int(row["image_h"])

    # Compute basic pole geometry.
    pole_w = max(x2 - x1, 1.0)
    pole_h = max(y2 - y1, 1.0)

    # Use the provided pole center when available.
    if "pole_cx" in row.index and pd.notna(row["pole_cx"]):
        pole_cx = float(row["pole_cx"])
    else:
        pole_cx = (x1 + x2) / 2.0

    # Anchor the requested ROI from the pole top.
    req_x1 = pole_cx - (FIXED_ROI_WIDTH / 2.0)
    req_y1 = y1 - POLE_TOP_BUFFER_ABOVE

    # Shift the fixed-size ROI inside the image first when possible.
    shifted = shift_box_inside_image(
        x1=req_x1,
        y1=req_y1,
        box_w=FIXED_ROI_WIDTH,
        box_h=FIXED_ROI_HEIGHT,
        image_w=image_w,
        image_h=image_h,
    )

    return {
        "pole_w": float(pole_w),
        "pole_h": float(pole_h),
        "pole_cx_used": float(pole_cx),
        "req_x1": int(shifted["req_x1"]),
        "req_y1": int(shifted["req_y1"]),
        "req_x2": int(shifted["req_x2"]),
        "req_y2": int(shifted["req_y2"]),
        "req_w": int(shifted["req_w"]),
        "req_h": int(shifted["req_h"]),
    }

# -----------------------------------------------------------------------------
# 8. HELPER: RENDER A FIXED-SIZE CANVAS FROM THE REQUESTED ROI
# -----------------------------------------------------------------------------
def render_fixed_canvas_roi(image_pil, roi_request):
    """
    Render the final fixed-size ROI canvas by cropping the overlapping source
    region and pasting it onto a fixed-size canvas.

    Returns:
        Dict[str, Any]:
            Final crop details and the fixed-size PIL canvas.
    """
    image_w, image_h = image_pil.size

    req_x1 = int(roi_request["req_x1"])
    req_y1 = int(roi_request["req_y1"])
    req_x2 = int(roi_request["req_x2"])
    req_y2 = int(roi_request["req_y2"])

    # Compute the overlapping source region.
    src_x1 = max(0, req_x1)
    src_y1 = max(0, req_y1)
    src_x2 = min(image_w, req_x2)
    src_y2 = min(image_h, req_y2)

    overlap_w = max(0, src_x2 - src_x1)
    overlap_h = max(0, src_y2 - src_y1)

    # Compute where the source crop should be pasted on the fixed canvas.
    dst_x1 = max(0, src_x1 - req_x1)
    dst_y1 = max(0, src_y1 - req_y1)

    # Create the fixed-size final canvas.
    roi_canvas = Image.new("RGB", (FIXED_ROI_WIDTH, FIXED_ROI_HEIGHT), PAD_RGB)

    # Paste the overlapping image crop onto the fixed-size canvas.
    if overlap_w > 0 and overlap_h > 0:
        src_crop = image_pil.crop((src_x1, src_y1, src_x2, src_y2))
        roi_canvas.paste(src_crop, (dst_x1, dst_y1))
    else:
        src_crop = None

    # Compute padding sizes on each side.
    pad_left = int(max(0, -req_x1))
    pad_top = int(max(0, -req_y1))
    pad_right = int(max(0, req_x2 - image_w))
    pad_bottom = int(max(0, req_y2 - image_h))

    return {
        "roi_canvas": roi_canvas,
        "src_x1": int(src_x1),
        "src_y1": int(src_y1),
        "src_x2": int(src_x2),
        "src_y2": int(src_y2),
        "src_w": int(overlap_w),
        "src_h": int(overlap_h),
        "dst_x1": int(dst_x1),
        "dst_y1": int(dst_y1),
        "pad_left": int(pad_left),
        "pad_top": int(pad_top),
        "pad_right": int(pad_right),
        "pad_bottom": int(pad_bottom),
        "was_padded": bool(
            (pad_left > 0) or
            (pad_top > 0) or
            (pad_right > 0) or
            (pad_bottom > 0)
        ),
    }

# -----------------------------------------------------------------------------
# 9. HELPER: BUILD DEBUG-STYLE LABELS
# -----------------------------------------------------------------------------
def build_pole_overlay_label(row):
    """
    Build the red pole-box label shown on the source-image gallery.
    """
    label_bits = ["POLE"]

    # Prefer the row-level prompt if available.
    prompt_text = None
    if "prompt" in row.index and pd.notna(row["prompt"]) and str(row["prompt"]).strip():
        prompt_text = str(row["prompt"]).strip()
    elif "POLE_PROMPT" in globals():
        prompt_text = str(POLE_PROMPT).strip()

    if prompt_text:
        label_bits.append(prompt_text)

    # Add raw confidence score when present.
    if "score" in row.index and pd.notna(row["score"]):
        label_bits.append(f"score={float(row['score']):.3f}")

    # Add final score when present.
    if "final_score" in row.index and pd.notna(row["final_score"]):
        label_bits.append(f"final={float(row['final_score']):.3f}")

    return " | ".join(label_bits)

def build_crop_overlay_label(row):
    """
    Build the cyan crop-box label shown on the source-image gallery.
    """
    label_bits = ["CROP BOX"]

    if "req_w" in row.index and "req_h" in row.index:
        if pd.notna(row["req_w"]) and pd.notna(row["req_h"]):
            label_bits.append(f"{int(row['req_w'])}x{int(row['req_h'])}")

    if "was_padded" in row.index and bool(row["was_padded"]):
        label_bits.append("padded")

    return " | ".join(label_bits)

# -----------------------------------------------------------------------------
# 10. HELPER: DRAW SOURCE IMAGE WITH POLE + ROI BOXES
# -----------------------------------------------------------------------------
def draw_source_and_roi_gallery(pole_rois_df, gallery_count=6):
    """
    Show a small gallery with the selected pole box and the requested ROI box.
    """
    import matplotlib.patches as patches

    gallery_df = pole_rois_df.head(gallery_count).copy()

    if gallery_df.empty:
        print("No ROI gallery rows to display.")
        return

    fig, axes = plt.subplots(len(gallery_df), 2, figsize=(16, 6 * len(gallery_df)))

    # Normalize axes shape for the single-row case.
    if len(gallery_df) == 1:
        axes = np.array([axes])

    for row_idx, (_, r) in enumerate(gallery_df.iterrows()):
        ax_left = axes[row_idx, 0]
        ax_right = axes[row_idx, 1]

        # Reload the source image only when the gallery is drawn.
        with Image.open(r["image_path"]) as img:
            img = img.convert("RGB")

            # Record original size before resizing for display.
            orig_w, orig_h = img.size

            # Downsample for display only.
            img.thumbnail((1600, 1600))

            # Compute scaling from original coordinates to display coordinates.
            disp_w, disp_h = img.size
            scale_x = disp_w / orig_w if orig_w > 0 else 1.0
            scale_y = disp_h / orig_h if orig_h > 0 else 1.0

            # Convert the display image to NumPy.
            image_rgb = np.array(img)

        # Show the source image.
        ax_left.imshow(image_rgb)
        ax_left.set_title(f"{r['file_name']}\nselected pole + requested ROI", fontsize=11)
        ax_left.axis("off")

        # Scale the selected pole box to the display image.
        pole_x1 = float(r["x1"]) * scale_x
        pole_y1 = float(r["y1"]) * scale_y
        pole_w = (float(r["x2"]) - float(r["x1"])) * scale_x
        pole_h = (float(r["y2"]) - float(r["y1"])) * scale_y

        # Draw the selected pole box in red.
        pole_rect = patches.Rectangle(
            (pole_x1, pole_y1),
            pole_w,
            pole_h,
            linewidth=3.0,
            edgecolor="red",
            facecolor="none",
        )
        ax_left.add_patch(pole_rect)

        # Add the red pole label in the same style as the debug cell.
        pole_label = build_pole_overlay_label(r)
        ax_left.text(
            max(pole_x1, 8),
            max(pole_y1 - 6, 8),
            pole_label,
            fontsize=8.5,
            color="white",
            bbox=dict(
                facecolor="red",
                alpha=0.85,
                edgecolor="none",
                pad=2.0,
            ),
        )

        # Scale the requested crop box to the display image.
        crop_x1 = float(r["req_x1"]) * scale_x
        crop_y1 = float(r["req_y1"]) * scale_y
        crop_w = (float(r["req_x2"]) - float(r["req_x1"])) * scale_x
        crop_h = (float(r["req_y2"]) - float(r["req_y1"])) * scale_y

        # Draw the requested ROI box in cyan dashed.
        roi_rect = patches.Rectangle(
            (crop_x1, crop_y1),
            crop_w,
            crop_h,
            linewidth=2.5,
            edgecolor="cyan",
            facecolor="none",
            linestyle="--",
        )
        ax_left.add_patch(roi_rect)

        # Add the cyan crop-box label in the same style as the debug cell.
        crop_label = build_crop_overlay_label(r)
        ax_left.text(
            max(crop_x1, 8),
            max(crop_y1 - 6, 8),
            crop_label,
            fontsize=8.5,
            color="black",
            bbox=dict(
                facecolor="cyan",
                alpha=0.85,
                edgecolor="none",
                pad=2.0,
            ),
        )

        # Load and show the saved fixed-size ROI.
        with Image.open(r["roi_image_path"]) as crop_img:
            crop_img = crop_img.convert("RGB")
            crop_rgb = np.array(crop_img)

        crop_title = f"{r['roi_file_name']}\nROI size={int(r['roi_w'])}x{int(r['roi_h'])}"
        if "was_padded" in r and bool(r["was_padded"]):
            crop_title += " | padded"

        ax_right.imshow(crop_rgb)
        ax_right.set_title(crop_title, fontsize=11)
        ax_right.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close()

# -----------------------------------------------------------------------------
# 11. RESET THE SILVER ROI FOLDER
# -----------------------------------------------------------------------------
# Clear the old ROI folder so there are no stale crops from a previous run.
if os.path.isdir(SILVER_POLE_ROIS):
    shutil.rmtree(SILVER_POLE_ROIS)

os.makedirs(SILVER_POLE_ROIS, exist_ok=True)

# -----------------------------------------------------------------------------
# 12. BUILD FIXED-SIZE ROI CANVASES AND SAVE TO SILVER
# -----------------------------------------------------------------------------
roi_rows = []

print(f"\nCreating fixed pole-top ROI crops for {len(selected_poles_df)} selected image(s)...")

for idx, row in selected_poles_df.iterrows():
    # Read key values safely.
    image_id = row["image_id"] if "image_id" in row.index and pd.notna(row["image_id"]) else None
    file_name = row["file_name"] if "file_name" in row.index and pd.notna(row["file_name"]) else None
    image_path = row["image_path"] if "image_path" in row.index and pd.notna(row["image_path"]) else None

    # Stop early if the image path is missing or invalid.
    if pd.isna(image_path) or not isinstance(image_path, str) or len(image_path.strip()) == 0:
        raise ValueError("A selected pole row is missing a valid image_path.")

    # Recover the filename from the path if needed.
    if (
        pd.isna(file_name)
        or not isinstance(file_name, str)
        or len(file_name.strip()) == 0
    ):
        file_name = os.path.basename(image_path)

    # Build the fixed pole-top ROI request.
    roi_request = build_pole_top_roi_request(row)

    # Build a stable saved crop filename.
    roi_stem = make_safe_stem(
        image_id if image_id is not None else os.path.splitext(file_name)[0]
    )
    roi_file_name = f"{roi_stem}__pole_roi.png"
    roi_image_path = os.path.join(SILVER_POLE_ROIS, roi_file_name)

    # Open the source image and render the fixed-size ROI canvas.
    with Image.open(image_path) as img:
        img = img.convert("RGB")

        roi_render = render_fixed_canvas_roi(
            image_pil=img,
            roi_request=roi_request,
        )

        # Save the fixed-size ROI canvas to Silver.
        roi_render["roi_canvas"].save(roi_image_path, format="PNG")

    # Add one manifest row for this saved ROI crop.
    roi_rows.append({
        "image_id": image_id,
        "file_name": file_name,
        "image_path": image_path,
        "selection_status": "selected",
        "prompt": row["prompt"] if "prompt" in row.index and pd.notna(row["prompt"]) else None,
        "score": float(row["score"]) if "score" in row.index and pd.notna(row["score"]) else np.nan,
        "final_score": float(row["final_score"]) if "final_score" in row.index and pd.notna(row["final_score"]) else np.nan,
        "det_idx": int(row["det_idx"]) if "det_idx" in row.index and pd.notna(row["det_idx"]) else np.nan,
        "image_w": int(row["image_w"]),
        "image_h": int(row["image_h"]),
        "x1": float(row["x1"]),
        "y1": float(row["y1"]),
        "x2": float(row["x2"]),
        "y2": float(row["y2"]),
        "pole_cx": float(row["pole_cx"]) if "pole_cx" in row.index and pd.notna(row["pole_cx"]) else np.nan,
        "pole_cy": float(row["pole_cy"]) if "pole_cy" in row.index and pd.notna(row["pole_cy"]) else np.nan,
        "pole_w": float(roi_request["pole_w"]),
        "pole_h": float(roi_request["pole_h"]),
        "pole_cx_used": float(roi_request["pole_cx_used"]),
        "req_x1": int(roi_request["req_x1"]),
        "req_y1": int(roi_request["req_y1"]),
        "req_x2": int(roi_request["req_x2"]),
        "req_y2": int(roi_request["req_y2"]),
        "req_w": int(roi_request["req_w"]),
        "req_h": int(roi_request["req_h"]),
        "src_x1": int(roi_render["src_x1"]),
        "src_y1": int(roi_render["src_y1"]),
        "src_x2": int(roi_render["src_x2"]),
        "src_y2": int(roi_render["src_y2"]),
        "src_w": int(roi_render["src_w"]),
        "src_h": int(roi_render["src_h"]),
        "dst_x1": int(roi_render["dst_x1"]),
        "dst_y1": int(roi_render["dst_y1"]),
        "pad_left": int(roi_render["pad_left"]),
        "pad_top": int(roi_render["pad_top"]),
        "pad_right": int(roi_render["pad_right"]),
        "pad_bottom": int(roi_render["pad_bottom"]),
        "was_padded": bool(roi_render["was_padded"]),
        "roi_w": int(FIXED_ROI_WIDTH),
        "roi_h": int(FIXED_ROI_HEIGHT),
        "roi_file_name": roi_file_name,
        "roi_image_path": roi_image_path,
        "source_layer": "silver",
        "source_folder": SILVER_POLE_ROIS,
    })

    # Print occasional progress updates.
    if (len(roi_rows) % 20 == 0) or (len(roi_rows) == 1) or (len(roi_rows) == len(selected_poles_df)):
        print(f"  [{len(roi_rows)}/{len(selected_poles_df)}] {roi_file_name}")

    # Release the loop-local render object.
    del roi_render

# Run one cleanup pass after all crops are saved.
gc.collect()

# -----------------------------------------------------------------------------
# 13. BUILD THE ROI MANIFEST
# -----------------------------------------------------------------------------
pole_rois_df = pd.DataFrame(roi_rows)

if pole_rois_df.empty:
    raise ValueError("pole_rois_df ended up empty. Please check the crop/save loop.")

# -----------------------------------------------------------------------------
# 14. PRINT SUMMARY
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("FIXED POLE-TOP ROI SUMMARY")
print("=" * 100)
print(f"Selected poles used      : {len(selected_poles_df)}")
print(f"Saved ROI crops          : {len(pole_rois_df)}")
print(f"Silver ROI folder        : {SILVER_POLE_ROIS}")
print(f"Fixed ROI size           : {FIXED_ROI_WIDTH}x{FIXED_ROI_HEIGHT}")
print(
    f"Padded ROI count         : "
    f"{int(pole_rois_df['was_padded'].sum()) if 'was_padded' in pole_rois_df.columns else 0}"
)

# -----------------------------------------------------------------------------
# 15. DISPLAY THE ROI TABLE
# -----------------------------------------------------------------------------
roi_cols = _existing_cols(
    pole_rois_df,
    [
        "image_id",
        "file_name",
        "score",
        "final_score",
        "pole_w",
        "pole_h",
        "req_x1",
        "req_y1",
        "req_x2",
        "req_y2",
        "roi_w",
        "roi_h",
        "pad_left",
        "pad_top",
        "pad_right",
        "pad_bottom",
        "was_padded",
        "roi_file_name",
        "roi_image_path",
    ]
)

# Display only columns that actually exist.
_safe_display(
    pole_rois_df[roi_cols]
    .sort_values(["file_name"], ascending=[True])
    .reset_index(drop=True)
)

# -----------------------------------------------------------------------------
# 16. SHOW A SMALL ROI GALLERY
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("FIXED POLE-TOP ROI GALLERY")
print("=" * 100)

draw_source_and_roi_gallery(
    pole_rois_df=pole_rois_df,
    gallery_count=POLE_ROI_GALLERY_COUNT,
)

# -----------------------------------------------------------------------------
# 17. SAVE OUTPUTS INTO NOTEBOOK STATE
# -----------------------------------------------------------------------------
if "save_state" in globals():
    save_state(
        df_names=[
            name for name in [
                "pole_selection_df",
                "pole_rois_df",
            ]
            if isinstance(globals().get(name), pd.DataFrame)
        ],
        config_extra={
            "FIXED_ROI_WIDTH": FIXED_ROI_WIDTH,
            "FIXED_ROI_HEIGHT": FIXED_ROI_HEIGHT,
            "POLE_TOP_BUFFER_ABOVE": POLE_TOP_BUFFER_ABOVE,
            "PAD_RGB": list(PAD_RGB),
            "POLE_ROI_GALLERY_COUNT": POLE_ROI_GALLERY_COUNT,
        },
        nb_globals=globals(),
    )

print("\nCell 8 completed.")
print("Persisted outputs:")
print("  - pole_rois_df")

In [ ]:
# =============================================================================
# CELL 9a — STREAM 1 BUG CELL: ASSET PROMPT BANK ON POLE ROIS
# =============================================================================
# EXPLANATION:
# This is an initial Silver-branch bug cell to see how many assets SAM3 can
# pick up from the fixed pole-top ROIs.
#
# WHAT THIS CELL DOES:
#   1) reads pole_rois_df from Cell 8
#   2) runs a prompt bank on each ROI image
#   3) keeps all raw detections (no ranking / no cleanup yet)
#   4) saves one combined colour overlay per ROI image
#   5) saves one binary mask file per detection when available
#   6) builds:
#        - asset_prompt_manifest_df
#        - asset_prompt_results_df
#        - asset_candidates_df
#
# IMPORTANT:
# - this is a bug / exploration cell
# - each prompt has its own fixed overlay colour
# - this cell is for finding "what fires at all", not for final selection
# =============================================================================

# -----------------------------------------------------------------------------
# 0. SAFETY CHECKS
# -----------------------------------------------------------------------------
required_globals = [
    "pole_rois_df",
    "model",
    "processor",
    "DEVICE",
    "SILVER_ASSET_PROMPT_RUNS",
    "SILVER_ASSET_OVERLAYS",
    "SILVER_ASSET_MASKS",
    "save_state",
]

# Check that all required globals exist before continuing.
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise NameError(
        "Cell 9a cannot run because some required variables are missing.\n"
        "Please make sure Cells 2, 5b, and 8 have already run.\n"
        f"Missing globals: {missing_globals}"
    )

# Confirm the ROI table is a pandas DataFrame.
if not isinstance(pole_rois_df, pd.DataFrame):
    raise TypeError("pole_rois_df exists but is not a pandas DataFrame.")

# Stop early if there are no ROI rows to process.
if pole_rois_df.empty:
    raise ValueError("pole_rois_df is empty. Please check Cell 8.")

# Confirm the ROI image path column exists.
if "roi_image_path" not in pole_rois_df.columns:
    raise ValueError("pole_rois_df is missing required column: roi_image_path")

# -----------------------------------------------------------------------------
# 1. PROMPT BANK
# -----------------------------------------------------------------------------
# EXPLANATION:
# This is a local prompt bank for the initial asset-detection bug test.
# Each prompt gets a different fixed colour in the overlay.
# -----------------------------------------------------------------------------
ASSET_PROMPT_BANK = [
    {
        "prompt": "steel cap",
        "text_score_threshold": 0.25,
        "mask_threshold": 0.50,
        "color_rgb": (0, 255, 255),
        "color_name": "green",
    }
]

# Show only a small gallery of overlay images.
ASSET_DEBUG_GALLERY_COUNT = min(6, len(pole_rois_df))

print("Asset prompt bank:")
for cfg in ASSET_PROMPT_BANK:
    print(
        f"  - {cfg['prompt']} | "
        f"text_thr={cfg['text_score_threshold']} | "
        f"mask_thr={cfg['mask_threshold']} | "
        f"color={cfg['color_name']}"
    )

# -----------------------------------------------------------------------------
# 2. HELPER: SAFE DISPLAY
# -----------------------------------------------------------------------------
def _safe_display(obj):
    """
    Display a pandas object in a notebook if possible; otherwise print it.
    """
    try:
        display(obj)
    except Exception:
        print(obj)

# -----------------------------------------------------------------------------
# 3. HELPER: SAFE COLUMN SUBSET
# -----------------------------------------------------------------------------
def _existing_cols(df, cols):
    """
    Return only the requested columns that actually exist in the DataFrame.
    """
    if df is None or not isinstance(df, pd.DataFrame):
        return []

    return [c for c in cols if c in df.columns]

# -----------------------------------------------------------------------------
# 4. HELPER: SAFE FILE-STEM
# -----------------------------------------------------------------------------
def make_safe_stem(text):
    """
    Convert a string into a filesystem-safe stem.
    """
    if text is None or (isinstance(text, float) and pd.isna(text)):
        text = "image"

    text = str(text).strip()
    if len(text) == 0:
        text = "image"

    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")

    return text if len(text) > 0 else "image"

# -----------------------------------------------------------------------------
# 5. HELPER: CLIP BOX TO IMAGE
# -----------------------------------------------------------------------------
def clip_box_to_image(x1, y1, x2, y2, image_w, image_h):
    """
    Clip box coordinates safely to image bounds.
    """
    x1 = float(np.clip(x1, 0, max(image_w - 1, 0)))
    y1 = float(np.clip(y1, 0, max(image_h - 1, 0)))
    x2 = float(np.clip(x2, 0, max(image_w - 1, 0)))
    y2 = float(np.clip(y2, 0, max(image_h - 1, 0)))

    x1, x2 = sorted([x1, x2])
    y1, y2 = sorted([y1, y2])

    return x1, y1, x2, y2

# -----------------------------------------------------------------------------
# 6. HELPER: NORMALIZE RAW BOXES
# -----------------------------------------------------------------------------
def normalize_raw_boxes(raw_boxes):
    """
    Normalize raw box output into a list of [x1, y1, x2, y2].
    """
    if raw_boxes is None:
        return []

    if isinstance(raw_boxes, torch.Tensor):
        raw_boxes = raw_boxes.detach().cpu().tolist()
    elif isinstance(raw_boxes, np.ndarray):
        raw_boxes = raw_boxes.tolist()

    if not isinstance(raw_boxes, (list, tuple)):
        return []

    if len(raw_boxes) == 0:
        return []

    if isinstance(raw_boxes[0], (int, float, np.number)):
        if len(raw_boxes) >= 4:
            return [[
                float(raw_boxes[0]),
                float(raw_boxes[1]),
                float(raw_boxes[2]),
                float(raw_boxes[3]),
            ]]
        return []

    boxes = []

    for box in raw_boxes:
        if isinstance(box, torch.Tensor):
            box = box.detach().cpu().tolist()
        elif isinstance(box, np.ndarray):
            box = box.tolist()

        if isinstance(box, (list, tuple)) and len(box) >= 4:
            boxes.append([
                float(box[0]),
                float(box[1]),
                float(box[2]),
                float(box[3]),
            ])

    return boxes

# -----------------------------------------------------------------------------
# 7. HELPER: NORMALIZE RAW SCORES
# -----------------------------------------------------------------------------
def normalize_raw_scores(raw_scores):
    """
    Normalize raw score output into a flat float list.
    """
    if raw_scores is None:
        return []

    if isinstance(raw_scores, torch.Tensor):
        raw_scores = raw_scores.detach().cpu().flatten().tolist()
    elif isinstance(raw_scores, np.ndarray):
        raw_scores = raw_scores.flatten().tolist()

    if not isinstance(raw_scores, (list, tuple)):
        return []

    scores = []

    for s in raw_scores:
        if isinstance(s, torch.Tensor):
            s = s.detach().cpu().item()
        elif isinstance(s, np.ndarray):
            s = float(np.asarray(s).reshape(-1)[0])

        scores.append(float(s))

    return scores

# -----------------------------------------------------------------------------
# 8. HELPER: NORMALIZE RAW MASKS
# -----------------------------------------------------------------------------
def normalize_raw_masks(raw_masks):
    """
    Normalize raw mask output into a list of boolean HxW masks.
    """
    if raw_masks is None:
        return []

    masks = []

    if isinstance(raw_masks, torch.Tensor):
        raw_masks = raw_masks.detach().cpu().numpy()
    elif isinstance(raw_masks, list):
        raw_masks = [
            m.detach().cpu().numpy() if isinstance(m, torch.Tensor) else np.asarray(m)
            for m in raw_masks
        ]

    if isinstance(raw_masks, np.ndarray):
        # Handle [H, W]
        if raw_masks.ndim == 2:
            raw_masks = [raw_masks]

        # Handle [N, H, W]
        elif raw_masks.ndim == 3:
            raw_masks = [raw_masks[i] for i in range(raw_masks.shape[0])]

        # Handle [N, 1, H, W] or [1, N, H, W]
        elif raw_masks.ndim == 4:
            raw_masks = [np.squeeze(raw_masks[i]) for i in range(raw_masks.shape[0])]

        else:
            raw_masks = []

    if not isinstance(raw_masks, list):
        return []

    for m in raw_masks:
        m = np.asarray(m)
        m = np.squeeze(m)

        if m.ndim != 2:
            continue

        # Convert logits / probabilities / ids to a boolean mask.
        if m.dtype == np.bool_:
            mask_bool = m
        else:
            mask_bool = m > 0.5

        masks.append(mask_bool.astype(bool))

    return masks

# -----------------------------------------------------------------------------
# 9. HELPER: FALLBACK MASK EXTRACTION FROM SEGMENTATION MAP
# -----------------------------------------------------------------------------
def extract_masks_from_segmentation_map(result):
    """
    Fallback path for results that expose a segmentation map + segments_info
    instead of an explicit masks list.
    """
    segmentation = result.get("segmentation", None)
    segments_info = result.get("segments_info", None)

    if segmentation is None or not isinstance(segments_info, list):
        return [], []

    if isinstance(segmentation, torch.Tensor):
        segmentation = segmentation.detach().cpu().numpy()
    else:
        segmentation = np.asarray(segmentation)

    segmentation = np.squeeze(segmentation)

    if segmentation.ndim != 2:
        return [], []

    masks = []
    scores = []

    for seg in segments_info:
        seg_id = seg.get("id", None)
        if seg_id is None:
            continue

        mask_bool = (segmentation == seg_id)
        if mask_bool.sum() == 0:
            continue

        masks.append(mask_bool.astype(bool))
        scores.append(float(seg["score"]) if "score" in seg and seg["score"] is not None else np.nan)

    return masks, scores

# -----------------------------------------------------------------------------
# 10. HELPER: BUILD A BOX FROM A MASK
# -----------------------------------------------------------------------------
def box_from_mask(mask_bool):
    """
    Build a bounding box from a boolean mask.
    """
    if mask_bool is None or mask_bool.sum() == 0:
        return None

    ys, xs = np.where(mask_bool)

    if len(xs) == 0 or len(ys) == 0:
        return None

    return [
        float(xs.min()),
        float(ys.min()),
        float(xs.max() + 1),
        float(ys.max() + 1),
    ]

# -----------------------------------------------------------------------------
# 11. HELPER: RUN ONE ASSET PROMPT ON ONE ROI IMAGE
# -----------------------------------------------------------------------------
def run_asset_prompt_on_roi(
    image_pil,
    prompt,
    text_threshold,
    mask_threshold,
):
    """
    Run one SAM3 asset prompt on one ROI image and return the first result dict.
    """
    image_w, image_h = image_pil.size

    # Tokenize the image + text prompt for SAM3.
    inputs = processor(
        images=image_pil,
        text=prompt,
        return_tensors="pt",
    )

    # Move the full batch to the active device.
    inputs = inputs.to(DEVICE)

    # Prefer processor-provided original sizes if available.
    if "original_sizes" in inputs:
        target_sizes = inputs["original_sizes"].detach().cpu().tolist()
    else:
        target_sizes = [[image_h, image_w]]

    # Run inference without gradients.
    with torch.no_grad():
        outputs = model(**inputs)

    # Convert model outputs into instance segmentation predictions.
    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=text_threshold,
        mask_threshold=mask_threshold,
        target_sizes=target_sizes,
    )

    # Pull out the first image result safely.
    result = results[0] if isinstance(results, list) and len(results) > 0 else {}

    return result

# -----------------------------------------------------------------------------
# 12. HELPER: BUILD DETECTION ROWS FOR ONE PROMPT RUN
# -----------------------------------------------------------------------------
def build_asset_detection_rows(
    result,
    row,
    prompt_cfg,
    roi_stem,
    image_w,
    image_h,
):
    """
    Convert one prompt-run result into standardized detection rows plus overlay
    payloads for visualization.
    """
    prompt = prompt_cfg["prompt"]
    prompt_safe = make_safe_stem(prompt)
    color_rgb = tuple(prompt_cfg["color_rgb"])
    color_name = prompt_cfg["color_name"]

    # Extract boxes / scores / masks from the SAM3 result.
    boxes = normalize_raw_boxes(result.get("boxes", None))
    scores = normalize_raw_scores(result.get("scores", None))
    masks = normalize_raw_masks(result.get("masks", None))

    # Fallback: derive masks from segmentation map if needed.
    if len(masks) == 0:
        seg_masks, seg_scores = extract_masks_from_segmentation_map(result)
        if len(seg_masks) > 0:
            masks = seg_masks
            if len(scores) == 0:
                scores = seg_scores

    # Use the largest available count across boxes / scores / masks.
    n = max(len(boxes), len(scores), len(masks))

    prompt_rows = []
    overlay_detections = []

    # Build a prompt-specific mask folder.
    prompt_mask_dir = os.path.join(SILVER_ASSET_MASKS, prompt_safe)
    os.makedirs(prompt_mask_dir, exist_ok=True)

    for det_idx in range(n):
        # Read the detection mask if present.
        mask_bool = masks[det_idx] if det_idx < len(masks) else None

        # Resize the mask to the ROI image size if needed.
        if mask_bool is not None and mask_bool.shape != (image_h, image_w):
            mask_bool = cv2.resize(
                mask_bool.astype(np.uint8),
                (image_w, image_h),
                interpolation=cv2.INTER_NEAREST,
            ).astype(bool)

        # Read the box if present, otherwise derive it from the mask.
        box = boxes[det_idx] if det_idx < len(boxes) else None
        if box is None and mask_bool is not None:
            box = box_from_mask(mask_bool)

        # Skip detections that have neither a usable box nor a usable mask.
        if box is None:
            continue

        # Clip the box safely to image bounds.
        x1, y1, x2, y2 = clip_box_to_image(
            box[0], box[1], box[2], box[3], image_w, image_h
        )

        # Read the score if present.
        score = scores[det_idx] if det_idx < len(scores) else np.nan
        score = float(score) if pd.notna(score) else np.nan

        # Compute box geometry.
        box_w = max(float(x2 - x1), 1.0)
        box_h = max(float(y2 - y1), 1.0)
        box_area = float(box_w * box_h)
        box_area_frac = box_area / float(max(image_w * image_h, 1))

        # Save the binary mask if present.
        has_mask = False
        mask_pixel_count = 0
        mask_area_frac = np.nan
        mask_file_name = None
        mask_image_path = None

        if mask_bool is not None:
            has_mask = True
            mask_pixel_count = int(mask_bool.sum())
            mask_area_frac = mask_pixel_count / float(max(image_w * image_h, 1))

            mask_file_name = (
                f"{roi_stem}__{prompt_safe}__det_{det_idx:03d}_mask.png"
            )
            mask_image_path = os.path.join(prompt_mask_dir, mask_file_name)

            mask_img = Image.fromarray((mask_bool.astype(np.uint8) * 255), mode="L")
            mask_img.save(mask_image_path, format="PNG")

        # Append one standardized result row.
        prompt_rows.append({
            "image_id": row["image_id"] if "image_id" in row.index and pd.notna(row["image_id"]) else None,
            "file_name": row["file_name"] if "file_name" in row.index and pd.notna(row["file_name"]) else None,
            "roi_file_name": row["roi_file_name"] if "roi_file_name" in row.index and pd.notna(row["roi_file_name"]) else None,
            "roi_image_path": row["roi_image_path"],
            "prompt": prompt,
            "prompt_safe": prompt_safe,
            "text_score_threshold": float(prompt_cfg["text_score_threshold"]),
            "mask_threshold": float(prompt_cfg["mask_threshold"]),
            "color_name": color_name,
            "color_rgb": list(color_rgb),
            "det_idx": int(det_idx),
            "score": score,
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),
            "box_w": float(box_w),
            "box_h": float(box_h),
            "box_area": float(box_area),
            "box_area_frac": float(box_area_frac),
            "has_mask": bool(has_mask),
            "mask_pixel_count": int(mask_pixel_count),
            "mask_area_frac": float(mask_area_frac) if pd.notna(mask_area_frac) else np.nan,
            "mask_file_name": mask_file_name,
            "mask_image_path": mask_image_path,
            "overlay_source": "stream1_prompt_bank",
            "source_layer": "silver",
            "source_folder": SILVER_ASSET_OVERLAYS,
        })

        # Append one overlay payload row for this detection.
        overlay_detections.append({
            "prompt": prompt,
            "score": score,
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),
            "color_rgb": color_rgb,
            "color_name": color_name,
            "mask_bool": mask_bool,
        })

    return prompt_rows, overlay_detections

# -----------------------------------------------------------------------------
# 13. HELPER: SAVE ONE COLOURED OVERLAY IMAGE
# -----------------------------------------------------------------------------
def save_asset_overlay_image(
    image_pil,
    detections,
    overlay_image_path,
    title_text,
):
    """
    Save one combined overlay image with colour masks, boxes, labels, and legend.
    """
    import matplotlib.patches as patches
    from matplotlib.lines import Line2D

    # Start from the plain ROI image.
    overlay_rgb = np.array(image_pil).copy()

    # Paint masks first so boxes / labels remain visible on top.
    alpha = 0.35

    for det in detections:
        mask_bool = det.get("mask_bool", None)
        color_rgb = np.asarray(det["color_rgb"], dtype=np.float32)

        if mask_bool is None:
            continue

        if mask_bool.shape != overlay_rgb.shape[:2]:
            continue

        overlay_rgb[mask_bool] = (
            (1.0 - alpha) * overlay_rgb[mask_bool] + alpha * color_rgb
        ).astype(np.uint8)

    # Draw the overlay using matplotlib so labels and legend are easy to add.
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(overlay_rgb)
    ax.set_title(title_text, fontsize=11)
    ax.axis("off")


        # Draw one box + label per detection.
    for det in detections:
        color_rgb = tuple(np.asarray(det["color_rgb"], dtype=np.float32) / 255.0)

        rect = patches.Rectangle(
            (det["x1"], det["y1"]),
            det["x2"] - det["x1"],
            det["y2"] - det["y1"],
            linewidth=3.5,          # thicker box
            edgecolor=color_rgb,
            facecolor="none",
        )
        ax.add_patch(rect)

        label = det["prompt"]
        if pd.notna(det["score"]):
            label += f" | score={float(det['score']):.3f}"

        # Put label slightly inside the box top-left for better visibility
        label_x = max(det["x1"] + 4, 8)
        label_y = max(det["y1"] + 14, 14)

        ax.text(
            label_x,
            label_y,
            label,
            fontsize=10.5,          # bigger text
            fontweight="bold",      # clearer
            color="white",          # white writing
            bbox=dict(
                facecolor=color_rgb, # green background
                alpha=0.98,          # more solid
                edgecolor="black",   # outline helps visibility
                linewidth=0.8,
                boxstyle="round,pad=0.25",
            ),
        )

    # Build a legend using only prompts that appear in this overlay.
    present_prompts = {det["prompt"] for det in detections}
    legend_handles = []

    for cfg in ASSET_PROMPT_BANK:
        if cfg["prompt"] in present_prompts:
            legend_handles.append(
                Line2D(
                    [0], [0],
                    color=tuple(np.asarray(cfg["color_rgb"], dtype=np.float32) / 255.0),
                    lw=4,
                    label=cfg["prompt"],
                )
            )

    # Add either the legend or a "no detections" note.
    if len(legend_handles) > 0:
        ax.legend(
            handles=legend_handles,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.08),
            ncol=2,
            fontsize=8,
            frameon=True,
        )
    else:
        ax.text(
            0.5,
            0.02,
            "No detections",
            transform=ax.transAxes,
            ha="center",
            va="bottom",
            fontsize=10,
            color="white",
            bbox=dict(
                facecolor="black",
                alpha=0.70,
                edgecolor="none",
                pad=3.0,
            ),
        )

    plt.tight_layout()
    fig.savefig(overlay_image_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

# -----------------------------------------------------------------------------
# 14. HELPER: SHOW A SMALL OVERLAY GALLERY
# -----------------------------------------------------------------------------
def show_asset_overlay_gallery(asset_prompt_manifest_df, gallery_count=6):
    """
    Show a small gallery of the saved combined overlay images.
    """
    gallery_df = (
        asset_prompt_manifest_df[
            ["roi_file_name", "overlay_image_path"]
        ]
        .drop_duplicates()
        .head(gallery_count)
        .reset_index(drop=True)
    )

    if gallery_df.empty:
        print("No overlay images were created.")
        return

    n = len(gallery_df)
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 6 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax in axes[n:]:
        ax.axis("off")

    for ax, (_, row_) in zip(axes, gallery_df.iterrows()):
        with Image.open(row_["overlay_image_path"]) as img:
            img = img.convert("RGB")
            image_rgb = np.array(img)

        ax.imshow(image_rgb)
        ax.set_title(row_["roi_file_name"], fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close()

# -----------------------------------------------------------------------------
# 15. RESET STREAM-1 SILVER OUTPUT FOLDERS
# -----------------------------------------------------------------------------
# EXPLANATION:
# This is a bug / exploratory cell, so we wipe the old stream-1 outputs on rerun
# to avoid stale overlays and masks.
# -----------------------------------------------------------------------------
for d in [
    SILVER_ASSET_PROMPT_RUNS,
    SILVER_ASSET_OVERLAYS,
    SILVER_ASSET_MASKS,
]:
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

# -----------------------------------------------------------------------------
# 16. RUN THE PROMPT BANK ON ALL POLE ROIS
# -----------------------------------------------------------------------------
asset_prompt_manifest_rows = []
asset_prompt_results_rows = []

print(f"\nRunning asset prompt bank on {len(pole_rois_df)} ROI image(s)...")

for idx, row in pole_rois_df.iterrows():
    # Read the ROI image path safely.
    roi_image_path = row["roi_image_path"] if "roi_image_path" in row.index and pd.notna(row["roi_image_path"]) else None

    # Stop early if the ROI image path is missing or invalid.
    if pd.isna(roi_image_path) or not isinstance(roi_image_path, str) or len(roi_image_path.strip()) == 0:
        raise ValueError("A pole_rois_df row is missing a valid roi_image_path.")

    # Read helper identifiers safely.
    image_id = row["image_id"] if "image_id" in row.index and pd.notna(row["image_id"]) else None
    file_name = row["file_name"] if "file_name" in row.index and pd.notna(row["file_name"]) else os.path.basename(roi_image_path)
    roi_file_name = row["roi_file_name"] if "roi_file_name" in row.index and pd.notna(row["roi_file_name"]) else os.path.basename(roi_image_path)

    # Build a stable stem for saved artifacts.
    roi_stem = make_safe_stem(
        image_id if image_id is not None else os.path.splitext(roi_file_name)[0]
    )

    # Open the ROI image once for all prompt runs on this row.
    with Image.open(roi_image_path) as img:
        image_pil = img.convert("RGB")
        image_pil.load()

    image_w, image_h = image_pil.size

    # Hold all detections for the combined per-image overlay.
    overlay_detections = []

    # Hold the per-prompt manifest rows for this one ROI image.
    per_image_manifest_rows = []

    # Run each prompt in the bank.
    for prompt_cfg in ASSET_PROMPT_BANK:
        prompt = prompt_cfg["prompt"]
        prompt_safe = make_safe_stem(prompt)

        # Create a prompt-specific run folder.
        prompt_run_dir = os.path.join(SILVER_ASSET_PROMPT_RUNS, prompt_safe)
        os.makedirs(prompt_run_dir, exist_ok=True)

        # Run the prompt on the ROI image.
        result = run_asset_prompt_on_roi(
            image_pil=image_pil,
            prompt=prompt,
            text_threshold=float(prompt_cfg["text_score_threshold"]),
            mask_threshold=float(prompt_cfg["mask_threshold"]),
        )

        # Convert the prompt-run result into standardized detection rows.
        prompt_rows, prompt_overlay_detections = build_asset_detection_rows(
            result=result,
            row=row,
            prompt_cfg=prompt_cfg,
            roi_stem=roi_stem,
            image_w=image_w,
            image_h=image_h,
        )

        # Save a lightweight JSON summary for this prompt-run.
        prompt_run_json_path = os.path.join(
            prompt_run_dir,
            f"{roi_stem}__{prompt_safe}__run.json",
        )

        with open(prompt_run_json_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "image_id": image_id,
                    "file_name": file_name,
                    "roi_file_name": roi_file_name,
                    "roi_image_path": roi_image_path,
                    "prompt": prompt,
                    "prompt_safe": prompt_safe,
                    "text_score_threshold": float(prompt_cfg["text_score_threshold"]),
                    "mask_threshold": float(prompt_cfg["mask_threshold"]),
                    "color_name": prompt_cfg["color_name"],
                    "color_rgb": list(prompt_cfg["color_rgb"]),
                    "n_detections": len(prompt_rows),
                    "has_any_detection": bool(len(prompt_rows) > 0),
                },
                f,
                indent=2,
                default=str,
            )

        # Extend the global detection row list.
        asset_prompt_results_rows.extend(prompt_rows)

        # Extend the per-image combined overlay list.
        overlay_detections.extend(prompt_overlay_detections)

        # Add one per-prompt manifest row.
        per_image_manifest_rows.append({
            "image_id": image_id,
            "file_name": file_name,
            "roi_file_name": roi_file_name,
            "roi_image_path": roi_image_path,
            "prompt": prompt,
            "prompt_safe": prompt_safe,
            "text_score_threshold": float(prompt_cfg["text_score_threshold"]),
            "mask_threshold": float(prompt_cfg["mask_threshold"]),
            "color_name": prompt_cfg["color_name"],
            "color_rgb": list(prompt_cfg["color_rgb"]),
            "n_detections": int(len(prompt_rows)),
            "has_any_detection": bool(len(prompt_rows) > 0),
            "prompt_run_json_path": prompt_run_json_path,
        })

    # Save one combined overlay image for this ROI image.
    overlay_file_name = f"{roi_stem}__asset_prompt_overlay.png"
    overlay_image_path = os.path.join(SILVER_ASSET_OVERLAYS, overlay_file_name)

    save_asset_overlay_image(
        image_pil=image_pil,
        detections=overlay_detections,
        overlay_image_path=overlay_image_path,
        title_text=f"{roi_file_name}\nSTREAM 1 — ASSET PROMPT BANK",
    )

    # Add the overlay path to every manifest row from this ROI image.
    for mrow in per_image_manifest_rows:
        mrow["overlay_image_path"] = overlay_image_path

    asset_prompt_manifest_rows.extend(per_image_manifest_rows)

    # Print occasional progress updates.
    if (idx + 1) % 20 == 0 or idx == 0 or idx == len(pole_rois_df) - 1:
        print(f"  [{idx + 1}/{len(pole_rois_df)}] {roi_file_name}")

    # Release loop-local objects.
    del image_pil

# Run one cleanup pass after all prompt runs are complete.
gc.collect()

# -----------------------------------------------------------------------------
# 17. BUILD FINAL OUTPUT TABLES
# -----------------------------------------------------------------------------
asset_prompt_manifest_df = pd.DataFrame(asset_prompt_manifest_rows)
asset_prompt_results_df = pd.DataFrame(asset_prompt_results_rows)

# Keep a convenience alias aligned with your existing registry.
asset_candidates_df = asset_prompt_results_df.copy()

# -----------------------------------------------------------------------------
# 18. PRINT SUMMARY
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("STREAM 1 ASSET PROMPT BANK SUMMARY")
print("=" * 100)
print(f"ROI images processed      : {len(pole_rois_df)}")
print(f"Prompt runs completed     : {len(asset_prompt_manifest_df)}")
print(f"Raw detections collected  : {len(asset_prompt_results_df)}")
print(f"Overlay folder            : {SILVER_ASSET_OVERLAYS}")
print(f"Mask folder               : {SILVER_ASSET_MASKS}")

# -----------------------------------------------------------------------------
# 19. DISPLAY PER-PROMPT SUMMARY
# -----------------------------------------------------------------------------
if asset_prompt_manifest_df.empty:
    print("No prompt-run manifest rows were created.")
else:
    prompt_summary_df = (
        asset_prompt_manifest_df
        .groupby("prompt", dropna=False)
        .agg(
            roi_images=("roi_file_name", "nunique"),
            images_with_detection=("has_any_detection", "sum"),
            total_detections=("n_detections", "sum"),
        )
        .reset_index()
        .sort_values(["total_detections", "images_with_detection"], ascending=[False, False])
        .reset_index(drop=True)
    )

    print("\nPer-prompt summary:")
    _safe_display(prompt_summary_df)

# -----------------------------------------------------------------------------
# 20. DISPLAY RESULTS TABLE PREVIEW
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("ASSET PROMPT RESULTS PREVIEW")
print("=" * 100)

if asset_prompt_results_df.empty:
    print("No raw detections were found by the current prompt bank.")
else:
    results_cols = _existing_cols(
        asset_prompt_results_df,
        [
            "image_id",
            "file_name",
            "roi_file_name",
            "prompt",
            "det_idx",
            "score",
            "x1",
            "y1",
            "x2",
            "y2",
            "box_w",
            "box_h",
            "box_area_frac",
            "has_mask",
            "mask_area_frac",
            "mask_image_path",
        ]
    )

    _safe_display(
        asset_prompt_results_df[results_cols]
        .sort_values(["roi_file_name", "prompt", "score"], ascending=[True, True, False])
        .reset_index(drop=True)
    )

# -----------------------------------------------------------------------------
# 21. SHOW A SMALL OVERLAY GALLERY
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("STREAM 1 OVERLAY GALLERY")
print("=" * 100)

show_asset_overlay_gallery(
    asset_prompt_manifest_df=asset_prompt_manifest_df,
    gallery_count=ASSET_DEBUG_GALLERY_COUNT,
)

# -----------------------------------------------------------------------------
# 22. SAVE OUTPUTS INTO NOTEBOOK STATE
# -----------------------------------------------------------------------------
if "save_state" in globals():
    save_state(
        df_names=[
            name for name in [
                "pole_rois_df",
                "asset_prompt_manifest_df",
                "asset_prompt_results_df",
                "asset_candidates_df",
            ]
            if isinstance(globals().get(name), pd.DataFrame)
        ],
        config_extra={
            "STREAM1_ASSET_PROMPT_BANK": [
                {
                    "prompt": cfg["prompt"],
                    "text_score_threshold": float(cfg["text_score_threshold"]),
                    "mask_threshold": float(cfg["mask_threshold"]),
                    "color_name": cfg["color_name"],
                    "color_rgb": list(cfg["color_rgb"]),
                }
                for cfg in ASSET_PROMPT_BANK
            ],
            "ASSET_DEBUG_GALLERY_COUNT": ASSET_DEBUG_GALLERY_COUNT,
        },
        nb_globals=globals(),
    )

print("\nCell 9a completed.")
print("Persisted outputs:")
print("  - asset_prompt_manifest_df")
print("  - asset_prompt_results_df")
print("  - asset_candidates_df")